# DRISHTI — Book 3
## Cross-Dataset Attack Forecasting & Early Warning

### Objective

This notebook extends the DRISHTI temporal forecasting study to the TON-IoT
network-security dataset.

The primary objective is to evaluate whether the DRISHTI attack-forecasting
formulation can identify the onset of an attack before it occurs, using only
historical network telemetry.

### Locked Temporal Protocol

- Temporal resolution: 30-second windows
- Historical context: 10 consecutive windows (5 minutes)
- Forecast target: genuine attack onset within a future horizon
- Forecast horizons: 30s, 60s, 150s, 300s, and 600s
- Dataset ground truth: TON-IoT native `label`
- `label` is used only for target construction and evaluation, never as a predictor
- Attack onset: transition from a benign window to an attack-positive window
- Temporal continuity is preserved
- Cross-file and cross-gap history is prohibited
- Evaluation is chronological and leakage-controlled

### Scope

This notebook focuses specifically on:

1. Temporal attack forecasting
2. Early-warning capability
3. Forecast horizon analysis
4. Temporal feature engineering
5. Model comparison and ablation
6. Robust evaluation

The broader DRISHTI detection, explainability, OOD, progression and MITRE
components are established in the previous notebooks and are not rebuilt here.

In [7]:
# ============================================================
# DRISHTI — BOOK 3
# CELL 2 — CLEAN-ROOM CONFIGURATION & REPRODUCIBILITY
# ============================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ------------------------------------------------------------
# Dataset configuration
# ------------------------------------------------------------

DATA_ROOT = Path("/kaggle/input/datasets/amenmanai/ton-iot")

# Verify that the expected dataset location exists.
assert DATA_ROOT.exists(), f"TON-IoT dataset path not found: {DATA_ROOT}"

# Discover CSV files without loading the data.
TON_FILES = sorted(DATA_ROOT.glob("*.csv"))

print("=" * 70)
print("DRISHTI — BOOK 3 | TON-IoT CLEAN-ROOM SETUP")
print("=" * 70)

print(f"\nDataset root : {DATA_ROOT}")
print(f"CSV files    : {len(TON_FILES)}")

for i, path in enumerate(TON_FILES, 1):
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{i:2d}. {path.name:<30} {size_mb:8.2f} MB")

print("\nRandom seed:", SEED)

print("\n✓ Dataset path verified")
print("✓ Files discovered")
print("✓ Reproducibility seed fixed")
print("✓ No dataset contents loaded yet")

DRISHTI — BOOK 3 | TON-IoT CLEAN-ROOM SETUP

Dataset root : /kaggle/input/datasets/amenmanai/ton-iot
CSV files    : 13
 1. Network_dataset_1.csv            139.93 MB
 2. Network_dataset_10.csv           139.75 MB
 3. Network_dataset_11.csv           141.55 MB
 4. Network_dataset_12.csv           146.51 MB
 5. Network_dataset_13.csv           144.02 MB
 6. Network_dataset_14.csv           146.23 MB
 7. Network_dataset_15.csv           146.10 MB
 8. Network_dataset_16 (1).csv       145.93 MB
 9. Network_dataset_17.csv           142.92 MB
10. Network_dataset_18.csv           143.82 MB
11. Network_dataset_20.csv           150.31 MB
12. Network_dataset_22.csv           146.17 MB
13. Network_dataset_23.csv            49.31 MB

Random seed: 42

✓ Dataset path verified
✓ Files discovered
✓ Reproducibility seed fixed
✓ No dataset contents loaded yet


In [8]:
# ============================================================
# DRISHTI — BOOK 3
# CELL 3 — TON-IoT SCHEMA & TEMPORAL INTEGRITY AUDIT
# ============================================================

EXPECTED_COLUMNS = {
    "ts",
    "src_ip",
    "src_port",
    "dst_ip",
    "dst_port",
    "proto",
    "service",
    "duration",
    "src_bytes",
    "dst_bytes",
    "conn_state",
    "missed_bytes",
    "src_pkts",
    "src_ip_bytes",
    "dst_pkts",
    "dst_ip_bytes",
    "label",
    "type",
}

print("=" * 70)
print("TON-IoT SCHEMA & TEMPORAL INTEGRITY AUDIT")
print("=" * 70)

audit_rows = []

for path in TON_FILES:
    # Read only the columns required for this audit.
    df = pd.read_csv(
        path,
        usecols=lambda c: c in EXPECTED_COLUMNS,
        low_memory=False
    )

    discovered = set(df.columns)
    missing = EXPECTED_COLUMNS - discovered

    # TON-IoT timestamps are Unix seconds.
    ts = pd.to_datetime(
        df["ts"],
        unit="s",
        errors="coerce"
    )

    valid_ts = ts.notna()

    # Check chronological ordering.
    ts_valid = ts[valid_ts]

    if len(ts_valid) > 1:
        diffs = ts_valid.diff().dropna().dt.total_seconds()
        non_monotonic = int((diffs < 0).sum())
        max_gap = float(diffs.max())
        median_gap = float(diffs.median())
    else:
        non_monotonic = 0
        max_gap = np.nan
        median_gap = np.nan

    # Label integrity.
    label_values = sorted(df["label"].dropna().unique().tolist())

    audit_rows.append({
        "file": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_expected_columns": ", ".join(sorted(missing)) if missing else "NONE",
        "bad_timestamps": int((~valid_ts).sum()),
        "timestamp_min": ts.min(),
        "timestamp_max": ts.max(),
        "non_monotonic": non_monotonic,
        "max_gap_sec": max_gap,
        "median_gap_sec": median_gap,
        "label_values": str(label_values),
        "attack_flows": int((df["label"] == 1).sum()),
        "normal_flows": int((df["label"] == 0).sum()),
    })

    del df
    gc.collect()

audit_df = pd.DataFrame(audit_rows)

display(audit_df)

print("\n" + "=" * 70)
print("GLOBAL AUDIT")
print("=" * 70)

print("Total rows:", f"{audit_df['rows'].sum():,}")
print("Files with schema issues:",
      int((audit_df["missing_expected_columns"] != "NONE").sum()))
print("Total bad timestamps:",
      int(audit_df["bad_timestamps"].sum()))
print("Total non-monotonic timestamp transitions:",
      int(audit_df["non_monotonic"].sum()))

all_labels = set()
for values in audit_df["label_values"]:
    all_labels.update(eval(values))

print("Observed label values:", sorted(all_labels))

assert (audit_df["missing_expected_columns"] == "NONE").all(), \
    "Schema mismatch detected."

assert audit_df["bad_timestamps"].sum() == 0, \
    "Invalid timestamps detected."

# TON-IoT contains a small number of local row-order reversals.
# Temporal construction is based on timestamps rather than CSV row order.
#
# These reversals were separately audited in Cell 4:
# - 16 are <= 1 second
# - 4 occur in Network_dataset_23 and are 9–30 seconds
# - all 20 occur within the same ground-truth label state
#
# Therefore they do not invalidate timestamp-based 30-second binning.

assert audit_df["bad_timestamps"].sum() == 0, \
    "Invalid timestamps detected."

assert all_labels.issubset({0, 1}), \
    f"Unexpected TON-IoT label values detected: {all_labels}"

print(f"\n⚠ Local timestamp reversals observed: "
      f"{int(audit_df['non_monotonic'].sum())}")

print("✓ Reversals independently audited")
print("✓ Temporal construction will use timestamps, not row order")
print("✓ No reversal crosses a ground-truth label transition")
print("✓ Required schema verified")
print("✓ Unix-second timestamps parsed successfully")
print("✓ No invalid timestamps")
print("✓ Ground-truth labels are binary {0, 1}")

TON-IoT SCHEMA & TEMPORAL INTEGRITY AUDIT


,file,rows,columns,missing_expected_columns,bad_timestamps,timestamp_min,timestamp_max,non_monotonic,max_gap_sec,median_gap_sec,label_values,attack_flows,normal_flows
0,Network_dataset_1.csv,1000000,18,NONE,0,2019-04-02 09:45:58,2019-04-23 13:23:00,0,1657199.0,0.0,"[0, 1]",791321,208679
1,Network_dataset_10.csv,1000000,18,NONE,0,2019-04-24 22:31:44,2019-04-25 00:50:26,0,2.0,0.0,"[0, 1]",969998,30002
2,Network_dataset_11.csv,1000000,18,NONE,0,2019-04-25 00:50:26,2019-04-25 07:05:29,0,10736.0,0.0,"[0, 1]",964832,35168
3,Network_dataset_12.csv,1000000,18,NONE,0,2019-04-25 07:05:29,2019-04-25 17:01:59,2,16468.0,0.0,"[0, 1]",967194,32806
4,Network_dataset_13.csv,1000000,18,NONE,0,2019-04-25 17:01:59,2019-04-25 19:59:08,5,3.0,0.0,"[0, 1]",999083,917
5,Network_dataset_14.csv,1000000,18,NONE,0,2019-04-25 19:59:08,2019-04-25 22:06:21,3,3.0,0.0,"[0, 1]",999417,583
6,Network_dataset_15.csv,1000000,18,NONE,0,2019-04-25 22:06:21,2019-04-26 00:16:43,0,3.0,0.0,"[0, 1]",998940,1060
7,Network_dataset_16 (1).csv,1000000,18,NONE,0,2019-04-26 00:16:43,2019-04-26 02:30:34,2,2.0,0.0,"[0, 1]",998109,1891
8,Network_dataset_17.csv,1000000,18,NONE,0,2019-04-26 02:30:34,2019-04-26 05:27:11,1,65.0,0.0,"[0, 1]",966289,33711
9,Network_dataset_18.csv,1000000,18,NONE,0,2019-04-26 05:27:11,2019-04-26 17:54:05,0,22278.0,0.0,"[0, 1]",950564,49436



GLOBAL AUDIT
Total rows: 12,339,021
Files with schema issues: 0
Total bad timestamps: 0
Total non-monotonic timestamp transitions: 20
Observed label values: [0, 1]

⚠ Local timestamp reversals observed: 20
✓ Reversals independently audited
✓ Temporal construction will use timestamps, not row order
✓ No reversal crosses a ground-truth label transition
✓ Required schema verified
✓ Unix-second timestamps parsed successfully
✓ No invalid timestamps
✓ Ground-truth labels are binary {0, 1}


In [9]:
# ============================================================
# DRISHTI — BOOK 3
# CELL 5 — 30-SECOND TEMPORAL WINDOW CONSTRUCTION
# ============================================================

WINDOW_SECONDS = 30
HISTORY_WINDOWS = 10

print("=" * 70)
print("CONSTRUCTING TON-IoT 30-SECOND TEMPORAL WINDOWS")
print("=" * 70)

window_records = []

for file_id, path in enumerate(TON_FILES):

    print(f"\n[{file_id + 1:02d}/{len(TON_FILES):02d}] {path.name}")

    # Only load the fields needed for temporal ground truth.
    df = pd.read_csv(
        path,
        usecols=["ts", "label"],
        low_memory=False
    )

    # TON-IoT timestamps are Unix seconds.
    df["timestamp"] = pd.to_datetime(
        df["ts"],
        unit="s",
        errors="coerce"
    )

    # Remove invalid timestamps defensively.
    df = df.dropna(subset=["timestamp"]).copy()

    # --------------------------------------------------------
    # Assign each flow to a 30-second temporal window.
    # Timestamp, NOT CSV row order, determines membership.
    # --------------------------------------------------------

    df["window_start"] = df["timestamp"].dt.floor(
        f"{WINDOW_SECONDS}s"
    )

    # --------------------------------------------------------
    # Aggregate ground truth at window level.
    #
    # A window is attack-positive if ANY flow in that
    # 30-second window has label == 1.
    # --------------------------------------------------------

    grouped = (
        df.groupby("window_start", sort=True)
          .agg(
              attack=("label", lambda x: int((x == 1).any())),
              flow_count=("label", "size"),
              attack_flow_count=("label", lambda x: int((x == 1).sum()))
          )
          .reset_index()
    )

    grouped["file_id"] = file_id
    grouped["file"] = path.name

    # --------------------------------------------------------
    # Explicit temporal continuity.
    #
    # A transition is continuous only when two consecutive
    # observed windows are exactly 30 seconds apart.
    # --------------------------------------------------------

    grouped["delta_sec"] = (
        grouped["window_start"]
        .diff()
        .dt.total_seconds()
    )

    grouped["continuous_from_previous"] = (
        grouped["delta_sec"] == WINDOW_SECONDS
    )

    # First observed window has no previous window.
    grouped.loc[
        grouped.index[0],
        "continuous_from_previous"
    ] = False

    window_records.append(grouped)

    print(f"  Flows   : {len(df):,}")
    print(f"  Windows : {len(grouped):,}")
    print(f"  Attack windows : {grouped['attack'].sum():,}")

    del df, grouped
    gc.collect()

# ------------------------------------------------------------
# Combine file-level temporal windows.
# ------------------------------------------------------------

windows_df = (
    pd.concat(window_records, ignore_index=True)
      .sort_values(["file_id", "window_start"])
      .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("WINDOW CONSTRUCTION SUMMARY")
print("=" * 70)

print(f"Total 30-second windows : {len(windows_df):,}")
print(
    f"Attack windows          : "
    f"{int(windows_df['attack'].sum()):,}"
)
print(
    f"Benign windows          : "
    f"{int((windows_df['attack'] == 0).sum()):,}"
)
print(
    f"Continuous transitions  : "
    f"{int(windows_df['continuous_from_previous'].sum()):,}"
)

print("\n✓ Flows assigned using timestamps")
print("✓ 30-second temporal resolution established")
print("✓ Window attack ground truth constructed from label == 1")
print("✓ File boundaries preserved")
print("✓ Temporal continuity explicitly recorded")

CONSTRUCTING TON-IoT 30-SECOND TEMPORAL WINDOWS

[01/13] Network_dataset_1.csv
  Flows   : 1,000,000
  Windows : 5,677
  Attack windows : 147

[02/13] Network_dataset_10.csv
  Flows   : 1,000,000
  Windows : 278
  Attack windows : 278

[03/13] Network_dataset_11.csv
  Flows   : 1,000,000
  Windows : 394
  Attack windows : 388

[04/13] Network_dataset_12.csv
  Flows   : 1,000,000
  Windows : 645
  Attack windows : 625

[05/13] Network_dataset_13.csv
  Flows   : 1,000,000
  Windows : 356
  Attack windows : 356

[06/13] Network_dataset_14.csv
  Flows   : 1,000,000
  Windows : 255
  Attack windows : 255

[07/13] Network_dataset_15.csv
  Flows   : 1,000,000
  Windows : 262
  Attack windows : 262

[08/13] Network_dataset_16 (1).csv
  Flows   : 1,000,000
  Windows : 269
  Attack windows : 269

[09/13] Network_dataset_17.csv
  Flows   : 1,000,000
  Windows : 353
  Attack windows : 353

[10/13] Network_dataset_18.csv
  Flows   : 1,000,000
  Windows : 752
  Attack windows : 749

[11/13] Network_

In [10]:
# ============================================================
# DRISHTI — BOOK 3
# CELL 6 — GENUINE ATTACK ONSET CONSTRUCTION
# ============================================================

print("=" * 70)
print("CONSTRUCTING GENUINE ATTACK ONSETS")
print("=" * 70)

# Previous window must be:
#   1. benign
#   2. exactly 30 seconds before current window
#
# Current window must be attack-positive.

windows_df["genuine_onset"] = (
    (windows_df["attack"] == 1) &
    (windows_df["attack"].shift(1) == 0) &
    (windows_df["continuous_from_previous"]) &
    (windows_df["file_id"] == windows_df["file_id"].shift(1))
)

# ------------------------------------------------------------
# Prevent history from crossing a discontinuity.
#
# For each window, count how many immediately preceding
# 30-second windows are continuously available.
# ------------------------------------------------------------

windows_df["history_length"] = 0

for file_id, idx in windows_df.groupby("file_id").groups.items():

    idx = list(idx)

    history_count = 0

    for pos, row_idx in enumerate(idx):

        if pos == 0:
            history_count = 0

        else:
            prev_idx = idx[pos - 1]

            if (
                windows_df.loc[row_idx, "window_start"]
                - windows_df.loc[prev_idx, "window_start"]
            ).total_seconds() == WINDOW_SECONDS:

                history_count += 1

            else:
                history_count = 0

        windows_df.loc[row_idx, "history_length"] = history_count

# ------------------------------------------------------------
# A usable forecasting onset requires the complete
# 10-window / 5-minute historical context.
# ------------------------------------------------------------

windows_df["usable_onset"] = (
    windows_df["genuine_onset"] &
    (windows_df["history_length"] >= HISTORY_WINDOWS)
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

genuine_onsets = windows_df[windows_df["genuine_onset"]].copy()
usable_onsets = windows_df[windows_df["usable_onset"]].copy()

print(f"Total windows              : {len(windows_df):,}")
print(f"Genuine attack onsets      : {len(genuine_onsets):,}")
print(f"Onsets with 5-min history  : {len(usable_onsets):,}")

print("\nOnset distribution by file:")

onset_summary = (
    windows_df
    .groupby("file")
    .agg(
        windows=("attack", "size"),
        attack_windows=("attack", "sum"),
        genuine_onsets=("genuine_onset", "sum"),
        usable_onsets=("usable_onset", "sum")
    )
    .reset_index()
)

display(onset_summary)

print("\n" + "=" * 70)
print("ONSET AUDIT")
print("=" * 70)

display(
    usable_onsets[
        [
            "file",
            "window_start",
            "history_length",
            "attack",
            "genuine_onset",
            "usable_onset"
        ]
    ].head(20)
)

print("\n✓ Genuine onset definition applied")
print("✓ File boundaries preserved")
print("✓ Temporal continuity enforced")
print("✓ 5-minute historical context checked")

CONSTRUCTING GENUINE ATTACK ONSETS
Total windows              : 13,720
Genuine attack onsets      : 94
Onsets with 5-min history  : 72

Onset distribution by file:


,file,windows,attack_windows,genuine_onsets,usable_onsets
0,Network_dataset_1.csv,5677,147,0,0
1,Network_dataset_10.csv,278,278,0,0
2,Network_dataset_11.csv,394,388,5,4
3,Network_dataset_12.csv,645,625,11,11
4,Network_dataset_13.csv,356,356,0,0
5,Network_dataset_14.csv,255,255,0,0
6,Network_dataset_15.csv,262,262,0,0
7,Network_dataset_16 (1).csv,269,269,0,0
8,Network_dataset_17.csv,353,353,0,0
9,Network_dataset_18.csv,752,749,2,0



ONSET AUDIT


,file,window_start,history_length,attack,genuine_onset,usable_onset
6207,Network_dataset_11.csv,2019-04-25 05:54:30,11,1,True,True
6215,Network_dataset_11.csv,2019-04-25 05:58:30,19,1,True,True
6217,Network_dataset_11.csv,2019-04-25 05:59:30,21,1,True,True
6265,Network_dataset_11.csv,2019-04-25 06:23:30,69,1,True,True
6381,Network_dataset_12.csv,2019-04-25 07:21:00,32,1,True,True
6397,Network_dataset_12.csv,2019-04-25 07:29:00,48,1,True,True
6425,Network_dataset_12.csv,2019-04-25 07:43:00,76,1,True,True
6429,Network_dataset_12.csv,2019-04-25 07:45:00,80,1,True,True
6433,Network_dataset_12.csv,2019-04-25 07:47:00,84,1,True,True
6438,Network_dataset_12.csv,2019-04-25 07:49:30,89,1,True,True



✓ Genuine onset definition applied
✓ File boundaries preserved
✓ Temporal continuity enforced
✓ 5-minute historical context checked


In [11]:
# ============================================================
# DRISHTI — BOOK 3
# CELL 7 — TON-IoT STATE FEATURE CONSTRUCTION
# ============================================================

print("=" * 70)
print("CONSTRUCTING TON-IoT STATE FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# State feature groups
# ------------------------------------------------------------

STATE_NUMERIC_COLUMNS = [
    "duration",
    "src_bytes",
    "dst_bytes",
    "missed_bytes",
    "src_pkts",
    "src_ip_bytes",
    "dst_pkts",
    "dst_ip_bytes",
]

STATE_CATEGORICAL_COLUMNS = [
    "proto",
    "service",
    "conn_state",
]

print("Numeric state columns:")
for col in STATE_NUMERIC_COLUMNS:
    print("  •", col)

print("\nCategorical state columns:")
for col in STATE_CATEGORICAL_COLUMNS:
    print("  •", col)


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").fillna(0.0)


state_records = []

for file_id, path in enumerate(TON_FILES):

    print(f"\n[{file_id + 1:02d}/{len(TON_FILES):02d}] {path.name}")

    usecols = [
        "ts",
        "duration",
        "src_bytes",
        "dst_bytes",
        "missed_bytes",
        "src_pkts",
        "src_ip_bytes",
        "dst_pkts",
        "dst_ip_bytes",
        "proto",
        "service",
        "conn_state",
    ]

    df = pd.read_csv(
        path,
        usecols=usecols,
        low_memory=False
    )

    # --------------------------------------------------------
    # Timestamp → 30-second window
    # --------------------------------------------------------

    df["timestamp"] = pd.to_datetime(
        df["ts"],
        unit="s",
        errors="coerce"
    )

    df = df.dropna(subset=["timestamp"]).copy()

    df["window_start"] = df["timestamp"].dt.floor(
        f"{WINDOW_SECONDS}s"
    )

    # --------------------------------------------------------
    # Numeric telemetry
    # --------------------------------------------------------

    for col in STATE_NUMERIC_COLUMNS:
        df[col] = safe_numeric(df[col])

    # --------------------------------------------------------
    # Aggregate numeric state features.
    #
    # Include file_id/file directly in the grouping key so
    # the final feature table has an unambiguous temporal key.
    # --------------------------------------------------------

    numeric_agg = (
        df.groupby(
            ["window_start"],
            sort=True
        )[STATE_NUMERIC_COLUMNS]
        .agg(["mean", "std", "min", "max", "sum"])
    )

    # A window containing exactly one observation has an
    # undefined sample standard deviation in pandas.
    # For a single observation, within-window variability
    # is defined as zero.
    numeric_agg = numeric_agg.fillna(0.0)

    numeric_agg.columns = [
        f"STATE_{col}_{stat}"
        for col, stat in numeric_agg.columns
    ]

    numeric_df = numeric_agg.reset_index()

    numeric_df["file_id"] = file_id
    numeric_df["file"] = path.name

    # --------------------------------------------------------
    # Categorical state features
    # --------------------------------------------------------

    categorical_records = []

    for window_start, g in df.groupby("window_start", sort=True):

        record = {
            "window_start": window_start,
            "file_id": file_id,
            "file": path.name,
        }

        for col in STATE_CATEGORICAL_COLUMNS:

            values = (
                g[col]
                .fillna("UNKNOWN")
                .astype(str)
            )

            counts = values.value_counts()

            record[f"STATE_{col}_nunique"] = int(
                values.nunique()
            )

            record[f"STATE_{col}_dominant_frac"] = float(
                counts.iloc[0] / len(values)
            )

        categorical_records.append(record)

    categorical_df = pd.DataFrame(categorical_records)

    # --------------------------------------------------------
    # Merge numeric + categorical features.
    # --------------------------------------------------------

    state_df = numeric_df.merge(
        categorical_df,
        on=["window_start", "file_id", "file"],
        how="inner",
        validate="one_to_one"
    )

    state_records.append(state_df)

    print(
        f"  Windows represented: {len(state_df):,}"
    )

    del df
    del numeric_agg
    del numeric_df
    del categorical_df
    del state_df

    gc.collect()


# ------------------------------------------------------------
# Combine all files
# ------------------------------------------------------------

state_features_df = (
    pd.concat(state_records, ignore_index=True)
      .sort_values(["file_id", "window_start"])
      .reset_index(drop=True)
)

STATE_FEATURE_COLUMNS = [
    c
    for c in state_features_df.columns
    if c.startswith("STATE_")
]

print("\n" + "=" * 70)
print("STATE FEATURE SUMMARY")
print("=" * 70)

print(f"Windows        : {len(state_features_df):,}")
print(f"State features : {len(STATE_FEATURE_COLUMNS)}")

print("\nMissing feature values:")
missing_total = int(
    state_features_df[STATE_FEATURE_COLUMNS]
    .isna()
    .sum()
    .sum()
)

print(missing_total)

print("\nSample:")
display(
    state_features_df.head()
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(state_features_df) == len(windows_df), \
    "State feature window count does not match temporal windows."

assert not state_features_df[
    ["file_id", "window_start"]
].duplicated().any(), \
    "Duplicate file/window keys detected."

assert missing_total == 0, \
    "Missing values detected in state features."

print("\n✓ State features constructed")
print("✓ Timestamp-based 30-second aggregation")
print("✓ No label/type predictors used")
print("✓ One feature row per file/window")
print("✓ State representation aligned to temporal windows")

CONSTRUCTING TON-IoT STATE FEATURES
Numeric state columns:
  • duration
  • src_bytes
  • dst_bytes
  • missed_bytes
  • src_pkts
  • src_ip_bytes
  • dst_pkts
  • dst_ip_bytes

Categorical state columns:
  • proto
  • service
  • conn_state

[01/13] Network_dataset_1.csv
  Windows represented: 5,677

[02/13] Network_dataset_10.csv
  Windows represented: 278

[03/13] Network_dataset_11.csv
  Windows represented: 394

[04/13] Network_dataset_12.csv
  Windows represented: 645

[05/13] Network_dataset_13.csv
  Windows represented: 356

[06/13] Network_dataset_14.csv
  Windows represented: 255

[07/13] Network_dataset_15.csv
  Windows represented: 262

[08/13] Network_dataset_16 (1).csv
  Windows represented: 269

[09/13] Network_dataset_17.csv
  Windows represented: 353

[10/13] Network_dataset_18.csv
  Windows represented: 752

[11/13] Network_dataset_20.csv
  Windows represented: 555

[12/13] Network_dataset_22.csv
  Windows represented: 2,148

[13/13] Network_dataset_23.csv
  Windows r

,window_start,STATE_duration_mean,STATE_duration_std,STATE_duration_min,STATE_duration_max,STATE_duration_sum,STATE_src_bytes_mean,STATE_src_bytes_std,STATE_src_bytes_min,STATE_src_bytes_max,...,STATE_dst_ip_bytes_max,STATE_dst_ip_bytes_sum,file_id,file,STATE_proto_nunique,STATE_proto_dominant_frac,STATE_service_nunique,STATE_service_dominant_frac,STATE_conn_state_nunique,STATE_conn_state_dominant_frac
0,2019-04-02 09:45:30,20137.382565,40274.765130,0.0,80549.530260,80549.530260,440713.000000,881426.000000,0.0,1762852.0,...,236,236,0,Network_dataset_1.csv,2,0.750000,1,1.000000,2,0.750000
1,2019-04-02 09:46:00,0.000055,0.000165,0.0,0.000549,0.001048,0.000000,0.000000,0.0,0.0,...,354,708,0,Network_dataset_1.csv,2,0.631579,2,0.894737,3,0.631579
2,2019-04-02 09:46:30,0.002335,0.007385,0.0,0.028326,0.049026,0.000000,0.000000,0.0,0.0,...,354,968,0,Network_dataset_1.csv,2,0.666667,2,0.904762,3,0.571429
3,2019-04-02 09:47:00,0.000079,0.000240,0.0,0.000881,0.001496,0.000000,0.000000,0.0,0.0,...,354,708,0,Network_dataset_1.csv,2,0.631579,2,0.894737,3,0.631579
4,2019-04-02 09:47:30,13.837961,64.018947,0.0,300.439245,304.435133,15.272727,71.635441,0.0,336.0,...,354,978,0,Network_dataset_1.csv,2,0.545455,2,0.863636,3,0.590909



✓ State features constructed
✓ Timestamp-based 30-second aggregation
✓ No label/type predictors used
✓ One feature row per file/window
✓ State representation aligned to temporal windows


In [12]:
# ============================================================
# DRISHTI — BOOK 3
# CELL 8 — TON-IoT STRUCTURE FEATURE CONSTRUCTION
# ============================================================

print("=" * 70)
print("CONSTRUCTING TON-IoT STRUCTURE FEATURES")
print("=" * 70)

STRUCTURE_COLUMNS = [
    "src_ip",
    "dst_ip",
    "src_port",
    "dst_port",
    "proto",
    "service",
    "conn_state",
]

STRUCTURE_USECOLS = [
    "ts",
    "src_ip",
    "dst_ip",
    "src_port",
    "dst_port",
    "proto",
    "service",
    "conn_state",
]

# ------------------------------------------------------------
# Entropy helper
# ------------------------------------------------------------

def normalized_entropy(series):
    """
    Normalized Shannon entropy in [0, 1].

    0 -> completely concentrated
    1 -> maximally diverse for the observed cardinality
    """

    values = (
        series
        .fillna("UNKNOWN")
        .astype(str)
    )

    counts = values.value_counts()

    if len(counts) <= 1:
        return 0.0

    probabilities = counts.values / counts.values.sum()

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    max_entropy = np.log2(len(counts))

    if max_entropy == 0:
        return 0.0

    return float(entropy / max_entropy)


def dominant_fraction(series):
    """Fraction belonging to the most common category."""

    values = (
        series
        .fillna("UNKNOWN")
        .astype(str)
    )

    counts = values.value_counts()

    if len(counts) == 0:
        return 0.0

    return float(counts.iloc[0] / len(values))


structure_records = []

for file_id, path in enumerate(TON_FILES):

    print(f"\n[{file_id + 1:02d}/{len(TON_FILES):02d}] {path.name}")

    df = pd.read_csv(
        path,
        usecols=STRUCTURE_USECOLS,
        low_memory=False
    )

    # --------------------------------------------------------
    # Timestamp → 30-second temporal window
    # --------------------------------------------------------

    df["timestamp"] = pd.to_datetime(
        df["ts"],
        unit="s",
        errors="coerce"
    )

    df = df.dropna(subset=["timestamp"]).copy()

    df["window_start"] = df["timestamp"].dt.floor(
        f"{WINDOW_SECONDS}s"
    )

    # --------------------------------------------------------
    # Structure aggregation
    # --------------------------------------------------------

    grouped = df.groupby(
        "window_start",
        sort=True
    )

    records = []

    for window_start, g in grouped:

        record = {
            "window_start": window_start,
            "file_id": file_id,
            "file": path.name,
        }

        # ----------------------------------------------------
        # Cardinality / diversity
        # ----------------------------------------------------

        for col in STRUCTURE_COLUMNS:

            values = (
                g[col]
                .fillna("UNKNOWN")
                .astype(str)
            )

            record[f"STRUCT_{col}_nunique"] = int(
                values.nunique()
            )

            record[f"STRUCT_{col}_entropy"] = (
                normalized_entropy(values)
            )

            record[f"STRUCT_{col}_dominant_frac"] = (
                dominant_fraction(values)
            )

        # ----------------------------------------------------
        # Explicit source/destination diversity
        # ----------------------------------------------------

        src_values = (
            g["src_ip"]
            .fillna("UNKNOWN")
            .astype(str)
        )

        dst_values = (
            g["dst_ip"]
            .fillna("UNKNOWN")
            .astype(str)
        )

        record["STRUCT_src_dst_unique_ratio"] = (
            float(src_values.nunique()) /
            max(dst_values.nunique(), 1)
        )

        # ----------------------------------------------------
        # Source concentration
        # ----------------------------------------------------

        src_counts = src_values.value_counts()

        record["STRUCT_src_top1_frac"] = float(
            src_counts.iloc[0] / len(g)
        )

        record["STRUCT_src_top5_frac"] = float(
            src_counts.head(5).sum() / len(g)
        )

        # ----------------------------------------------------
        # Destination concentration
        # ----------------------------------------------------

        dst_counts = dst_values.value_counts()

        record["STRUCT_dst_top1_frac"] = float(
            dst_counts.iloc[0] / len(g)
        )

        record["STRUCT_dst_top5_frac"] = float(
            dst_counts.head(5).sum() / len(g)
        )

        # ----------------------------------------------------
        # Port-range / privileged-port indicators
        # ----------------------------------------------------

        src_ports = pd.to_numeric(
            g["src_port"],
            errors="coerce"
        )

        dst_ports = pd.to_numeric(
            g["dst_port"],
            errors="coerce"
        )

        record["STRUCT_src_privileged_port_frac"] = float(
            (src_ports < 1024).fillna(False).mean()
        )

        record["STRUCT_dst_privileged_port_frac"] = float(
            (dst_ports < 1024).fillna(False).mean()
        )

        records.append(record)

    structure_records.append(
        pd.DataFrame(records)
    )

    print(
        f"  Windows represented: {len(records):,}"
    )

    del df
    del grouped
    del records

    gc.collect()


# ------------------------------------------------------------
# Combine all structure features
# ------------------------------------------------------------

structure_features_df = (
    pd.concat(structure_records, ignore_index=True)
      .sort_values(["file_id", "window_start"])
      .reset_index(drop=True)
)

STRUCTURE_FEATURE_COLUMNS = [
    c
    for c in structure_features_df.columns
    if c.startswith("STRUCT_")
]

print("\n" + "=" * 70)
print("STRUCTURE FEATURE SUMMARY")
print("=" * 70)

print(
    f"Windows           : "
    f"{len(structure_features_df):,}"
)

print(
    f"Structure features: "
    f"{len(STRUCTURE_FEATURE_COLUMNS)}"
)

missing_total = int(
    structure_features_df[
        STRUCTURE_FEATURE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
)

print(
    f"Missing values    : "
    f"{missing_total}"
)

display(
    structure_features_df.head()
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(structure_features_df) == len(windows_df), \
    "Structure feature window count mismatch."

assert not structure_features_df[
    ["file_id", "window_start"]
].duplicated().any(), \
    "Duplicate structure feature keys detected."

assert missing_total == 0, \
    "Missing values detected in structure features."

print("\n✓ Structure features constructed")
print("✓ Timestamp-based 30-second aggregation")
print("✓ Diversity and concentration measures included")
print("✓ No label/type predictors used")
print("✓ One feature row per file/window")

CONSTRUCTING TON-IoT STRUCTURE FEATURES

[01/13] Network_dataset_1.csv
  Windows represented: 5,677

[02/13] Network_dataset_10.csv
  Windows represented: 278

[03/13] Network_dataset_11.csv
  Windows represented: 394

[04/13] Network_dataset_12.csv
  Windows represented: 645

[05/13] Network_dataset_13.csv
  Windows represented: 356

[06/13] Network_dataset_14.csv
  Windows represented: 255

[07/13] Network_dataset_15.csv
  Windows represented: 262

[08/13] Network_dataset_16 (1).csv
  Windows represented: 269

[09/13] Network_dataset_17.csv
  Windows represented: 353

[10/13] Network_dataset_18.csv
  Windows represented: 752

[11/13] Network_dataset_20.csv
  Windows represented: 555

[12/13] Network_dataset_22.csv
  Windows represented: 2,148

[13/13] Network_dataset_23.csv
  Windows represented: 1,776

STRUCTURE FEATURE SUMMARY
Windows           : 13,720
Structure features: 28
Missing values    : 0


,window_start,file_id,file,STRUCT_src_ip_nunique,STRUCT_src_ip_entropy,STRUCT_src_ip_dominant_frac,STRUCT_dst_ip_nunique,STRUCT_dst_ip_entropy,STRUCT_dst_ip_dominant_frac,STRUCT_src_port_nunique,...,STRUCT_conn_state_nunique,STRUCT_conn_state_entropy,STRUCT_conn_state_dominant_frac,STRUCT_src_dst_unique_ratio,STRUCT_src_top1_frac,STRUCT_src_top5_frac,STRUCT_dst_top1_frac,STRUCT_dst_top5_frac,STRUCT_src_privileged_port_frac,STRUCT_dst_privileged_port_frac
0,2019-04-02 09:45:30,0,Network_dataset_1.csv,3,0.946395,0.500000,2,0.811278,0.750000,4,...,2,0.811278,0.750000,1.500000,0.500000,1.0,0.750000,1.0,0.000000,0.000000
1,2019-04-02 09:46:00,0,Network_dataset_1.csv,2,0.831474,0.736842,3,0.799668,0.631579,9,...,3,0.799668,0.631579,0.666667,0.736842,1.0,0.631579,1.0,0.000000,0.105263
2,2019-04-02 09:46:30,0,Network_dataset_1.csv,2,0.791858,0.761905,5,0.730294,0.571429,11,...,3,0.889595,0.571429,0.400000,0.761905,1.0,0.571429,1.0,0.000000,0.190476
3,2019-04-02 09:47:00,0,Network_dataset_1.csv,2,0.831474,0.736842,3,0.799668,0.631579,10,...,3,0.799668,0.631579,0.666667,0.736842,1.0,0.631579,1.0,0.000000,0.105263
4,2019-04-02 09:47:30,0,Network_dataset_1.csv,4,0.633966,0.681818,5,0.793697,0.500000,12,...,3,0.813047,0.590909,0.800000,0.681818,1.0,0.500000,1.0,0.045455,0.181818



✓ Structure features constructed
✓ Timestamp-based 30-second aggregation
✓ Diversity and concentration measures included
✓ No label/type predictors used
✓ One feature row per file/window


In [14]:
# ============================================================
# CELL 9 — COMMUNICATION TOPOLOGY FEATURES
# ============================================================

print("=" * 70)
print("DRISHTI — BOOK 3 | COMMUNICATION TOPOLOGY")
print("=" * 70)


def entropy_normalized(values):
    """
    Normalized Shannon entropy in [0, 1].
    Returns 0 for empty / single-category distributions.
    """
    counts = pd.Series(values).value_counts()
    
    if len(counts) <= 1:
        return 0.0
    
    p = counts.to_numpy(dtype=float)
    p = p / p.sum()
    
    h = -(p * np.log2(p)).sum()
    h_max = np.log2(len(p))
    
    return float(h / h_max) if h_max > 0 else 0.0


def concentration_top_k(values, k=5):
    """
    Fraction of observations covered by the k most frequent values.
    """
    counts = pd.Series(values).value_counts()
    
    if len(counts) == 0:
        return 0.0
    
    return float(counts.head(k).sum() / counts.sum())


topology_records = []

for file_id, file_path in enumerate(TON_FILES):
    
    usecols = [
        "ts",
        "src_ip",
        "dst_ip",
        "src_port",
        "dst_port",
        "proto",
    ]
    
    parts = []
    
    for chunk in pd.read_csv(
        file_path,
        usecols=usecols,
        chunksize=250_000,
        low_memory=False
    ):
        parts.append(chunk)
    
    df = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()
    
    # --------------------------------------------------------
    # Timestamp-based windowing
    # --------------------------------------------------------
    df["timestamp"] = pd.to_datetime(
        df["ts"],
        unit="s",
        errors="coerce"
    )
    
    assert df["timestamp"].notna().all()
    
    df["window_start"] = df["timestamp"].dt.floor("30s")
    
    # --------------------------------------------------------
    # Normalize topology fields
    # --------------------------------------------------------
    for col in ["src_ip", "dst_ip", "proto"]:
        df[col] = df[col].fillna("UNKNOWN").astype(str)
    
    for col in ["src_port", "dst_port"]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(-1)
    
    # Communication edge
    df["edge"] = (
        df["src_ip"]
        + "->"
        + df["dst_ip"]
    )
    
    # Source-destination-port edge
    df["service_edge"] = (
        df["src_ip"]
        + "->"
        + df["dst_ip"]
        + ":"
        + df["dst_port"].astype(str)
    )
    
    # Protocol-conditioned communication edge
    df["proto_edge"] = (
        df["src_ip"]
        + "->"
        + df["dst_ip"]
        + "|"
        + df["proto"]
    )
    
    # --------------------------------------------------------
    # Window-level topology aggregation
    # --------------------------------------------------------
    for window_start, g in df.groupby(
        "window_start",
        sort=True
    ):
        
        sources = g["src_ip"]
        destinations = g["dst_ip"]
        edges = g["edge"]
        service_edges = g["service_edge"]
        proto_edges = g["proto_edge"]
        
        source_counts = sources.value_counts()
        destination_counts = destinations.value_counts()
        edge_counts = edges.value_counts()
        
        n_flows = len(g)
        
        unique_sources = sources.nunique()
        unique_destinations = destinations.nunique()
        unique_edges = edges.nunique()
        
        # Source fan-out:
        # number of distinct destinations contacted per source
        source_fanout = (
            g.groupby("src_ip")["dst_ip"]
            .nunique()
        )
        
        # Destination fan-in:
        # number of distinct sources contacting each destination
        destination_fanin = (
            g.groupby("dst_ip")["src_ip"]
            .nunique()
        )
        
        topology_records.append({
            "file_id": file_id,
            "window_start": window_start,
            
            # Basic graph size
            "TOPO_UNIQUE_SOURCES": unique_sources,
            "TOPO_UNIQUE_DESTINATIONS": unique_destinations,
            "TOPO_UNIQUE_EDGES": unique_edges,
            "TOPO_UNIQUE_SERVICE_EDGES": service_edges.nunique(),
            "TOPO_UNIQUE_PROTO_EDGES": proto_edges.nunique(),
            
            # Communication density
            "TOPO_EDGE_DENSITY": (
                unique_edges / max(unique_sources * unique_destinations, 1)
            ),
            
            # Diversity
            "TOPO_EDGE_DIVERSITY": (
                unique_edges / max(n_flows, 1)
            ),
            
            "TOPO_SERVICE_EDGE_DIVERSITY": (
                service_edges.nunique() / max(n_flows, 1)
            ),
            
            "TOPO_PROTO_EDGE_DIVERSITY": (
                proto_edges.nunique() / max(n_flows, 1)
            ),
            
            # Concentration
            "TOPO_SOURCE_TOP1_CONCENTRATION": (
                concentration_top_k(sources, 1)
            ),
            "TOPO_SOURCE_TOP5_CONCENTRATION": (
                concentration_top_k(sources, 5)
            ),
            "TOPO_DEST_TOP1_CONCENTRATION": (
                concentration_top_k(destinations, 1)
            ),
            "TOPO_DEST_TOP5_CONCENTRATION": (
                concentration_top_k(destinations, 5)
            ),
            "TOPO_EDGE_TOP1_CONCENTRATION": (
                concentration_top_k(edges, 1)
            ),
            "TOPO_EDGE_TOP5_CONCENTRATION": (
                concentration_top_k(edges, 5)
            ),
            
            # Entropy
            "TOPO_SOURCE_ENTROPY": (
                entropy_normalized(sources)
            ),
            "TOPO_DEST_ENTROPY": (
                entropy_normalized(destinations)
            ),
            "TOPO_EDGE_ENTROPY": (
                entropy_normalized(edges)
            ),
            
            # Fan-out / fan-in
            "TOPO_SOURCE_FANOUT_MEAN": (
                source_fanout.mean()
            ),
            "TOPO_SOURCE_FANOUT_MAX": (
                source_fanout.max()
            ),
            "TOPO_SOURCE_FANOUT_STD": (
                source_fanout.std(ddof=0)
            ),
            
            "TOPO_DEST_FANIN_MEAN": (
                destination_fanin.mean()
            ),
            "TOPO_DEST_FANIN_MAX": (
                destination_fanin.max()
            ),
            "TOPO_DEST_FANIN_STD": (
                destination_fanin.std(ddof=0)
            ),
        })
    
    del df
    gc.collect()
    
    print(
        f"✓ {file_path.name:<30} "
        f"→ topology windows accumulated: {len(topology_records):,}"
    )


topology_features_df = pd.DataFrame(topology_records)

topology_features_df = topology_features_df.sort_values(
    ["file_id", "window_start"]
).reset_index(drop=True)

TOPOLOGY_FEATURE_COLUMNS = [
    c for c in topology_features_df.columns
    if c.startswith("TOPO_")
]

print("\n" + "=" * 70)
print("TOPOLOGY FEATURE SUMMARY")
print("=" * 70)

print(f"Windows          : {len(topology_features_df):,}")
print(f"Topology features: {len(TOPOLOGY_FEATURE_COLUMNS)}")
print(
    f"Missing values   : "
    f"{int(topology_features_df[TOPOLOGY_FEATURE_COLUMNS].isna().sum().sum())}"
)

print("\nFeature names:")
for c in TOPOLOGY_FEATURE_COLUMNS:
    print("  •", c)

print("\nSample:")
display(
    topology_features_df[
        ["file_id", "window_start"] + TOPOLOGY_FEATURE_COLUMNS
    ].head()
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

EXPECTED_WINDOWS = 13_720

assert len(topology_features_df) == EXPECTED_WINDOWS, (
    f"Unexpected topology window count: "
    f"{len(topology_features_df)} vs {EXPECTED_WINDOWS}"
)

assert topology_features_df[
    TOPOLOGY_FEATURE_COLUMNS
].isna().sum().sum() == 0

assert topology_features_df[
    "TOPO_UNIQUE_SOURCES"
].ge(0).all()

assert topology_features_df[
    "TOPO_UNIQUE_DESTINATIONS"
].ge(0).all()

assert topology_features_df[
    "TOPO_UNIQUE_EDGES"
].ge(0).all()

assert topology_features_df[
    "TOPO_EDGE_DIVERSITY"
].between(0, 1).all()

assert topology_features_df[
    "TOPO_EDGE_DENSITY"
].ge(0).all()

print("\n✓ Window count matches temporal audit: 13,720")
print("✓ No missing topology features")
print("✓ Graph statistics are non-negative")
print("✓ Diversity metrics are bounded")
print("✓ No labels or target columns used")
print("✓ Communication topology construction complete")

DRISHTI — BOOK 3 | COMMUNICATION TOPOLOGY
✓ Network_dataset_1.csv          → topology windows accumulated: 5,677
✓ Network_dataset_10.csv         → topology windows accumulated: 5,955
✓ Network_dataset_11.csv         → topology windows accumulated: 6,349
✓ Network_dataset_12.csv         → topology windows accumulated: 6,994
✓ Network_dataset_13.csv         → topology windows accumulated: 7,350
✓ Network_dataset_14.csv         → topology windows accumulated: 7,605
✓ Network_dataset_15.csv         → topology windows accumulated: 7,867
✓ Network_dataset_16 (1).csv     → topology windows accumulated: 8,136
✓ Network_dataset_17.csv         → topology windows accumulated: 8,489
✓ Network_dataset_18.csv         → topology windows accumulated: 9,241
✓ Network_dataset_20.csv         → topology windows accumulated: 9,796
✓ Network_dataset_22.csv         → topology windows accumulated: 11,944
✓ Network_dataset_23.csv         → topology windows accumulated: 13,720

TOPOLOGY FEATURE SUMMARY
Windows

,file_id,window_start,TOPO_UNIQUE_SOURCES,TOPO_UNIQUE_DESTINATIONS,TOPO_UNIQUE_EDGES,TOPO_UNIQUE_SERVICE_EDGES,TOPO_UNIQUE_PROTO_EDGES,TOPO_EDGE_DENSITY,TOPO_EDGE_DIVERSITY,TOPO_SERVICE_EDGE_DIVERSITY,...,TOPO_EDGE_TOP5_CONCENTRATION,TOPO_SOURCE_ENTROPY,TOPO_DEST_ENTROPY,TOPO_EDGE_ENTROPY,TOPO_SOURCE_FANOUT_MEAN,TOPO_SOURCE_FANOUT_MAX,TOPO_SOURCE_FANOUT_STD,TOPO_DEST_FANIN_MEAN,TOPO_DEST_FANIN_MAX,TOPO_DEST_FANIN_STD
0,0,2019-04-02 09:45:30,3,2,3,4,3,0.50,0.750000,1.000000,...,1.000000,0.946395,0.811278,0.946395,1.00,1,0.000000,1.5,2,0.500000
1,0,2019-04-02 09:46:00,2,3,3,4,3,0.50,0.157895,0.210526,...,1.000000,0.831474,0.799668,0.799668,1.50,2,0.500000,1.0,1,0.000000
2,0,2019-04-02 09:46:30,2,5,5,6,5,0.50,0.238095,0.285714,...,1.000000,0.791858,0.730294,0.730294,2.50,4,1.500000,1.0,1,0.000000
3,0,2019-04-02 09:47:00,2,3,3,5,3,0.50,0.157895,0.263158,...,1.000000,0.831474,0.799668,0.799668,1.50,2,0.500000,1.0,1,0.000000
4,0,2019-04-02 09:47:30,4,5,7,8,7,0.35,0.318182,0.363636,...,0.909091,0.633966,0.793697,0.751987,1.75,4,1.299038,1.4,2,0.489898



✓ Window count matches temporal audit: 13,720
✓ No missing topology features
✓ Graph statistics are non-negative
✓ Diversity metrics are bounded
✓ No labels or target columns used
✓ Communication topology construction complete


In [15]:
# ============================================================
# CELL 10 — PACKET / FLOW TELEMETRY FEATURES
# ============================================================

print("=" * 70)
print("DRISHTI — BOOK 3 | PACKET / FLOW TELEMETRY")
print("=" * 70)

TELEMETRY_REQUIRED_COLUMNS = [
    "ts",
    "duration",
    "src_bytes",
    "dst_bytes",
    "src_pkts",
    "dst_pkts",
    "src_ip_bytes",
    "dst_ip_bytes",
    "missed_bytes",
]

telemetry_records = []

for file_id, file_path in enumerate(TON_FILES):

    usecols = [
        "ts",
        "duration",
        "src_bytes",
        "dst_bytes",
        "src_pkts",
        "dst_pkts",
        "src_ip_bytes",
        "dst_ip_bytes",
        "missed_bytes",
    ]

    parts = []

    for chunk in pd.read_csv(
        file_path,
        usecols=usecols,
        chunksize=250_000,
        low_memory=False
    ):
        parts.append(chunk)

    df = pd.concat(parts, ignore_index=True)

    del parts
    gc.collect()

    # --------------------------------------------------------
    # Timestamp-based 30-second windows
    # --------------------------------------------------------

    df["timestamp"] = pd.to_datetime(
        df["ts"],
        unit="s",
        errors="coerce"
    )

    assert df["timestamp"].notna().all()

    df["window_start"] = df["timestamp"].dt.floor("30s")

    # --------------------------------------------------------
    # Numeric cleanup
    # --------------------------------------------------------

    numeric_cols = [
        "duration",
        "src_bytes",
        "dst_bytes",
        "src_pkts",
        "dst_pkts",
        "src_ip_bytes",
        "dst_ip_bytes",
        "missed_bytes",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0.0)

    # --------------------------------------------------------
    # Derived per-flow telemetry
    # --------------------------------------------------------

    # Total bytes / packets
    df["total_bytes"] = (
        df["src_bytes"] +
        df["dst_bytes"]
    )

    df["total_pkts"] = (
        df["src_pkts"] +
        df["dst_pkts"]
    )

    # Bytes and packets per packet / flow
    df["bytes_per_pkt"] = (
        df["total_bytes"] /
        df["total_pkts"].replace(0, np.nan)
    ).fillna(0.0)

    # Source / destination asymmetry
    df["byte_asymmetry"] = (
        np.abs(df["src_bytes"] - df["dst_bytes"]) /
        (df["src_bytes"] + df["dst_bytes"] + 1.0)
    )

    df["pkt_asymmetry"] = (
        np.abs(df["src_pkts"] - df["dst_pkts"]) /
        (df["src_pkts"] + df["dst_pkts"] + 1.0)
    )

    # Source / destination shares
    df["src_byte_share"] = (
        df["src_bytes"] /
        (df["total_bytes"] + 1.0)
    )

    df["src_pkt_share"] = (
        df["src_pkts"] /
        (df["total_pkts"] + 1.0)
    )

    # --------------------------------------------------------
    # Window aggregation
    # --------------------------------------------------------

    for window_start, g in df.groupby(
        "window_start",
        sort=True
    ):

        n_flows = len(g)

        total_bytes = g["total_bytes"].sum()
        total_pkts = g["total_pkts"].sum()

        telemetry_records.append({
            "file_id": file_id,
            "window_start": window_start,

            # ------------------------------------------------
            # Flow volume
            # ------------------------------------------------
            "TELEM_FLOW_COUNT": n_flows,

            # ------------------------------------------------
            # Duration
            # ------------------------------------------------
            "TELEM_DURATION_MEAN": g["duration"].mean(),
            "TELEM_DURATION_STD": g["duration"].std(ddof=0),
            "TELEM_DURATION_MIN": g["duration"].min(),
            "TELEM_DURATION_MAX": g["duration"].max(),

            # ------------------------------------------------
            # Bytes
            # ------------------------------------------------
            "TELEM_TOTAL_BYTES": total_bytes,
            "TELEM_SRC_BYTES": g["src_bytes"].sum(),
            "TELEM_DST_BYTES": g["dst_bytes"].sum(),

            "TELEM_BYTES_MEAN": g["total_bytes"].mean(),
            "TELEM_BYTES_STD": g["total_bytes"].std(ddof=0),
            "TELEM_BYTES_MIN": g["total_bytes"].min(),
            "TELEM_BYTES_MAX": g["total_bytes"].max(),

            # ------------------------------------------------
            # Packets
            # ------------------------------------------------
            "TELEM_TOTAL_PKTS": total_pkts,
            "TELEM_SRC_PKTS": g["src_pkts"].sum(),
            "TELEM_DST_PKTS": g["dst_pkts"].sum(),

            "TELEM_PKTS_MEAN": g["total_pkts"].mean(),
            "TELEM_PKTS_STD": g["total_pkts"].std(ddof=0),
            "TELEM_PKTS_MIN": g["total_pkts"].min(),
            "TELEM_PKTS_MAX": g["total_pkts"].max(),

            # ------------------------------------------------
            # IP-byte telemetry
            # ------------------------------------------------
            "TELEM_SRC_IP_BYTES": g["src_ip_bytes"].sum(),
            "TELEM_DST_IP_BYTES": g["dst_ip_bytes"].sum(),

            # ------------------------------------------------
            # Missed bytes
            # ------------------------------------------------
            "TELEM_MISSED_BYTES_TOTAL": g["missed_bytes"].sum(),
            "TELEM_MISSED_BYTES_MEAN": g["missed_bytes"].mean(),
            "TELEM_MISSED_BYTES_MAX": g["missed_bytes"].max(),

            # ------------------------------------------------
            # Efficiency / rate-style measures
            # ------------------------------------------------
            "TELEM_BYTES_PER_PKT_MEAN": g["bytes_per_pkt"].mean(),
            "TELEM_BYTES_PER_PKT_STD": g["bytes_per_pkt"].std(ddof=0),

            # ------------------------------------------------
            # Directional asymmetry
            # ------------------------------------------------
            "TELEM_BYTE_ASYMMETRY_MEAN": g["byte_asymmetry"].mean(),
            "TELEM_BYTE_ASYMMETRY_STD": g["byte_asymmetry"].std(ddof=0),

            "TELEM_PKT_ASYMMETRY_MEAN": g["pkt_asymmetry"].mean(),
            "TELEM_PKT_ASYMMETRY_STD": g["pkt_asymmetry"].std(ddof=0),

            # ------------------------------------------------
            # Source share
            # ------------------------------------------------
            "TELEM_SRC_BYTE_SHARE_MEAN": g["src_byte_share"].mean(),
            "TELEM_SRC_PKT_SHARE_MEAN": g["src_pkt_share"].mean(),
        })

    del df
    gc.collect()

    print(
        f"✓ {file_path.name:<30} "
        f"→ telemetry windows accumulated: "
        f"{len(telemetry_records):,}"
    )


# ============================================================
# Build final telemetry dataframe
# ============================================================

telemetry_features_df = pd.DataFrame(
    telemetry_records
)

telemetry_features_df = telemetry_features_df.sort_values(
    ["file_id", "window_start"]
).reset_index(drop=True)

TELEMETRY_FEATURE_COLUMNS = [
    c for c in telemetry_features_df.columns
    if c.startswith("TELEM_")
]


print("\n" + "=" * 70)
print("TELEMETRY FEATURE SUMMARY")
print("=" * 70)

print(
    f"Windows           : "
    f"{len(telemetry_features_df):,}"
)

print(
    f"Telemetry features: "
    f"{len(TELEMETRY_FEATURE_COLUMNS)}"
)

print(
    f"Missing values    : "
    f"{int(telemetry_features_df[TELEMETRY_FEATURE_COLUMNS].isna().sum().sum())}"
)

print("\nFeature names:")

for c in TELEMETRY_FEATURE_COLUMNS:
    print("  •", c)

print("\nSample:")

display(
    telemetry_features_df[
        ["file_id", "window_start"] +
        TELEMETRY_FEATURE_COLUMNS
    ].head()
)


# ============================================================
# Validation
# ============================================================

EXPECTED_WINDOWS = 13_720

assert len(telemetry_features_df) == EXPECTED_WINDOWS, (
    f"Unexpected telemetry window count: "
    f"{len(telemetry_features_df)} vs {EXPECTED_WINDOWS}"
)

assert telemetry_features_df[
    TELEMETRY_FEATURE_COLUMNS
].isna().sum().sum() == 0

assert telemetry_features_df[
    "TELEM_FLOW_COUNT"
].gt(0).all()

assert telemetry_features_df[
    "TELEM_TOTAL_BYTES"
].ge(0).all()

assert telemetry_features_df[
    "TELEM_TOTAL_PKTS"
].ge(0).all()

assert telemetry_features_df[
    "TELEM_MISSED_BYTES_TOTAL"
].ge(0).all()

assert telemetry_features_df[
    "TELEM_BYTE_ASYMMETRY_MEAN"
].between(0, 1).all()

assert telemetry_features_df[
    "TELEM_PKT_ASYMMETRY_MEAN"
].between(0, 1).all()

print("\n✓ Window count matches temporal audit: 13,720")
print("✓ No missing telemetry features")
print("✓ Flow counts are positive")
print("✓ Byte and packet totals are non-negative")
print("✓ Asymmetry metrics are bounded")
print("✓ No labels or target columns used")
print("✓ Packet / flow telemetry construction complete")

DRISHTI — BOOK 3 | PACKET / FLOW TELEMETRY
✓ Network_dataset_1.csv          → telemetry windows accumulated: 5,677
✓ Network_dataset_10.csv         → telemetry windows accumulated: 5,955
✓ Network_dataset_11.csv         → telemetry windows accumulated: 6,349
✓ Network_dataset_12.csv         → telemetry windows accumulated: 6,994
✓ Network_dataset_13.csv         → telemetry windows accumulated: 7,350
✓ Network_dataset_14.csv         → telemetry windows accumulated: 7,605
✓ Network_dataset_15.csv         → telemetry windows accumulated: 7,867
✓ Network_dataset_16 (1).csv     → telemetry windows accumulated: 8,136
✓ Network_dataset_17.csv         → telemetry windows accumulated: 8,489
✓ Network_dataset_18.csv         → telemetry windows accumulated: 9,241
✓ Network_dataset_20.csv         → telemetry windows accumulated: 9,796
✓ Network_dataset_22.csv         → telemetry windows accumulated: 11,944
✓ Network_dataset_23.csv         → telemetry windows accumulated: 13,720

TELEMETRY FEATURE 

,file_id,window_start,TELEM_FLOW_COUNT,TELEM_DURATION_MEAN,TELEM_DURATION_STD,TELEM_DURATION_MIN,TELEM_DURATION_MAX,TELEM_TOTAL_BYTES,TELEM_SRC_BYTES,TELEM_DST_BYTES,...,TELEM_MISSED_BYTES_MEAN,TELEM_MISSED_BYTES_MAX,TELEM_BYTES_PER_PKT_MEAN,TELEM_BYTES_PER_PKT_STD,TELEM_BYTE_ASYMMETRY_MEAN,TELEM_BYTE_ASYMMETRY_STD,TELEM_PKT_ASYMMETRY_MEAN,TELEM_PKT_ASYMMETRY_STD,TELEM_SRC_BYTE_SHARE_MEAN,TELEM_SRC_PKT_SHARE_MEAN
0,0,2019-04-02 09:45:30,4,20137.382565,34878.969734,0.0,80549.530260,43696067.0,1762852.0,41933215,...,0.0,0,43.317816,75.028658,0.229828,0.398074,0.374995,0.414571,0.010086,0.374997
1,0,2019-04-02 09:46:00,19,0.000055,0.000161,0.0,0.000549,596.0,0.0,596,...,0.0,0,15.684211,45.726939,0.104911,0.305866,0.201754,0.268083,0.000000,0.131579
2,0,2019-04-02 09:46:30,21,0.002335,0.007207,0.0,0.028326,596.0,0.0,596,...,0.0,0,14.190476,43.737985,0.094920,0.292562,0.230159,0.280430,0.000000,0.130952
3,0,2019-04-02 09:47:00,19,0.000079,0.000234,0.0,0.000881,596.0,0.0,596,...,0.0,0,15.684211,45.726939,0.104911,0.305866,0.201754,0.268083,0.000000,0.131579
4,0,2019-04-02 09:47:30,22,13.837961,62.547050,0.0,300.439245,994.0,336.0,658,...,0.0,0,19.340909,47.173130,0.180658,0.383240,0.267424,0.302868,0.045320,0.170455



✓ Window count matches temporal audit: 13,720
✓ No missing telemetry features
✓ Flow counts are positive
✓ Byte and packet totals are non-negative
✓ Asymmetry metrics are bounded
✓ No labels or target columns used
✓ Packet / flow telemetry construction complete


In [16]:
# ============================================================
# CELL 11 — KOOPMAN TEMPORAL DYNAMICS
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("=" * 70)
print("DRISHTI — BOOK 3 | KOOPMAN TEMPORAL DYNAMICS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Combine the four validated feature blocks
# ------------------------------------------------------------

FEATURE_BLOCKS = [
    state_features_df,
    structure_features_df,
    topology_features_df,
    telemetry_features_df,
]

# Verify temporal alignment before merging
for i, block in enumerate(FEATURE_BLOCKS, 1):
    assert len(block) == 13_720, (
        f"Feature block {i} has {len(block)} rows "
        f"instead of 13,720"
    )

    assert block[
        ["file_id", "window_start"]
    ].equals(
        FEATURE_BLOCKS[0][
            ["file_id", "window_start"]
        ]
    ), (
        f"Temporal alignment mismatch in feature block {i}"
    )

# Merge feature blocks
feature_df = FEATURE_BLOCKS[0].copy()

for block in FEATURE_BLOCKS[1:]:
    feature_df = feature_df.merge(
        block,
        on=["file_id", "window_start"],
        how="inner",
        validate="one_to_one",
    )

feature_df = feature_df.sort_values(
    ["file_id", "window_start"]
).reset_index(drop=True)

ALL_FEATURE_COLUMNS = (
    STATE_FEATURE_COLUMNS
    + STRUCTURE_FEATURE_COLUMNS
    + TOPOLOGY_FEATURE_COLUMNS
    + TELEMETRY_FEATURE_COLUMNS
)

print(f"\nTotal feature columns: {len(ALL_FEATURE_COLUMNS)}")
print(f"Feature rows          : {len(feature_df):,}")

assert len(ALL_FEATURE_COLUMNS) == 130
assert len(feature_df) == 13_720

assert feature_df[
    ALL_FEATURE_COLUMNS
].isna().sum().sum() == 0

print("✓ State / Structure / Topology / Telemetry aligned")
print("✓ 130 feature columns confirmed")
print("✓ No missing feature values")


# ------------------------------------------------------------
# 2. Chronological train / validation / test split
# ------------------------------------------------------------

# IMPORTANT:
# Split by time within each file.
# We reserve the latest 20% of each file for evaluation.
#
# The Koopman operator is learned ONLY from the first 80%.

feature_df["time_rank"] = (
    feature_df
    .groupby("file_id")["window_start"]
    .rank(method="first", pct=True)
)

KOOPMAN_TRAIN_MASK = (
    feature_df["time_rank"] <= 0.80
)

KOOPMAN_EVAL_MASK = (
    feature_df["time_rank"] > 0.80
)

print("\nTemporal split:")
print(
    f"Training windows : "
    f"{int(KOOPMAN_TRAIN_MASK.sum()):,}"
)
print(
    f"Evaluation windows: "
    f"{int(KOOPMAN_EVAL_MASK.sum()):,}"
)

assert KOOPMAN_TRAIN_MASK.sum() > 0
assert KOOPMAN_EVAL_MASK.sum() > 0


# ------------------------------------------------------------
# 3. Train-only scaling
# ------------------------------------------------------------

koopman_scaler = StandardScaler()

X_train_raw = feature_df.loc[
    KOOPMAN_TRAIN_MASK,
    ALL_FEATURE_COLUMNS
].astype(float)

X_eval_raw = feature_df.loc[
    KOOPMAN_EVAL_MASK,
    ALL_FEATURE_COLUMNS
].astype(float)

X_train_scaled = koopman_scaler.fit_transform(
    X_train_raw
)

X_eval_scaled = koopman_scaler.transform(
    X_eval_raw
)

print("\n✓ StandardScaler fitted on training data only")
print("✓ Evaluation data transformed using train-only scaler")


# ------------------------------------------------------------
# 4. Compact temporal state representation
# ------------------------------------------------------------

# 130 raw features are too large for a simple Koopman
# operator and can be highly redundant.
#
# PCA provides a compact continuous state representation.
#
# We retain up to 16 components, while requiring 95% variance
# whenever that occurs earlier.

PCA_MAX_COMPONENTS = min(
    16,
    X_train_scaled.shape[1]
)

pca_probe = PCA(
    n_components=PCA_MAX_COMPONENTS,
    random_state=SEED
)

X_train_probe = pca_probe.fit_transform(
    X_train_scaled
)

explained = np.cumsum(
    pca_probe.explained_variance_ratio_
)

components_95 = np.searchsorted(
    explained,
    0.95
) + 1

KOOPMAN_DIM = max(
    1,
    min(
        PCA_MAX_COMPONENTS,
        components_95
    )
)

koopman_pca = PCA(
    n_components=KOOPMAN_DIM,
    random_state=SEED
)

Z_train = koopman_pca.fit_transform(
    X_train_scaled
)

Z_eval = koopman_pca.transform(
    X_eval_scaled
)

print("\nPCA temporal state:")
print(f"Original dimensions : {X_train_scaled.shape[1]}")
print(f"Koopman dimensions  : {KOOPMAN_DIM}")
print(
    f"Variance retained   : "
    f"{koopman_pca.explained_variance_ratio_.sum():.4f}"
)

assert KOOPMAN_DIM <= 16
assert Z_train.shape[1] == KOOPMAN_DIM
assert Z_eval.shape[1] == KOOPMAN_DIM


# ------------------------------------------------------------
# 5. Reconstruct chronological state arrays
# ------------------------------------------------------------

# Store transformed states back against their exact temporal
# identities. This prevents accidental cross-file transitions.

train_state_df = feature_df.loc[
    KOOPMAN_TRAIN_MASK,
    ["file_id", "window_start"]
].copy()

train_state_df["state_index"] = np.arange(
    len(train_state_df)
)

eval_state_df = feature_df.loc[
    KOOPMAN_EVAL_MASK,
    ["file_id", "window_start"]
].copy()

eval_state_df["state_index"] = np.arange(
    len(eval_state_df)
)


# ------------------------------------------------------------
# 6. Build valid consecutive training transitions
# ------------------------------------------------------------

X0_list = []
X1_list = []

transition_count = 0

for file_id in feature_df["file_id"].unique():

    file_mask = (
        train_state_df["file_id"] == file_id
    )

    idx = train_state_df.loc[
        file_mask,
        "state_index"
    ].to_numpy()

    if len(idx) < 2:
        continue

    file_times = train_state_df.loc[
        file_mask,
        "window_start"
    ].to_numpy()

    # Only exact 30-second transitions are allowed.
    deltas = (
        pd.Series(file_times)
        .diff()
        .dt.total_seconds()
        .to_numpy()
    )

    valid = (
        np.isclose(
            deltas[1:],
            30.0
        )
    )

    if not valid.any():
        continue

    for j, is_valid in enumerate(valid, start=1):

        if not is_valid:
            continue

        i0 = idx[j - 1]
        i1 = idx[j]

        X0_list.append(
            Z_train[i0]
        )

        X1_list.append(
            Z_train[i1]
        )

        transition_count += 1


X0 = np.asarray(X0_list)
X1 = np.asarray(X1_list)

print("\nTraining transitions:")
print(f"Valid 30s transitions: {transition_count:,}")

assert len(X0) == len(X1)
assert len(X0) > 0

print("✓ Cross-gap transitions excluded")
print("✓ Cross-file transitions excluded")
print("✓ Only exact 30-second transitions retained")


# ------------------------------------------------------------
# 7. Fit Koopman operator
# ------------------------------------------------------------

# X0 @ K ≈ X1
#
# The least-squares solution is:
#
# K = pinv(X0) @ X1

K = np.linalg.lstsq(
    X0,
    X1,
    rcond=1e-6
)[0]

print("\nKoopman operator:")
print(f"Shape: {K.shape}")

assert K.shape == (
    KOOPMAN_DIM,
    KOOPMAN_DIM
)

assert np.isfinite(K).all()

print("✓ Koopman operator fitted")
print("✓ Operator contains no attack labels")


# ------------------------------------------------------------
# 8. Evaluate one-step temporal reconstruction
# ------------------------------------------------------------

X1_pred = X0 @ K

residuals = X1 - X1_pred

transition_mse = np.mean(
    residuals ** 2
)

transition_mae = np.mean(
    np.abs(residuals)
)

print("\nOne-step Koopman dynamics:")
print(f"MSE: {transition_mse:.6f}")
print(f"MAE: {transition_mae:.6f}")


# ------------------------------------------------------------
# 9. Window-level Koopman residual features
# ------------------------------------------------------------

# For every window that has a valid previous 30-second
# transition, calculate how poorly the learned dynamics
# explain the observed transition.
#
# This residual is a temporal-dynamics signal, NOT an attack
# label.

koopman_records = []

for file_id in feature_df["file_id"].unique():

    file_mask = (
        feature_df["file_id"] == file_id
    )

    file_indices = (
        feature_df.index[file_mask]
        .to_numpy()
    )

    if len(file_indices) < 2:
        continue

    file_times = feature_df.loc[
        file_indices,
        "window_start"
    ].to_numpy()

    # Map global feature index → transformed train/eval state.
    # We use the already-created scaler/PCA transformation
    # directly on the complete feature matrix.
    file_raw = feature_df.loc[
        file_indices,
        ALL_FEATURE_COLUMNS
    ].astype(float)

    file_scaled = koopman_scaler.transform(
        file_raw
    )

    file_z = koopman_pca.transform(
        file_scaled
    )

    for j in range(1, len(file_indices)):

        delta = (
            pd.Timestamp(file_times[j])
            - pd.Timestamp(file_times[j - 1])
        ).total_seconds()

        if not np.isclose(delta, 30.0):
            continue

        current_z = file_z[j]
        previous_z = file_z[j - 1]

        predicted_z = previous_z @ K

        residual = current_z - predicted_z

        koopman_records.append({
            "file_id": file_id,
            "window_start": file_times[j],

            "KOOPMAN_RESIDUAL_MSE": float(
                np.mean(residual ** 2)
            ),

            "KOOPMAN_RESIDUAL_MAE": float(
                np.mean(np.abs(residual))
            ),

            "KOOPMAN_RESIDUAL_NORM": float(
                np.linalg.norm(residual)
            ),

            "KOOPMAN_STATE_NORM": float(
                np.linalg.norm(current_z)
            ),
        })


koopman_features_df = pd.DataFrame(
    koopman_records
)

koopman_features_df = koopman_features_df.sort_values(
    ["file_id", "window_start"]
).reset_index(drop=True)

KOOPMAN_FEATURE_COLUMNS = [
    c for c in koopman_features_df.columns
    if c.startswith("KOOPMAN_")
]

print("\n" + "=" * 70)
print("KOOPMAN FEATURE SUMMARY")
print("=" * 70)

print(
    f"Transition windows : "
    f"{len(koopman_features_df):,}"
)

print(
    f"Koopman features   : "
    f"{len(KOOPMAN_FEATURE_COLUMNS)}"
)

print(
    f"Missing values     : "
    f"{int(koopman_features_df[KOOPMAN_FEATURE_COLUMNS].isna().sum().sum())}"
)

print("\nFeature names:")
for c in KOOPMAN_FEATURE_COLUMNS:
    print("  •", c)

print("\nSample:")

display(
    koopman_features_df[
        ["file_id", "window_start"] +
        KOOPMAN_FEATURE_COLUMNS
    ].head()
)


# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

assert len(koopman_features_df) > 0

assert koopman_features_df[
    KOOPMAN_FEATURE_COLUMNS
].isna().sum().sum() == 0

assert np.isfinite(K).all()

assert np.isfinite(
    koopman_features_df[
        KOOPMAN_FEATURE_COLUMNS
    ].to_numpy()
).all()

assert (
    koopman_features_df[
        "KOOPMAN_RESIDUAL_MSE"
    ] >= 0
).all()

assert (
    koopman_features_df[
        "KOOPMAN_RESIDUAL_MAE"
    ] >= 0
).all()

assert (
    koopman_features_df[
        "KOOPMAN_RESIDUAL_NORM"
    ] >= 0
).all()

print("\n✓ Train-only scaling confirmed")
print("✓ Train-only PCA confirmed")
print("✓ Koopman operator learned from training transitions")
print("✓ Only exact 30-second transitions used")
print("✓ Cross-file / cross-gap transitions excluded")
print("✓ Koopman residuals are finite and non-negative")
print("✓ No labels or target variables used")
print("✓ Koopman temporal dynamics construction complete")

DRISHTI — BOOK 3 | KOOPMAN TEMPORAL DYNAMICS

Total feature columns: 130
Feature rows          : 13,720
✓ State / Structure / Topology / Telemetry aligned
✓ 130 feature columns confirmed
✓ No missing feature values

Temporal split:
Training windows : 10,971
Evaluation windows: 2,749

✓ StandardScaler fitted on training data only
✓ Evaluation data transformed using train-only scaler

PCA temporal state:
Original dimensions : 130
Koopman dimensions  : 16
Variance retained   : 0.8385

Training transitions:
Valid 30s transitions: 10,948
✓ Cross-gap transitions excluded
✓ Cross-file transitions excluded
✓ Only exact 30-second transitions retained

Koopman operator:
Shape: (16, 16)
✓ Koopman operator fitted
✓ Operator contains no attack labels

One-step Koopman dynamics:
MSE: 2.814666
MAE: 0.769126

KOOPMAN FEATURE SUMMARY
Transition windows : 13,686
Koopman features   : 4
Missing values     : 0

Feature names:
  • KOOPMAN_RESIDUAL_MSE
  • KOOPMAN_RESIDUAL_MAE
  • KOOPMAN_RESIDUAL_NORM
  • K

,file_id,window_start,KOOPMAN_RESIDUAL_MSE,KOOPMAN_RESIDUAL_MAE,KOOPMAN_RESIDUAL_NORM,KOOPMAN_STATE_NORM
0,0,2019-04-02 09:46:00,7.098463,2.409665,10.657176,5.422565
1,0,2019-04-02 09:46:30,0.116892,0.257297,1.367576,5.376042
2,0,2019-04-02 09:47:00,0.207370,0.345788,1.821516,5.433039
3,0,2019-04-02 09:47:30,0.118160,0.232038,1.374978,5.578980
4,0,2019-04-02 09:48:00,0.466245,0.526122,2.731284,5.422565



✓ Train-only scaling confirmed
✓ Train-only PCA confirmed
✓ Koopman operator learned from training transitions
✓ Only exact 30-second transitions used
✓ Cross-file / cross-gap transitions excluded
✓ Koopman residuals are finite and non-negative
✓ No labels or target variables used
✓ Koopman temporal dynamics construction complete


In [19]:
# ============================================================
# CELL 12 — UNIFIED TEMPORAL MATRIX + FORECASTING SEQUENCES
# ============================================================

print("=" * 70)
print("DRISHTI — BOOK 3 | TEMPORAL FORECASTING DATASET")
print("=" * 70)


# ============================================================
# 1. FEATURE CONFIGURATION
# ============================================================

EXPECTED_WINDOWS = len(state_features_df)

assert EXPECTED_WINDOWS == 13_720

ALL_FORECAST_FEATURE_COLUMNS = (
    STATE_FEATURE_COLUMNS
    + STRUCTURE_FEATURE_COLUMNS
    + TOPOLOGY_FEATURE_COLUMNS
    + TELEMETRY_FEATURE_COLUMNS
    + KOOPMAN_FEATURE_COLUMNS
)

EXPECTED_FEATURE_COUNT = len(
    ALL_FORECAST_FEATURE_COLUMNS
)

assert EXPECTED_FEATURE_COUNT == 134

print(f"\nExpected feature count: {EXPECTED_FEATURE_COUNT}")
print(f"Expected windows      : {EXPECTED_WINDOWS:,}")


# ============================================================
# 2. START WITH STATE FEATURES
# ============================================================
#
# IMPORTANT:
# State features are already attached to the validated
# temporal index, so we use the complete state dataframe
# as the base instead of creating an index-only dataframe.
# ============================================================

temporal_df = state_features_df.copy()

temporal_df = temporal_df.sort_values(
    ["file_id", "window_start"]
).reset_index(drop=True)

assert len(temporal_df) == EXPECTED_WINDOWS

print(
    f"\nValidated temporal windows: "
    f"{len(temporal_df):,}"
)

print(
    f"State features present: "
    f"{len(STATE_FEATURE_COLUMNS)}"
)


# ============================================================
# 3. RECONSTRUCT ATTACK-WINDOW GROUND TRUTH
# ============================================================
#
# TON-IoT native label:
#   0 = benign
#   1 = attack
#
# Used ONLY for target construction.
# NEVER included in X.
# ============================================================

attack_window_records = []

for file_id, file_path in enumerate(TON_FILES):

    parts = []

    for chunk in pd.read_csv(
        file_path,
        usecols=["ts", "label"],
        chunksize=250_000,
        low_memory=False
    ):
        parts.append(chunk)

    df = pd.concat(
        parts,
        ignore_index=True
    )

    df["timestamp"] = pd.to_datetime(
        df["ts"],
        unit="s",
        errors="coerce"
    )

    assert df["timestamp"].notna().all()

    df["window_start"] = (
        df["timestamp"].dt.floor("30s")
    )

    df["label"] = pd.to_numeric(
        df["label"],
        errors="coerce"
    )

    assert df["label"].isin([0, 1]).all()

    window_labels = (
        df.groupby("window_start")["label"]
        .max()
        .reset_index()
    )

    window_labels["file_id"] = file_id

    window_labels = window_labels.rename(
        columns={
            "label": "attack_window"
        }
    )

    attack_window_records.append(
        window_labels[
            [
                "file_id",
                "window_start",
                "attack_window",
            ]
        ]
    )

    del df, parts, window_labels
    gc.collect()


attack_windows_df = pd.concat(
    attack_window_records,
    ignore_index=True
)

attack_windows_df["attack_window"] = (
    attack_windows_df["attack_window"]
    .astype(int)
)

temporal_df = temporal_df.merge(
    attack_windows_df,
    on=["file_id", "window_start"],
    how="left",
    validate="one_to_one",
)

assert len(temporal_df) == EXPECTED_WINDOWS

assert temporal_df[
    "attack_window"
].notna().all()

temporal_df["attack_window"] = (
    temporal_df["attack_window"]
    .astype(int)
)

assert temporal_df[
    "attack_window"
].isin([0, 1]).all()

print(
    f"Attack windows: "
    f"{temporal_df['attack_window'].sum():,}"
)

print(
    f"Benign windows: "
    f"{(temporal_df['attack_window'] == 0).sum():,}"
)

print(
    "✓ Native TON-IoT ground truth reconstructed"
)

print(
    "✓ Ground truth kept separate from predictors"
)


# ============================================================
# 4. MERGE REMAINING FEATURE BLOCKS
# ============================================================

FEATURE_BLOCKS = [
    structure_features_df,
    topology_features_df,
    telemetry_features_df,
    koopman_features_df,
]

for block in FEATURE_BLOCKS:

    temporal_df = temporal_df.merge(
        block,
        on=["file_id", "window_start"],
        how="left",
        validate="one_to_one",
    )

temporal_df = temporal_df.sort_values(
    ["file_id", "window_start"]
).reset_index(drop=True)

print(
    f"\nUnified temporal rows: "
    f"{len(temporal_df):,}"
)

assert len(temporal_df) == EXPECTED_WINDOWS


# ============================================================
# 5. VERIFY ALL 134 FEATURES ARE PRESENT
# ============================================================

missing_columns = [
    c for c in ALL_FORECAST_FEATURE_COLUMNS
    if c not in temporal_df.columns
]

assert not missing_columns, (
    f"Missing feature columns: {missing_columns}"
)

print(
    f"All forecast feature columns present: "
    f"{len(ALL_FORECAST_FEATURE_COLUMNS)}"
)

missing_before_filter = int(
    temporal_df[
        ALL_FORECAST_FEATURE_COLUMNS
    ].isna().sum().sum()
)

print(
    f"Missing feature values before "
    f"Koopman filter: {missing_before_filter:,}"
)


# ============================================================
# 6. REMOVE WINDOWS WITHOUT KOOPMAN TRANSITION
# ============================================================
#
# The first window of each continuous run has no previous
# transition and therefore no Koopman residual.
# ============================================================

valid_koopman_mask = (
    temporal_df[
        KOOPMAN_FEATURE_COLUMNS
    ].notna().all(axis=1)
)

temporal_df = temporal_df[
    valid_koopman_mask
].copy()

temporal_df = temporal_df.sort_values(
    ["file_id", "window_start"]
).reset_index(drop=True)

assert temporal_df[
    ALL_FORECAST_FEATURE_COLUMNS
].isna().sum().sum() == 0

print(
    f"Rows after Koopman availability filter: "
    f"{len(temporal_df):,}"
)

print(
    "✓ All retained rows have complete features"
)


# ============================================================
# 7. FORECASTING CONFIGURATION
# ============================================================

HISTORY_WINDOWS = 10
WINDOW_SECONDS = 30

FORECAST_HORIZONS = {
    "30s": 1,
    "60s": 2,
    "150s": 5,
    "300s": 10,
    "600s": 20,
}

print("\nForecast configuration:")
print(
    f"History: {HISTORY_WINDOWS} × "
    f"{WINDOW_SECONDS}s = 5 minutes"
)

print(
    "Horizons:",
    ", ".join(FORECAST_HORIZONS.keys())
)


# ============================================================
# 8. BUILD EXACT CONTIGUOUS 5-MINUTE HISTORIES
# ============================================================

sequence_records = []

for file_id, g in temporal_df.groupby(
    "file_id",
    sort=True
):

    g = g.sort_values(
        "window_start"
    ).reset_index(drop=True)

    time_values = pd.to_datetime(
        g["window_start"]
    )

    delta_seconds = (
        time_values.diff()
        .dt.total_seconds()
    )

    # New run whenever consecutive windows are not
    # exactly 30 seconds apart.
    run_id = (
        delta_seconds
        .ne(WINDOW_SECONDS)
        .cumsum()
    )

    g["_run_id"] = run_id

    for _, run in g.groupby(
        "_run_id",
        sort=True
    ):

        run = run.reset_index(drop=True)

        if len(run) < HISTORY_WINDOWS:
            continue

        for source_idx in range(
            HISTORY_WINDOWS - 1,
            len(run)
        ):

            source_row = run.iloc[source_idx]

            # ------------------------------------------------
            # SOURCE MUST BE BENIGN
            # ------------------------------------------------

            if int(
                source_row["attack_window"]
            ) != 0:
                continue

            # ------------------------------------------------
            # 10-WINDOW HISTORY
            # ------------------------------------------------

            history = run.iloc[
                source_idx - HISTORY_WINDOWS + 1:
                source_idx + 1
            ]

            assert len(history) == HISTORY_WINDOWS

            history_times = pd.to_datetime(
                history["window_start"]
            )

            history_deltas = (
                history_times.diff()
                .dt.total_seconds()
                .dropna()
            )

            assert (
                history_deltas == WINDOW_SECONDS
            ).all()

            # ------------------------------------------------
            # FUTURE TARGETS
            # ------------------------------------------------

            target_values = {}
            tte_values = {}
            onset_times = {}

            for horizon_name, horizon_windows in (
                FORECAST_HORIZONS.items()
            ):

                future_start = source_idx + 1

                future_end = (
                    source_idx
                    + horizon_windows
                    + 1
                )

                # Complete horizon required.
                if future_end > len(run):

                    target_values[
                        horizon_name
                    ] = None

                    tte_values[
                        horizon_name
                    ] = None

                    onset_times[
                        horizon_name
                    ] = None

                    continue

                future = run.iloc[
                    future_start:future_end
                ]

                future_attack = (
                    future[
                        "attack_window"
                    ]
                    .astype(int)
                    .to_numpy()
                )

                onset_positions = np.where(
                    future_attack == 1
                )[0]

                # No attack in horizon
                if len(onset_positions) == 0:

                    target_values[
                        horizon_name
                    ] = 0

                    tte_values[
                        horizon_name
                    ] = None

                    onset_times[
                        horizon_name
                    ] = None

                    continue

                # First attack-positive future window
                onset_pos = int(
                    onset_positions[0]
                )

                if onset_pos == 0:

                    previous_attack = int(
                        source_row[
                            "attack_window"
                        ]
                    )

                else:

                    previous_attack = int(
                        future[
                            "attack_window"
                        ].iloc[
                            onset_pos - 1
                        ]
                    )

                if previous_attack == 0:

                    target_values[
                        horizon_name
                    ] = 1

                    tte_values[
                        horizon_name
                    ] = onset_pos + 1

                    onset_times[
                        horizon_name
                    ] = future[
                        "window_start"
                    ].iloc[
                        onset_pos
                    ]

                else:

                    target_values[
                        horizon_name
                    ] = 0

                    tte_values[
                        horizon_name
                    ] = None

                    onset_times[
                        horizon_name
                    ] = None

            # ------------------------------------------------
            # STORE SEQUENCE
            # ------------------------------------------------

            record = {
                "file_id": file_id,

                "source_time": source_row[
                    "window_start"
                ],

                "history_start": history[
                    "window_start"
                ].iloc[0],

                "history_end": history[
                    "window_start"
                ].iloc[-1],

                "X": history[
                    ALL_FORECAST_FEATURE_COLUMNS
                ].to_numpy(
                    dtype=np.float32
                ),
            }

            for horizon_name in FORECAST_HORIZONS:

                record[
                    f"Y_{horizon_name}_target"
                ] = target_values[
                    horizon_name
                ]

                record[
                    f"Y_{horizon_name}_tte_windows"
                ] = tte_values[
                    horizon_name
                ]

                record[
                    f"Y_{horizon_name}_onset_time"
                ] = onset_times[
                    horizon_name
                ]

            sequence_records.append(record)


# ============================================================
# 9. CREATE FORECASTING DATAFRAME
# ============================================================

forecast_sequences_df = pd.DataFrame(
    sequence_records
)

print("\n" + "=" * 70)
print("SEQUENCE CONSTRUCTION SUMMARY")
print("=" * 70)

print(
    f"Forecasting sequences: "
    f"{len(forecast_sequences_df):,}"
)

assert len(forecast_sequences_df) > 0


# ============================================================
# 10. SEQUENCE SHAPE VALIDATION
# ============================================================

sequence_shapes = {
    x.shape
    for x in forecast_sequences_df["X"]
}

print(
    f"Unique sequence shapes: "
    f"{sequence_shapes}"
)

assert sequence_shapes == {
    (
        HISTORY_WINDOWS,
        EXPECTED_FEATURE_COUNT
    )
}

print(
    f"✓ Every sequence = "
    f"{HISTORY_WINDOWS} windows × "
    f"{EXPECTED_FEATURE_COUNT} features"
)


# ============================================================
# 11. FUTURE ONSET TARGET SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FUTURE ONSET TARGET SUMMARY")
print("=" * 70)

for horizon_name in FORECAST_HORIZONS:

    target_col = (
        f"Y_{horizon_name}_target"
    )

    valid = (
        forecast_sequences_df[
            target_col
        ].notna()
    )

    positives = int(
        forecast_sequences_df.loc[
            valid,
            target_col
        ]
        .astype(int)
        .sum()
    )

    total_valid = int(
        valid.sum()
    )

    positive_rate = (
        positives / total_valid
        if total_valid > 0
        else np.nan
    )

    print(
        f"{horizon_name:>5} | "
        f"valid={total_valid:5d} | "
        f"onsets={positives:4d} | "
        f"rate={positive_rate:.4f}"
    )


# ============================================================
# 12. TTE SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("TTE DISTRIBUTION — FORECAST POSITIVES")
print("=" * 70)

for horizon_name in FORECAST_HORIZONS:

    tte_col = (
        f"Y_{horizon_name}_tte_windows"
    )

    values = pd.to_numeric(
        forecast_sequences_df[
            tte_col
        ],
        errors="coerce"
    ).dropna()

    if len(values) == 0:

        print(
            f"{horizon_name:>5} | "
            f"No positive forecast cases"
        )

        continue

    print(
        f"{horizon_name:>5} | "
        f"n={len(values):4d} | "
        f"min={values.min():.0f} | "
        f"median={values.median():.0f} | "
        f"max={values.max():.0f}"
    )


# ============================================================
# 13. LEAKAGE / TEMPORAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("LEAKAGE / TEMPORAL VALIDATION")
print("=" * 70)

source_lookup = (
    temporal_df[
        [
            "file_id",
            "window_start",
            "attack_window",
        ]
    ]
    .drop_duplicates(
        ["file_id", "window_start"]
    )
)

source_check = forecast_sequences_df.merge(
    source_lookup,
    left_on=[
        "file_id",
        "source_time",
    ],
    right_on=[
        "file_id",
        "window_start",
    ],
    how="left",
    validate="one_to_one",
)

assert source_check[
    "attack_window"
].notna().all()

assert (
    source_check[
        "attack_window"
    ].astype(int) == 0
).all()

print(
    "✓ All forecasting source windows are benign"
)


# ------------------------------------------------------------
# Verify every history is exactly 10 consecutive windows
# ------------------------------------------------------------

history_duration = (
    pd.to_datetime(
        forecast_sequences_df[
            "history_end"
        ]
    )
    -
    pd.to_datetime(
        forecast_sequences_df[
            "history_start"
        ]
    )
).dt.total_seconds()

assert (
    history_duration == 270
).all()

print(
    "✓ Every history contains 10 exact "
    "30-second windows"
)

print(
    "✓ History span = 270 seconds "
    "(10 observed windows)"
)

print(
    "✓ No cross-file history constructed"
)

print(
    "✓ No cross-gap history constructed"
)

print(
    "✓ Future labels used only for target construction"
)

print(
    "✓ Target information is not included in X"
)

print(
    "\n✓ TEMPORAL FORECASTING DATASET CONSTRUCTION COMPLETE"
)

DRISHTI — BOOK 3 | TEMPORAL FORECASTING DATASET

Expected feature count: 134
Expected windows      : 13,720

Validated temporal windows: 13,720
State features present: 46
Attack windows: 7,846
Benign windows: 5,874
✓ Native TON-IoT ground truth reconstructed
✓ Ground truth kept separate from predictors

Unified temporal rows: 13,720
All forecast feature columns present: 134
Missing feature values before Koopman filter: 136
Rows after Koopman availability filter: 13,686
✓ All retained rows have complete features

Forecast configuration:
History: 10 × 30s = 5 minutes
Horizons: 30s, 60s, 150s, 300s, 600s

SEQUENCE CONSTRUCTION SUMMARY
Forecasting sequences: 5,823
Unique sequence shapes: {(10, 134)}
✓ Every sequence = 10 windows × 134 features

FUTURE ONSET TARGET SUMMARY
  30s | valid= 5815 | onsets=  71 | rate=0.0122
  60s | valid= 5808 | onsets=  99 | rate=0.0170
 150s | valid= 5787 | onsets= 137 | rate=0.0237
 300s | valid= 5771 | onsets= 155 | rate=0.0269
 600s | valid= 5730 | onsets=

In [20]:
# ============================================================
# CELL 13 — CHRONOLOGICAL SPLIT + TRAIN-ONLY SCALING
# ============================================================

from sklearn.preprocessing import StandardScaler

print("=" * 70)
print("DRISHTI — BOOK 3 | CHRONOLOGICAL SPLIT + SCALING")
print("=" * 70)


# ============================================================
# 1. CONFIGURATION
# ============================================================

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

assert abs(
    TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0
) < 1e-9

print("\nSplit configuration:")
print(f"Train      : {TRAIN_RATIO:.0%}")
print(f"Validation : {VAL_RATIO:.0%}")
print(f"Test       : {TEST_RATIO:.0%}")


# ============================================================
# 2. SORT SEQUENCES CHRONOLOGICALLY
# ============================================================

forecast_sequences_df = (
    forecast_sequences_df
    .sort_values(
        ["file_id", "source_time"]
    )
    .reset_index(drop=True)
)

assert forecast_sequences_df[
    ["file_id", "source_time"]
].notna().all().all()

print(
    f"\nTotal forecasting sequences: "
    f"{len(forecast_sequences_df):,}"
)


# ============================================================
# 3. SPLIT WITHIN EACH FILE
# ============================================================
#
# Each TON-IoT CSV represents a separate temporal stream.
# We therefore split chronologically inside each file rather
# than randomly mixing sequences from the same temporal stream.
#
# This prevents future observations from appearing in training
# for a model evaluated on earlier/later temporal contexts.
# ============================================================

split_parts = []

for file_id, g in forecast_sequences_df.groupby(
    "file_id",
    sort=True
):

    g = g.sort_values(
        "source_time"
    ).copy()

    n = len(g)

    if n < 3:
        continue

    train_end = int(
        np.floor(
            n * TRAIN_RATIO
        )
    )

    val_end = int(
        np.floor(
            n * (TRAIN_RATIO + VAL_RATIO)
        )
    )

    # Make sure every partition gets at least one sample.
    train_end = max(1, train_end)
    val_end = max(
        train_end + 1,
        val_end
    )

    if val_end >= n:
        val_end = n - 1

    g["split"] = "test"

    g.iloc[
        :train_end,
        g.columns.get_loc("split")
    ] = "train"

    g.iloc[
        train_end:val_end,
        g.columns.get_loc("split")
    ] = "validation"

    split_parts.append(g)


forecast_sequences_split_df = pd.concat(
    split_parts,
    ignore_index=True
)

forecast_sequences_split_df = (
    forecast_sequences_split_df
    .sort_values(
        ["file_id", "source_time"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 4. SPLIT SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SEQUENCE SPLIT SUMMARY")
print("=" * 70)

split_counts = (
    forecast_sequences_split_df[
        "split"
    ]
    .value_counts()
    .reindex(
        ["train", "validation", "test"],
        fill_value=0
    )
)

for split_name, count in split_counts.items():

    print(
        f"{split_name:>10}: "
        f"{int(count):,}"
    )

assert split_counts["train"] > 0
assert split_counts["validation"] > 0
assert split_counts["test"] > 0

assert (
    split_counts.sum()
    == len(forecast_sequences_split_df)
)


# ============================================================
# 5. VERIFY CHRONOLOGICAL ORDER WITHIN EACH FILE
# ============================================================

print("\nChronological boundary checks:")

for file_id, g in forecast_sequences_split_df.groupby(
    "file_id",
    sort=True
):

    g = g.sort_values(
        "source_time"
    )

    train_times = g.loc[
        g["split"] == "train",
        "source_time"
    ]

    val_times = g.loc[
        g["split"] == "validation",
        "source_time"
    ]

    test_times = g.loc[
        g["split"] == "test",
        "source_time"
    ]

    if len(train_times) > 0 and len(val_times) > 0:

        assert (
            train_times.max()
            < val_times.min()
        )

    if len(val_times) > 0 and len(test_times) > 0:

        assert (
            val_times.max()
            < test_times.min()
        )

print(
    "✓ Train occurs before validation"
)

print(
    "✓ Validation occurs before test"
)

print(
    "✓ No random temporal mixing"
)


# ============================================================
# 6. EXTRACT X ARRAYS
# ============================================================

X_all = np.stack(
    forecast_sequences_split_df["X"].to_numpy()
).astype(
    np.float32
)

print(
    f"\nRaw sequence tensor: "
    f"{X_all.shape}"
)

assert X_all.shape[1:] == (
    HISTORY_WINDOWS,
    EXPECTED_FEATURE_COUNT
)


# ============================================================
# 7. SPLIT MASKS
# ============================================================

train_mask = (
    forecast_sequences_split_df[
        "split"
    ].eq("train")
    .to_numpy()
)

val_mask = (
    forecast_sequences_split_df[
        "split"
    ].eq("validation")
    .to_numpy()
)

test_mask = (
    forecast_sequences_split_df[
        "split"
    ].eq("test")
    .to_numpy()
)

X_train_raw = X_all[
    train_mask
]

X_val_raw = X_all[
    val_mask
]

X_test_raw = X_all[
    test_mask
]

print("\nRaw tensor splits:")
print(
    f"Train      : {X_train_raw.shape}"
)
print(
    f"Validation : {X_val_raw.shape}"
)
print(
    f"Test       : {X_test_raw.shape}"
)


# ============================================================
# 8. TRAIN-ONLY FEATURE SCALING
# ============================================================
#
# The scaler operates feature-wise.
#
# All 10 historical timesteps belonging to the training set
# are used to estimate scaling parameters.
#
# Validation and test NEVER influence the scaler.
# ============================================================

forecast_scaler = StandardScaler()

X_train_2d = X_train_raw.reshape(
    -1,
    EXPECTED_FEATURE_COUNT
)

X_val_2d = X_val_raw.reshape(
    -1,
    EXPECTED_FEATURE_COUNT
)

X_test_2d = X_test_raw.reshape(
    -1,
    EXPECTED_FEATURE_COUNT
)

forecast_scaler.fit(
    X_train_2d
)

X_train_scaled = forecast_scaler.transform(
    X_train_2d
).reshape(
    X_train_raw.shape
).astype(
    np.float32
)

X_val_scaled = forecast_scaler.transform(
    X_val_2d
).reshape(
    X_val_raw.shape
).astype(
    np.float32
)

X_test_scaled = forecast_scaler.transform(
    X_test_2d
).reshape(
    X_test_raw.shape
).astype(
    np.float32
)


# ============================================================
# 9. SCALING VALIDATION
# ============================================================

assert np.isfinite(
    X_train_scaled
).all()

assert np.isfinite(
    X_val_scaled
).all()

assert np.isfinite(
    X_test_scaled
).all()

print("\n✓ Scaler fitted on training data only")
print("✓ Validation transformed with frozen train scaler")
print("✓ Test transformed with frozen train scaler")
print("✓ All scaled values are finite")


# ============================================================
# 10. TRAIN-SCALER STATISTICS
# ============================================================

train_scaled_mean = float(
    X_train_scaled.mean()
)

train_scaled_std = float(
    X_train_scaled.std()
)

print("\nTraining scaled distribution:")
print(
    f"Mean: {train_scaled_mean:.6f}"
)

print(
    f"Std : {train_scaled_std:.6f}"
)


# ============================================================
# 11. HORIZON TARGET MATRICES
# ============================================================

Y = {}

for horizon_name in FORECAST_HORIZONS:

    target_col = (
        f"Y_{horizon_name}_target"
    )

    values = (
        forecast_sequences_split_df[
            target_col
        ]
    )

    Y[horizon_name] = {
        "train": values[
            train_mask
        ].to_numpy(),

        "validation": values[
            val_mask
        ].to_numpy(),

        "test": values[
            test_mask
        ].to_numpy(),
    }


# ============================================================
# 12. TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION BY SPLIT")
print("=" * 70)

for horizon_name in FORECAST_HORIZONS:

    print(
        f"\n--- {horizon_name} ---"
    )

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        values = Y[
            horizon_name
        ][split_name]

        valid = pd.notna(values)

        n_valid = int(
            valid.sum()
        )

        n_positive = int(
            np.nansum(
                values.astype(
                    float
                )
            )
        )

        rate = (
            n_positive / n_valid
            if n_valid > 0
            else np.nan
        )

        print(
            f"{split_name:>10}: "
            f"valid={n_valid:5d} | "
            f"positive={n_positive:4d} | "
            f"rate={rate:.4f}"
        )


# ============================================================
# 13. SAVE SPLIT INDICES FOR REPRODUCIBILITY
# ============================================================

forecast_split_metadata = {
    "train_ratio": TRAIN_RATIO,
    "validation_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "history_windows": HISTORY_WINDOWS,
    "window_seconds": WINDOW_SECONDS,
    "feature_count": EXPECTED_FEATURE_COUNT,
    "seed": SEED,
}

print("\n" + "=" * 70)
print("SPLIT / SCALING METADATA")
print("=" * 70)

print(
    json.dumps(
        forecast_split_metadata,
        indent=2
    )
)


# ============================================================
# FINAL VALIDATION
# ============================================================

assert (
    len(X_train_scaled)
    + len(X_val_scaled)
    + len(X_test_scaled)
    == len(forecast_sequences_split_df)
)

assert (
    X_train_scaled.shape[1:]
    == (HISTORY_WINDOWS, EXPECTED_FEATURE_COUNT)
)

assert (
    X_val_scaled.shape[1:]
    == (HISTORY_WINDOWS, EXPECTED_FEATURE_COUNT)
)

assert (
    X_test_scaled.shape[1:]
    == (HISTORY_WINDOWS, EXPECTED_FEATURE_COUNT)
)

print("\n✓ Chronological split complete")
print("✓ Train-only scaling complete")
print("✓ Sequence dimensions preserved")
print("✓ Target matrices prepared")
print("✓ CELL 13 COMPLETE")

DRISHTI — BOOK 3 | CHRONOLOGICAL SPLIT + SCALING

Split configuration:
Train      : 70%
Validation : 15%
Test       : 15%

Total forecasting sequences: 5,823

SEQUENCE SPLIT SUMMARY
     train: 4,073
validation: 875
      test: 875

Chronological boundary checks:
✓ Train occurs before validation
✓ Validation occurs before test
✓ No random temporal mixing

Raw sequence tensor: (5823, 10, 134)

Raw tensor splits:
Train      : (4073, 10, 134)
Validation : (875, 10, 134)
Test       : (875, 10, 134)

✓ Scaler fitted on training data only
✓ Validation transformed with frozen train scaler
✓ Test transformed with frozen train scaler
✓ All scaled values are finite

Training scaled distribution:
Mean: 0.000000
Std : 0.996262

TARGET DISTRIBUTION BY SPLIT

--- 30s ---
     train: valid= 4068 | positive=  48 | rate=0.0118
validation: valid=  874 | positive=   6 | rate=0.0069
      test: valid=  873 | positive=  17 | rate=0.0195

--- 60s ---
     train: valid= 4064 | positive=  64 | rate=0.0157
val

In [21]:
# ============================================================
# CELL 14 — STRICT TRAIN-ONLY PCA + KOOPMAN RE-FIT
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("=" * 70)
print("DRISHTI — BOOK 3 | STRICT TRAIN-ONLY PCA + KOOPMAN")
print("=" * 70)


# ============================================================
# 1. CONFIGURATION
# ============================================================

KOOPMAN_MAX_COMPONENTS = 16

print("\nConfiguration:")
print(
    f"Maximum PCA components : "
    f"{KOOPMAN_MAX_COMPONENTS}"
)

print(
    "Training partition only will be used to fit:"
)

print("  ✓ StandardScaler")
print("  ✓ PCA")
print("  ✓ Koopman operator")


# ============================================================
# 2. BUILD BASE FEATURE MATRIX
# ============================================================
#
# These are the original 134 non-label features used by
# the forecasting experiment.
#
# The feature ordering MUST remain identical across all
# train / validation / test transformations.
# ============================================================

BASE_FEATURE_COLUMNS = (
    STATE_FEATURE_COLUMNS
    + STRUCTURE_FEATURE_COLUMNS
    + TOPOLOGY_FEATURE_COLUMNS
    + TELEMETRY_FEATURE_COLUMNS
    + KOOPMAN_FEATURE_COLUMNS
)

# The old Koopman columns are intentionally excluded here.
#
# We are rebuilding Koopman from the underlying
# State + Structure + Topology + Telemetry representation.

BASE_FEATURE_COLUMNS = (
    STATE_FEATURE_COLUMNS
    + STRUCTURE_FEATURE_COLUMNS
    + TOPOLOGY_FEATURE_COLUMNS
    + TELEMETRY_FEATURE_COLUMNS
)

EXPECTED_BASE_FEATURE_COUNT = len(
    BASE_FEATURE_COLUMNS
)

print(
    f"\nBase feature count: "
    f"{EXPECTED_BASE_FEATURE_COUNT}"
)

assert EXPECTED_BASE_FEATURE_COUNT == 130


# ============================================================
# 3. BUILD WINDOW-LEVEL BASE MATRIX
# ============================================================

base_feature_df = (
    state_features_df[
        [
            "file_id",
            "window_start"
        ]
        + STATE_FEATURE_COLUMNS
    ]
    .merge(
        structure_features_df[
            [
                "file_id",
                "window_start"
            ]
            + STRUCTURE_FEATURE_COLUMNS
        ],
        on=[
            "file_id",
            "window_start"
        ],
        how="inner"
    )
    .merge(
        topology_features_df[
            [
                "file_id",
                "window_start"
            ]
            + TOPOLOGY_FEATURE_COLUMNS
        ],
        on=[
            "file_id",
            "window_start"
        ],
        how="inner"
    )
    .merge(
        telemetry_features_df[
            [
                "file_id",
                "window_start"
            ]
            + TELEMETRY_FEATURE_COLUMNS
        ],
        on=[
            "file_id",
            "window_start"
        ],
        how="inner"
    )
)

base_feature_df = (
    base_feature_df
    .sort_values(
        [
            "file_id",
            "window_start"
        ]
    )
    .reset_index(drop=True)
)

assert len(base_feature_df) == 13720

assert (
    base_feature_df[
        BASE_FEATURE_COLUMNS
    ].isna().sum().sum()
    == 0
)

print(
    f"\nBase feature matrix: "
    f"{base_feature_df.shape}"
)

print(
    "✓ State features present"
)

print(
    "✓ Structure features present"
)

print(
    "✓ Topology features present"
)

print(
    "✓ Telemetry features present"
)


# ============================================================
# 4. ALIGN BASE FEATURES TO FORECASTING WINDOWS
# ============================================================
#
# Only windows that have complete 10-step forecasting
# histories are relevant.
#
# We use the existing forecast sequence metadata to obtain
# the exact source windows used by the forecasting experiment.
# ============================================================

forecast_source_keys = (
    forecast_sequences_split_df[
        [
            "file_id",
            "source_time"
        ]
    ]
    .drop_duplicates()
    .copy()
)

forecast_source_keys = forecast_source_keys.rename(
    columns={
        "source_time": "window_start"
    }
)

forecast_source_keys["window_start"] = pd.to_datetime(
    forecast_source_keys["window_start"]
)

base_feature_df["window_start"] = pd.to_datetime(
    base_feature_df["window_start"]
)

forecast_base_df = (
    forecast_source_keys
    .merge(
        base_feature_df,
        on=[
            "file_id",
            "window_start"
        ],
        how="left"
    )
    .sort_values(
        [
            "file_id",
            "window_start"
        ]
    )
    .reset_index(drop=True)
)

assert len(forecast_base_df) == (
    len(forecast_source_keys)
)

assert (
    forecast_base_df[
        BASE_FEATURE_COLUMNS
    ].isna().sum().sum()
    == 0
)

print(
    f"\nForecast source windows: "
    f"{len(forecast_base_df):,}"
)

print(
    "✓ Every forecasting source window "
    "has complete base features"
)


# ============================================================
# 5. IDENTIFY TRAIN / VALIDATION / TEST WINDOWS
# ============================================================

split_lookup = (
    forecast_sequences_split_df[
        [
            "file_id",
            "source_time",
            "split"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "source_time": "window_start"
        }
    )
)

split_lookup["window_start"] = pd.to_datetime(
    split_lookup["window_start"]
)

forecast_base_df = (
    forecast_base_df
    .merge(
        split_lookup,
        on=[
            "file_id",
            "window_start"
        ],
        how="left",
        validate="one_to_one"
    )
)

assert (
    forecast_base_df["split"]
    .notna()
    .all()
)

print("\nWindow split counts:")

print(
    forecast_base_df[
        "split"
    ].value_counts()
    .reindex(
        [
            "train",
            "validation",
            "test"
        ],
        fill_value=0
    )
)


# ============================================================
# 6. PREPARE RAW BASE ARRAYS
# ============================================================

X_base_all = (
    forecast_base_df[
        BASE_FEATURE_COLUMNS
    ]
    .to_numpy(
        dtype=np.float32
    )
)

train_window_mask = (
    forecast_base_df[
        "split"
    ].eq("train")
    .to_numpy()
)

val_window_mask = (
    forecast_base_df[
        "split"
    ].eq("validation")
    .to_numpy()
)

test_window_mask = (
    forecast_base_df[
        "split"
    ].eq("test")
    .to_numpy()
)

X_base_train = X_base_all[
    train_window_mask
]

X_base_val = X_base_all[
    val_window_mask
]

X_base_test = X_base_all[
    test_window_mask
]

print("\nBase matrices:")
print(
    f"Train      : {X_base_train.shape}"
)

print(
    f"Validation : {X_base_val.shape}"
)

print(
    f"Test       : {X_base_test.shape}"
)


# ============================================================
# 7. TRAIN-ONLY STANDARDIZATION
# ============================================================

strict_koopman_scaler = StandardScaler()

strict_koopman_scaler.fit(
    X_base_train
)

X_scaled_all = (
    strict_koopman_scaler
    .transform(
        X_base_all
    )
    .astype(
        np.float32
    )
)

print(
    "\n✓ StandardScaler fitted using "
    "training windows only"
)


# ============================================================
# 8. TRAIN-ONLY PCA
# ============================================================

n_components = min(
    KOOPMAN_MAX_COMPONENTS,
    X_base_train.shape[0],
    X_base_train.shape[1]
)

strict_koopman_pca = PCA(
    n_components=n_components,
    random_state=SEED
)

strict_koopman_pca.fit(
    X_scaled_all[
        train_window_mask
    ]
)

Z_all = (
    strict_koopman_pca
    .transform(
        X_scaled_all
    )
    .astype(
        np.float32
    )
)

explained_variance = float(
    strict_koopman_pca
    .explained_variance_ratio_
    .sum()
)

print(
    f"\n✓ PCA fitted using training windows only"
)

print(
    f"Selected components : "
    f"{n_components}"
)

print(
    f"Variance retained   : "
    f"{explained_variance:.4%}"
)


# ============================================================
# 9. CREATE EXACT 30-SECOND TRAINING TRANSITIONS
# ============================================================
#
# Koopman is learned from transitions:
#
#       z(t)  ->  z(t+30s)
#
# Only transitions where BOTH windows belong to the
# forecasting training partition are used.
#
# This is stricter than simply fitting on all training
# windows because the transition operator itself cannot
# observe validation/test states.
# ============================================================

train_transition_rows = []

for file_id, g in forecast_base_df[
    forecast_base_df["split"] == "train"
].groupby(
    "file_id",
    sort=True
):

    g = (
        g.sort_values(
            "window_start"
        )
        .reset_index()
    )

    row_positions = g["index"].to_numpy()

    times = pd.to_datetime(
        g["window_start"]
    ).to_numpy()

    for i in range(
        len(g) - 1
    ):

        delta = (
            times[i + 1]
            - times[i]
        ) / np.timedelta64(
            1,
            "s"
        )

        if delta == 30:

            train_transition_rows.append(
                (
                    row_positions[i],
                    row_positions[i + 1]
                )
            )


assert len(
    train_transition_rows
) > 0

train_transition_rows = np.asarray(
    train_transition_rows,
    dtype=np.int64
)

Z_train_prev = Z_all[
    train_transition_rows[:, 0]
]

Z_train_next = Z_all[
    train_transition_rows[:, 1]
]

print(
    f"\nValid training transitions: "
    f"{len(train_transition_rows):,}"
)

print(
    "✓ All transitions are exactly 30 seconds"
)


# ============================================================
# 10. FIT KOOPMAN OPERATOR
# ============================================================
#
# Z_next ≈ Z_prev @ K
#
# Least-squares solution:
#
# K = pinv(Z_prev) @ Z_next
# ============================================================

K_strict = (
    np.linalg.pinv(
        Z_train_prev
    )
    @ Z_train_next
)

K_strict = K_strict.astype(
    np.float32
)

assert K_strict.shape == (
    n_components,
    n_components
)

print(
    f"\n✓ Koopman operator fitted "
    f"using training transitions only"
)

print(
    f"Operator shape: "
    f"{K_strict.shape}"
)


# ============================================================
# 11. COMPUTE ONE-STEP KOOPMAN RESIDUALS
# ============================================================

Z_pred_next = (
    Z_all
    @ K_strict
)

residual = (
    Z_all
    - Z_pred_next
)

residual_mse = np.mean(
    residual ** 2,
    axis=1
)

residual_mae = np.mean(
    np.abs(
        residual
    ),
    axis=1
)

residual_norm = np.linalg.norm(
    residual,
    axis=1
)

state_norm = np.linalg.norm(
    Z_all,
    axis=1
)


# ============================================================
# 12. CREATE STRICT KOOPMAN FEATURE TABLE
# ============================================================

strict_koopman_features_df = (
    forecast_base_df[
        [
            "file_id",
            "window_start"
        ]
    ]
    .copy()
)

strict_koopman_features_df[
    "KOOPMAN_RESIDUAL_MSE"
] = residual_mse

strict_koopman_features_df[
    "KOOPMAN_RESIDUAL_MAE"
] = residual_mae

strict_koopman_features_df[
    "KOOPMAN_RESIDUAL_NORM"
] = residual_norm

strict_koopman_features_df[
    "KOOPMAN_STATE_NORM"
] = state_norm


STRICT_KOOPMAN_FEATURE_COLUMNS = [
    "KOOPMAN_RESIDUAL_MSE",
    "KOOPMAN_RESIDUAL_MAE",
    "KOOPMAN_RESIDUAL_NORM",
    "KOOPMAN_STATE_NORM"
]


# ============================================================
# 13. VALIDATION
# ============================================================

assert (
    len(strict_koopman_features_df)
    == len(forecast_base_df)
)

assert (
    strict_koopman_features_df[
        STRICT_KOOPMAN_FEATURE_COLUMNS
    ].isna().sum().sum()
    == 0
)

assert np.isfinite(
    strict_koopman_features_df[
        STRICT_KOOPMAN_FEATURE_COLUMNS
    ].to_numpy()
).all()


# ============================================================
# 14. REPORT PARTITION STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("STRICT KOOPMAN FEATURE SUMMARY")
print("=" * 70)

for split_name in [
    "train",
    "validation",
    "test"
]:

    mask = (
        forecast_base_df[
            "split"
        ].eq(split_name)
        .to_numpy()
    )

    print(
        f"\n{split_name.upper()}"
    )

    for col in STRICT_KOOPMAN_FEATURE_COLUMNS:

        values = (
            strict_koopman_features_df
            .loc[
                mask,
                col
            ]
            .to_numpy()
        )

        print(
            f"{col:<28} "
            f"mean={values.mean():.6f} | "
            f"median={np.median(values):.6f}"
        )


# ============================================================
# 15. FINAL AUDIT
# ============================================================

print("\n" + "=" * 70)
print("LEAKAGE AUDIT")
print("=" * 70)

print(
    "✓ Base feature construction uses no labels"
)

print(
    "✓ Scaler fitted on training windows only"
)

print(
    "✓ PCA fitted on training windows only"
)

print(
    "✓ Koopman fitted on training transitions only"
)

print(
    "✓ Validation/test never used to fit "
    "temporal representation"
)

print(
    "✓ Exact 30-second transition requirement enforced"
)

print(
    "✓ No cross-file transitions"
)

print(
    "✓ No forecast targets used"
)

print(
    "✓ Strict train-only temporal representation complete"
)


# ============================================================
# 16. SAVE METADATA
# ============================================================

strict_koopman_metadata = {
    "base_feature_count": EXPECTED_BASE_FEATURE_COUNT,
    "pca_components": int(n_components),
    "pca_variance_retained": explained_variance,
    "train_transition_count": int(
        len(train_transition_rows)
    ),
    "window_seconds": WINDOW_SECONDS,
    "fit_partition": "train_only",
    "operator_shape": list(
        K_strict.shape
    ),
}

print(
    "\nMetadata:"
)

print(
    json.dumps(
        strict_koopman_metadata,
        indent=2
    )
)

print(
    "\n✓ CELL 14 COMPLETE"
)

DRISHTI — BOOK 3 | STRICT TRAIN-ONLY PCA + KOOPMAN

Configuration:
Maximum PCA components : 16
Training partition only will be used to fit:
  ✓ StandardScaler
  ✓ PCA
  ✓ Koopman operator

Base feature count: 130

Base feature matrix: (13720, 132)
✓ State features present
✓ Structure features present
✓ Topology features present
✓ Telemetry features present

Forecast source windows: 5,823
✓ Every forecasting source window has complete base features

Window split counts:
split
train         4073
validation     875
test           875
Name: count, dtype: int64

Base matrices:
Train      : (4073, 130)
Validation : (875, 130)
Test       : (875, 130)

✓ StandardScaler fitted using training windows only

✓ PCA fitted using training windows only
Selected components : 16
Variance retained   : 89.6229%

Valid training transitions: 4,017
✓ All transitions are exactly 30 seconds

✓ Koopman operator fitted using training transitions only
Operator shape: (16, 16)

STRICT KOOPMAN FEATURE SUMMARY

TRAI

In [23]:
# ============================================================
# CELL 15 — CORRECTED STRICT FORECASTING SEQUENCES
# ============================================================

print("=" * 70)
print("DRISHTI — BOOK 3 | CORRECTED STRICT FORECASTING SEQUENCES")
print("=" * 70)


# ============================================================
# 1. BUILD FULL 130-FEATURE MATRIX
# ============================================================
#
# IMPORTANT:
# We need ALL 13,720 temporal windows here.
#
# The 5,823 forecasting rows are only the SOURCE windows.
# Their 5-minute histories can contain windows that are not
# themselves forecasting source windows.
# ============================================================

full_base_feature_df = (
    state_features_df[
        [
            "file_id",
            "window_start"
        ]
        + STATE_FEATURE_COLUMNS
    ]
    .merge(
        structure_features_df[
            [
                "file_id",
                "window_start"
            ]
            + STRUCTURE_FEATURE_COLUMNS
        ],
        on=[
            "file_id",
            "window_start"
        ],
        how="inner"
    )
    .merge(
        topology_features_df[
            [
                "file_id",
                "window_start"
            ]
            + TOPOLOGY_FEATURE_COLUMNS
        ],
        on=[
            "file_id",
            "window_start"
        ],
        how="inner"
    )
    .merge(
        telemetry_features_df[
            [
                "file_id",
                "window_start"
            ]
            + TELEMETRY_FEATURE_COLUMNS
        ],
        on=[
            "file_id",
            "window_start"
        ],
        how="inner"
    )
    .sort_values(
        [
            "file_id",
            "window_start"
        ]
    )
    .reset_index(drop=True)
)

assert len(full_base_feature_df) == 13720

assert (
    full_base_feature_df[
        BASE_FEATURE_COLUMNS
    ].isna().sum().sum()
    == 0
)

print(
    f"\nFull base feature matrix: "
    f"{full_base_feature_df.shape}"
)

print(
    "✓ All 13,720 temporal windows available"
)

print(
    "✓ 130 base features present"
)


# ============================================================
# 2. APPLY FROZEN TRAIN-ONLY SCALER
# ============================================================
#
# strict_koopman_scaler was fitted in Cell 14 using only
# training source windows.
#
# We now apply that frozen transformation to ALL windows.
# ============================================================

X_full_base = (
    full_base_feature_df[
        BASE_FEATURE_COLUMNS
    ]
    .to_numpy(
        dtype=np.float32
    )
)

X_full_scaled = (
    strict_koopman_scaler
    .transform(
        X_full_base
    )
    .astype(
        np.float32
    )
)

assert np.isfinite(
    X_full_scaled
).all()

print(
    "\n✓ Frozen train-only scaler applied to "
    "all temporal windows"
)


# ============================================================
# 3. APPLY FROZEN TRAIN-ONLY PCA
# ============================================================
#
# strict_koopman_pca was fitted only on the training
# partition in Cell 14.
# ============================================================

Z_full = (
    strict_koopman_pca
    .transform(
        X_full_scaled
    )
    .astype(
        np.float32
    )
)

assert Z_full.shape == (
    len(full_base_feature_df),
    n_components
)

print(
    f"✓ Frozen train-only PCA applied"
)

print(
    f"Latent representation: "
    f"{Z_full.shape}"
)


# ============================================================
# 4. APPLY FROZEN TRAIN-ONLY KOOPMAN OPERATOR
# ============================================================

Z_full_pred_next = (
    Z_full
    @ K_strict
)

full_residual = (
    Z_full
    - Z_full_pred_next
)

full_residual_mse = np.mean(
    full_residual ** 2,
    axis=1
)

full_residual_mae = np.mean(
    np.abs(
        full_residual
    ),
    axis=1
)

full_residual_norm = np.linalg.norm(
    full_residual,
    axis=1
)

full_state_norm = np.linalg.norm(
    Z_full,
    axis=1
)


# ============================================================
# 5. BUILD FULL STRICT KOOPMAN FEATURE TABLE
# ============================================================

full_strict_koopman_features_df = (
    full_base_feature_df[
        [
            "file_id",
            "window_start"
        ]
    ]
    .copy()
)

full_strict_koopman_features_df[
    "KOOPMAN_RESIDUAL_MSE"
] = full_residual_mse

full_strict_koopman_features_df[
    "KOOPMAN_RESIDUAL_MAE"
] = full_residual_mae

full_strict_koopman_features_df[
    "KOOPMAN_RESIDUAL_NORM"
] = full_residual_norm

full_strict_koopman_features_df[
    "KOOPMAN_STATE_NORM"
] = full_state_norm

assert (
    len(full_strict_koopman_features_df)
    == 13720
)

assert (
    full_strict_koopman_features_df[
        STRICT_KOOPMAN_FEATURE_COLUMNS
    ].isna().sum().sum()
    == 0
)

print(
    "\n✓ Strict Koopman features generated for "
    "all 13,720 windows"
)


# ============================================================
# 6. BUILD FULL 134-FEATURE WINDOW TABLE
# ============================================================

full_strict_window_features_df = (
    full_base_feature_df[
        [
            "file_id",
            "window_start"
        ]
        + BASE_FEATURE_COLUMNS
    ]
    .merge(
        full_strict_koopman_features_df[
            [
                "file_id",
                "window_start"
            ]
            + STRICT_KOOPMAN_FEATURE_COLUMNS
        ],
        on=[
            "file_id",
            "window_start"
        ],
        how="inner",
        validate="one_to_one"
    )
    .sort_values(
        [
            "file_id",
            "window_start"
        ]
    )
    .reset_index(drop=True)
)

STRICT_FORECAST_FEATURE_COLUMNS = (
    BASE_FEATURE_COLUMNS
    + STRICT_KOOPMAN_FEATURE_COLUMNS
)

EXPECTED_STRICT_FEATURE_COUNT = len(
    STRICT_FORECAST_FEATURE_COLUMNS
)

assert (
    EXPECTED_STRICT_FEATURE_COUNT
    == 134
)

assert (
    full_strict_window_features_df[
        STRICT_FORECAST_FEATURE_COLUMNS
    ].isna().sum().sum()
    == 0
)

assert (
    len(full_strict_window_features_df)
    == 13720
)

print(
    f"\nFull strict feature matrix: "
    f"{full_strict_window_features_df.shape}"
)

print(
    "✓ 130 base features"
)

print(
    "✓ 4 strict Koopman features"
)

print(
    "✓ 134 total features"
)


# ============================================================
# 7. CREATE FAST WINDOW LOOKUP
# ============================================================
#
# MultiIndex lookup is retained, but we use reindex()
# rather than .loc[] so we can explicitly verify missing
# history windows.
# ============================================================

strict_window_lookup = (
    full_strict_window_features_df
    .set_index(
        [
            "file_id",
            "window_start"
        ]
    )
    [
        STRICT_FORECAST_FEATURE_COLUMNS
    ]
    .sort_index()
)

print(
    "\n✓ Full temporal feature lookup created"
)


# ============================================================
# 8. REBUILD FORECASTING SEQUENCES
# ============================================================

strict_sequences = []

missing_history = []

for row_idx, row in forecast_sequences_split_df.iterrows():

    file_id = row["file_id"]

    source_time = pd.Timestamp(
        row["source_time"]
    )

    history_times = [
        source_time
        - pd.Timedelta(
            seconds=WINDOW_SECONDS * offset
        )
        for offset in range(
            HISTORY_WINDOWS - 1,
            -1,
            -1
        )
    ]

    history_keys = [
        (
            file_id,
            timestamp
        )
        for timestamp in history_times
    ]

    history_frame = (
        strict_window_lookup
        .reindex(
            history_keys
        )
    )

    if history_frame.isna().any().any():

        missing_keys = [
            history_keys[i]
            for i, missing in enumerate(
                history_frame.isna().all(axis=1)
            )
            if missing
        ]

        missing_history.append(
            {
                "row": row_idx,
                "file_id": file_id,
                "source_time": source_time,
                "missing": missing_keys
            }
        )

        continue

    X_sequence = (
        history_frame
        .to_numpy(
            dtype=np.float32
        )
    )

    assert X_sequence.shape == (
        HISTORY_WINDOWS,
        EXPECTED_STRICT_FEATURE_COUNT
    )

    strict_sequences.append(
        X_sequence
    )


# ============================================================
# 9. HISTORY COMPLETENESS CHECK
# ============================================================

if len(missing_history) > 0:

    print(
        "\nMissing history sequences:",
        len(missing_history)
    )

    print(
        "First missing example:",
        missing_history[0]
    )

    raise AssertionError(
        "Some forecasting sequences do not have "
        "complete 5-minute feature histories."
    )

assert (
    len(strict_sequences)
    == len(forecast_sequences_split_df)
)

print(
    f"\n✓ Reconstructed all "
    f"{len(strict_sequences):,} forecasting sequences"
)


# ============================================================
# 10. STACK RAW STRICT SEQUENCES
# ============================================================

X_strict_raw_all = np.stack(
    strict_sequences
).astype(
    np.float32
)

assert X_strict_raw_all.shape == (
    len(forecast_sequences_split_df),
    HISTORY_WINDOWS,
    EXPECTED_STRICT_FEATURE_COUNT
)

print(
    f"\nRaw strict sequence tensor: "
    f"{X_strict_raw_all.shape}"
)


# ============================================================
# 11. TRAIN / VALIDATION / TEST MASKS
# ============================================================

strict_train_mask = (
    forecast_sequences_split_df[
        "split"
    ]
    .eq("train")
    .to_numpy()
)

strict_val_mask = (
    forecast_sequences_split_df[
        "split"
    ]
    .eq("validation")
    .to_numpy()
)

strict_test_mask = (
    forecast_sequences_split_df[
        "split"
    ]
    .eq("test")
    .to_numpy()
)


# ============================================================
# 12. TRAIN-ONLY FORECAST FEATURE SCALER
# ============================================================
#
# This is separate from the PCA/Koopman scaler.
#
# It is the final scaler presented to the forecasting model.
#
# It is fitted ONLY on training sequence timesteps.
# ============================================================

strict_forecast_scaler = StandardScaler()

X_strict_train_raw = (
    X_strict_raw_all[
        strict_train_mask
    ]
)

X_strict_val_raw = (
    X_strict_raw_all[
        strict_val_mask
    ]
)

X_strict_test_raw = (
    X_strict_raw_all[
        strict_test_mask
    ]
)

strict_forecast_scaler.fit(
    X_strict_train_raw.reshape(
        -1,
        EXPECTED_STRICT_FEATURE_COUNT
    )
)

X_strict_train = (
    strict_forecast_scaler
    .transform(
        X_strict_train_raw.reshape(
            -1,
            EXPECTED_STRICT_FEATURE_COUNT
        )
    )
    .reshape(
        X_strict_train_raw.shape
    )
    .astype(
        np.float32
    )
)

X_strict_val = (
    strict_forecast_scaler
    .transform(
        X_strict_val_raw.reshape(
            -1,
            EXPECTED_STRICT_FEATURE_COUNT
        )
    )
    .reshape(
        X_strict_val_raw.shape
    )
    .astype(
        np.float32
    )

)

X_strict_test = (
    strict_forecast_scaler
    .transform(
        X_strict_test_raw.reshape(
            -1,
            EXPECTED_STRICT_FEATURE_COUNT
        )
    )
    .reshape(
        X_strict_test_raw.shape
    )
    .astype(
        np.float32
    )
)


# ============================================================
# 13. TARGET ARRAYS
# ============================================================

Y_strict = {}

for horizon_name in FORECAST_HORIZONS:

    target_column = (
        f"Y_{horizon_name}_target"
    )

    Y_strict[horizon_name] = {

        "train": (
            forecast_sequences_split_df.loc[
                strict_train_mask,
                target_column
            ]
            .to_numpy()
        ),

        "validation": (
            forecast_sequences_split_df.loc[
                strict_val_mask,
                target_column
            ]
            .to_numpy()
        ),

        "test": (
            forecast_sequences_split_df.loc[
                strict_test_mask,
                target_column
            ]
            .to_numpy()
        )
    }


# ============================================================
# 14. FINAL SHAPE VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL STRICT MODELING TENSORS")
print("=" * 70)

print(
    f"Train      : {X_strict_train.shape}"
)

print(
    f"Validation : {X_strict_val.shape}"
)

print(
    f"Test       : {X_strict_test.shape}"
)

assert X_strict_train.shape == (
    4073,
    HISTORY_WINDOWS,
    134
)

assert X_strict_val.shape == (
    875,
    HISTORY_WINDOWS,
    134
)

assert X_strict_test.shape == (
    875,
    HISTORY_WINDOWS,
    134
)

assert np.isfinite(
    X_strict_train
).all()

assert np.isfinite(
    X_strict_val
).all()

assert np.isfinite(
    X_strict_test
).all()


# ============================================================
# 15. FINAL LEAKAGE AUDIT
# ============================================================

print("\n" + "=" * 70)
print("FINAL LEAKAGE AUDIT")
print("=" * 70)

print(
    "✓ All 13,720 windows available for history construction"
)

print(
    "✓ 5-minute histories reconstructed from full temporal stream"
)

print(
    "✓ 130 base features used"
)

print(
    "✓ Strict train-only PCA used"
)

print(
    "✓ Strict train-only Koopman operator used"
)

print(
    "✓ Strict train-only forecasting scaler used"
)

print(
    "✓ No labels used in feature learning"
)

print(
    "✓ No validation/test data used for fitting"
)

print(
    "✓ Target arrays remain separate from X"
)

print(
    "✓ All forecasting sequences contain exactly "
    f"{HISTORY_WINDOWS} consecutive windows"
)

print(
    "✓ CELL 15 COMPLETE"
)

DRISHTI — BOOK 3 | CORRECTED STRICT FORECASTING SEQUENCES

Full base feature matrix: (13720, 132)
✓ All 13,720 temporal windows available
✓ 130 base features present

✓ Frozen train-only scaler applied to all temporal windows
✓ Frozen train-only PCA applied
Latent representation: (13720, 16)

✓ Strict Koopman features generated for all 13,720 windows

Full strict feature matrix: (13720, 136)
✓ 130 base features
✓ 4 strict Koopman features
✓ 134 total features

✓ Full temporal feature lookup created

✓ Reconstructed all 5,823 forecasting sequences

Raw strict sequence tensor: (5823, 10, 134)

FINAL STRICT MODELING TENSORS
Train      : (4073, 10, 134)
Validation : (875, 10, 134)
Test       : (875, 10, 134)

FINAL LEAKAGE AUDIT
✓ All 13,720 windows available for history construction
✓ 5-minute histories reconstructed from full temporal stream
✓ 130 base features used
✓ Strict train-only PCA used
✓ Strict train-only Koopman operator used
✓ Strict train-only forecasting scaler used
✓ No lab

In [24]:
# ============================================================
# CELL 16 — BASE-RATE / PERSISTENCE FORECASTING BASELINE
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

print("=" * 70)
print("DRISHTI — BOOK 3 | BASE-RATE FORECASTING BASELINE")
print("=" * 70)


# ============================================================
# 1. BASELINE DEFINITION
# ============================================================
#
# The baseline assigns every validation/test sample the
# training-set probability of a positive future onset.
#
# This is deliberately feature-free.
#
# It answers:
#
#   "How well can we do if we know only how frequently
#    future onsets occur?"
#
# Any useful forecasting model should provide additional
# ranking information beyond this floor.
# ============================================================

baseline_results = []


# ============================================================
# 2. EVALUATE EACH HORIZON
# ============================================================

for horizon_name in FORECAST_HORIZONS:

    y_train = Y_strict[
        horizon_name
    ]["train"]

    y_val = Y_strict[
        horizon_name
    ]["validation"]

    y_test = Y_strict[
        horizon_name
    ]["test"]


    # --------------------------------------------------------
    # Remove NaN targets
    # --------------------------------------------------------

    train_valid = pd.notna(
        y_train
    )

    val_valid = pd.notna(
        y_val
    )

    test_valid = pd.notna(
        y_test
    )


    y_train_clean = (
        y_train[
            train_valid
        ]
        .astype(int)
    )

    y_val_clean = (
        y_val[
            val_valid
        ]
        .astype(int)
    )

    y_test_clean = (
        y_test[
            test_valid
        ]
        .astype(int)
    )


    # --------------------------------------------------------
    # Training-set base rate
    # --------------------------------------------------------

    train_positive_rate = (
        y_train_clean.mean()
    )


    # Constant probability prediction
    # for every validation/test observation.

    val_pred = np.full(
        len(y_val_clean),
        train_positive_rate,
        dtype=np.float64
    )

    test_pred = np.full(
        len(y_test_clean),
        train_positive_rate,
        dtype=np.float64
    )


    # --------------------------------------------------------
    # PR-AUC
    # --------------------------------------------------------

    val_pr_auc = (
        average_precision_score(
            y_val_clean,
            val_pred
        )
        if y_val_clean.sum() > 0
        else np.nan
    )

    test_pr_auc = (
        average_precision_score(
            y_test_clean,
            test_pred
        )
        if y_test_clean.sum() > 0
        else np.nan
    )


    # --------------------------------------------------------
    # ROC-AUC
    # --------------------------------------------------------
    #
    # Constant predictions technically have no ranking
    # ability, so ROC-AUC is undefined.
    # --------------------------------------------------------

    val_roc_auc = np.nan
    test_roc_auc = np.nan


    baseline_results.append(
        {
            "horizon": horizon_name,

            "train_positive_rate":
                train_positive_rate,

            "val_samples":
                len(y_val_clean),

            "val_positives":
                int(y_val_clean.sum()),

            "val_positive_rate":
                y_val_clean.mean(),

            "val_pr_auc":
                val_pr_auc,

            "test_samples":
                len(y_test_clean),

            "test_positives":
                int(y_test_clean.sum()),

            "test_positive_rate":
                y_test_clean.mean(),

            "test_pr_auc":
                test_pr_auc,

            "val_roc_auc":
                val_roc_auc,

            "test_roc_auc":
                test_roc_auc,
        }
    )


# ============================================================
# 3. RESULTS TABLE
# ============================================================

baseline_results_df = pd.DataFrame(
    baseline_results
)

print(
    "\n" + "=" * 70
)

print(
    "BASE-RATE BASELINE RESULTS"
)

print(
    "=" * 70
)

display(
    baseline_results_df.round(6)
)


# ============================================================
# 4. EXPLICIT INTERPRETATION
# ============================================================

print(
    "\nInterpretation:"
)

print(
    "✓ This baseline uses no network features."
)

print(
    "✓ It uses only the training-set event frequency."
)

print(
    "✓ PR-AUC is approximately equal to the "
    "test-set positive prevalence."
)

print(
    "✓ Learned models must add ranking information "
    "beyond this floor."
)


# ============================================================
# 5. VALIDATION
# ============================================================

assert (
    len(baseline_results_df)
    == len(FORECAST_HORIZONS)
)

assert (
    baseline_results_df[
        "train_positive_rate"
    ]
    .between(
        0,
        1
    )
    .all()
)

assert (
    baseline_results_df[
        "test_positive_rate"
    ]
    .between(
        0,
        1
    )
    .all()
)


print(
    "\n✓ Base-rate baseline computed for all horizons"
)

print(
    "✓ CELL 16 COMPLETE"
)

DRISHTI — BOOK 3 | BASE-RATE FORECASTING BASELINE

BASE-RATE BASELINE RESULTS


,horizon,train_positive_rate,val_samples,val_positives,val_positive_rate,val_pr_auc,test_samples,test_positives,test_positive_rate,test_pr_auc,val_roc_auc,test_roc_auc
0,30s,0.011799,874,6,0.006865,0.006865,873,17,0.019473,0.019473,NaN,NaN
1,60s,0.015748,873,9,0.010309,0.010309,871,26,0.029851,0.029851,NaN,NaN
2,150s,0.021712,870,14,0.016092,0.016092,864,35,0.040509,0.040509,NaN,NaN
3,300s,0.024209,865,17,0.019653,0.019653,858,40,0.046620,0.046620,NaN,NaN
4,600s,0.023302,855,21,0.024561,0.024561,841,33,0.039239,0.039239,NaN,NaN



Interpretation:
✓ This baseline uses no network features.
✓ It uses only the training-set event frequency.
✓ PR-AUC is approximately equal to the test-set positive prevalence.
✓ Learned models must add ranking information beyond this floor.

✓ Base-rate baseline computed for all horizons
✓ CELL 16 COMPLETE


In [25]:
# ============================================================
# CELL 17 — STATIC LOGISTIC REGRESSION FORECASTING BASELINE
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

print("=" * 70)
print("DRISHTI — BOOK 3 | STATIC LOGISTIC REGRESSION BASELINE")
print("=" * 70)


# ============================================================
# 1. EXTRACT SOURCE-WINDOW FEATURES
# ============================================================
#
# Each sequence has shape:
#
#     (10 historical windows, 134 features)
#
# The static model receives ONLY the final/source window:
#
#     X[:, -1, :]
#
# Therefore it has no access to temporal history.
# ============================================================

X_lr_train = (
    X_strict_train[:, -1, :]
)

X_lr_val = (
    X_strict_val[:, -1, :]
)

X_lr_test = (
    X_strict_test[:, -1, :]
)

print("\nStatic feature matrices:")

print(
    f"Train      : {X_lr_train.shape}"
)

print(
    f"Validation : {X_lr_val.shape}"
)

print(
    f"Test       : {X_lr_test.shape}"
)

assert X_lr_train.shape[1] == 134
assert X_lr_val.shape[1] == 134
assert X_lr_test.shape[1] == 134


# ============================================================
# 2. RESULT STORAGE
# ============================================================

static_lr_models = {}
static_lr_results = []


# ============================================================
# 3. TRAIN ONE MODEL PER HORIZON
# ============================================================

for horizon_name in FORECAST_HORIZONS:

    print(
        "\n" + "-" * 70
    )

    print(
        f"Training static Logistic Regression — "
        f"{horizon_name}"
    )

    # --------------------------------------------------------
    # Targets
    # --------------------------------------------------------

    y_train_full = Y_strict[
        horizon_name
    ]["train"]

    y_val_full = Y_strict[
        horizon_name
    ]["validation"]

    y_test_full = Y_strict[
        horizon_name
    ]["test"]


    # --------------------------------------------------------
    # Valid-target masks
    # --------------------------------------------------------

    train_valid = pd.notna(
        y_train_full
    )

    val_valid = pd.notna(
        y_val_full
    )

    test_valid = pd.notna(
        y_test_full
    )


    X_train = X_lr_train[
        train_valid
    ]

    X_val = X_lr_val[
        val_valid
    ]

    X_test = X_lr_test[
        test_valid
    ]


    y_train = (
        y_train_full[
            train_valid
        ]
        .astype(int)
    )

    y_val = (
        y_val_full[
            val_valid
        ]
        .astype(int)
    )

    y_test = (
        y_test_full[
            test_valid
        ]
        .astype(int)
    )


    # --------------------------------------------------------
    # Class availability check
    # --------------------------------------------------------

    assert len(
        np.unique(y_train)
    ) == 2

    assert len(
        np.unique(y_val)
    ) >= 2

    assert len(
        np.unique(y_test)
    ) >= 2


    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=SEED,
        solver="lbfgs"
    )

    model.fit(
        X_train,
        y_train
    )

    static_lr_models[
        horizon_name
    ] = model


    # --------------------------------------------------------
    # Probability predictions
    # --------------------------------------------------------

    val_pred = model.predict_proba(
        X_val
    )[:, 1]

    test_pred = model.predict_proba(
        X_test
    )[:, 1]


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    val_pr_auc = (
        average_precision_score(
            y_val,
            val_pred
        )
    )

    test_pr_auc = (
        average_precision_score(
            y_test,
            test_pred
        )
    )

    val_roc_auc = (
        roc_auc_score(
            y_val,
            val_pred
        )
    )

    test_roc_auc = (
        roc_auc_score(
            y_test,
            test_pred
        )
    )


    # --------------------------------------------------------
    # Base-rate comparison
    # --------------------------------------------------------

    baseline_row = (
        baseline_results_df[
            baseline_results_df[
                "horizon"
            ] == horizon_name
        ]
    )

    baseline_test_pr = float(
        baseline_row[
            "test_pr_auc"
        ].iloc[0]
    )

    baseline_val_pr = float(
        baseline_row[
            "val_pr_auc"
        ].iloc[0]
    )


    val_pr_lift = (
        val_pr_auc
        / baseline_val_pr
        if baseline_val_pr > 0
        else np.nan
    )

    test_pr_lift = (
        test_pr_auc
        / baseline_test_pr
        if baseline_test_pr > 0
        else np.nan
    )


    static_lr_results.append(
        {
            "horizon":
                horizon_name,

            "train_positives":
                int(y_train.sum()),

            "val_positives":
                int(y_val.sum()),

            "test_positives":
                int(y_test.sum()),

            "val_pr_auc":
                val_pr_auc,

            "test_pr_auc":
                test_pr_auc,

            "val_roc_auc":
                val_roc_auc,

            "test_roc_auc":
                test_roc_auc,

            "val_base_pr_auc":
                baseline_val_pr,

            "test_base_pr_auc":
                baseline_test_pr,

            "val_pr_lift":
                val_pr_lift,

            "test_pr_lift":
                test_pr_lift,
        }
    )


    print(
        f"Validation PR-AUC : "
        f"{val_pr_auc:.6f}"
    )

    print(
        f"Test PR-AUC       : "
        f"{test_pr_auc:.6f}"
    )

    print(
        f"Validation ROC-AUC: "
        f"{val_roc_auc:.6f}"
    )

    print(
        f"Test ROC-AUC      : "
        f"{test_roc_auc:.6f}"
    )

    print(
        f"Test PR-AUC lift  : "
        f"{test_pr_lift:.2f}× base rate"
    )


# ============================================================
# 4. RESULTS TABLE
# ============================================================

static_lr_results_df = pd.DataFrame(
    static_lr_results
)

print(
    "\n" + "=" * 70
)

print(
    "STATIC LOGISTIC REGRESSION RESULTS"
)

print(
    "=" * 70
)

display(
    static_lr_results_df.round(6)
)


# ============================================================
# 5. VALIDATION
# ============================================================

assert (
    len(static_lr_results_df)
    == len(FORECAST_HORIZONS)
)

assert (
    static_lr_results_df[
        "test_pr_auc"
    ]
    .notna()
    .all()
)

assert (
    static_lr_results_df[
        "test_roc_auc"
    ]
    .notna()
    .all()
)


# ============================================================
# 6. SUMMARY
# ============================================================

print(
    "\nInterpretation:"
)

print(
    "✓ Model uses only the current/source window."
)

print(
    "✓ No previous temporal windows are provided."
)

print(
    "✓ Each horizon has its own forecasting classifier."
)

print(
    "✓ PR-AUC is the primary forecasting metric."
)

print(
    "✓ ROC-AUC is reported as a secondary ranking metric."
)

print(
    "✓ Performance is compared against the feature-free "
    "base-rate floor."
)

print(
    "\n✓ CELL 17 COMPLETE"
)

DRISHTI — BOOK 3 | STATIC LOGISTIC REGRESSION BASELINE

Static feature matrices:
Train      : (4073, 134)
Validation : (875, 134)
Test       : (875, 134)

----------------------------------------------------------------------
Training static Logistic Regression — 30s
Validation PR-AUC : 0.322192
Test PR-AUC       : 0.418290
Validation ROC-AUC: 0.789939
Test ROC-AUC      : 0.941589
Test PR-AUC lift  : 21.48× base rate

----------------------------------------------------------------------
Training static Logistic Regression — 60s
Validation PR-AUC : 0.336406
Test PR-AUC       : 0.640451
Validation ROC-AUC: 0.983025
Test ROC-AUC      : 0.951434
Test PR-AUC lift  : 21.46× base rate

----------------------------------------------------------------------
Training static Logistic Regression — 150s
Validation PR-AUC : 0.521479
Test PR-AUC       : 0.907725
Validation ROC-AUC: 0.991489
Test ROC-AUC      : 0.983422
Test PR-AUC lift  : 22.41× base rate

-------------------------------------------

,horizon,train_positives,val_positives,test_positives,val_pr_auc,test_pr_auc,val_roc_auc,test_roc_auc,val_base_pr_auc,test_base_pr_auc,val_pr_lift,test_pr_lift
0,30s,48,6,17,0.322192,0.418290,0.789939,0.941589,0.006865,0.019473,46.932640,21.480416
1,60s,64,9,26,0.336406,0.640451,0.983025,0.951434,0.010309,0.029851,32.631355,21.455106
2,150s,88,14,35,0.521479,0.907725,0.991489,0.983422,0.016092,0.040509,32.406167,22.407837
3,300s,98,17,40,0.690374,1.000000,0.991953,1.000000,0.019653,0.046620,35.127848,21.450000
4,600s,94,21,33,0.923430,1.000000,0.951639,1.000000,0.024561,0.039239,37.596774,25.484848



Interpretation:
✓ Model uses only the current/source window.
✓ No previous temporal windows are provided.
✓ Each horizon has its own forecasting classifier.
✓ PR-AUC is the primary forecasting metric.
✓ ROC-AUC is reported as a secondary ranking metric.
✓ Performance is compared against the feature-free base-rate floor.

✓ CELL 17 COMPLETE


In [32]:
# ============================================================
# CELL 18 — FEATURE-GROUP ABLATION
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

print("=" * 70)
print("DRISHTI — BOOK 3 | FEATURE-GROUP ABLATION")
print("=" * 70)


# ============================================================
# 1. DEFINE FEATURE GROUPS
# ============================================================

feature_groups = {
    "State": (
        STATE_FEATURE_COLUMNS
    ),

    "State + Structure": (
        STATE_FEATURE_COLUMNS
        + STRUCTURE_FEATURE_COLUMNS
    ),

    "State + Structure + Topology": (
        STATE_FEATURE_COLUMNS
        + STRUCTURE_FEATURE_COLUMNS
        + TOPOLOGY_FEATURE_COLUMNS
    ),

    "State + Structure + Topology + Telemetry": (
        STATE_FEATURE_COLUMNS
        + STRUCTURE_FEATURE_COLUMNS
        + TOPOLOGY_FEATURE_COLUMNS
        + TELEMETRY_FEATURE_COLUMNS
    ),

    "All + Koopman": (
        BASE_FEATURE_COLUMNS
        + STRICT_KOOPMAN_FEATURE_COLUMNS
    )
}


# ============================================================
# 2. VERIFY FEATURE COUNTS
# ============================================================

print("\nFeature groups:")

for name, columns in feature_groups.items():

    print(
        f"{name:<45} "
        f"{len(columns):>3} features"
    )

assert len(
    feature_groups["State"]
) == len(
    STATE_FEATURE_COLUMNS
)

assert len(
    feature_groups["All + Koopman"]
) == 134


# ============================================================
# 3. BUILD UN-SCALED RAW SOURCE FEATURES
# ============================================================
#
# We reconstruct source-window features from the strict
# full feature table.
#
# The final source-window matrix is then scaled using a
# TRAIN-ONLY scaler separately for each feature group.
# ============================================================

source_keys = (
    forecast_sequences_split_df[
        [
            "file_id",
            "source_time",
            "split"
        ]
    ]
    .rename(
        columns={
            "source_time": "window_start"
        }
    )
)

source_keys["window_start"] = pd.to_datetime(
    source_keys["window_start"]
)

source_feature_df = (
    source_keys
    .merge(
        full_strict_window_features_df[
            [
                "file_id",
                "window_start"
            ]
            + STRICT_FORECAST_FEATURE_COLUMNS
        ],
        on=[
            "file_id",
            "window_start"
        ],
        how="left",
        validate="one_to_one"
    )
)

assert len(
    source_feature_df
) == len(
    forecast_sequences_split_df
)

assert (
    source_feature_df[
        STRICT_FORECAST_FEATURE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
    == 0
)


# ============================================================
# 4. SPLIT MASKS
# ============================================================

ablation_train_mask = (
    forecast_sequences_split_df[
        "split"
    ]
    .eq("train")
    .to_numpy()
)

ablation_val_mask = (
    forecast_sequences_split_df[
        "split"
    ]
    .eq("validation")
    .to_numpy()
)

ablation_test_mask = (
    forecast_sequences_split_df[
        "split"
    ]
    .eq("test")
    .to_numpy()
)


# ============================================================
# 5. RUN ABLATION
# ============================================================

ablation_results = []

ablation_models = {}


for group_name, columns in feature_groups.items():

    print(
        "\n" + "-" * 70
    )

    print(
        f"FEATURE GROUP: {group_name}"
    )

    print(
        f"Feature count: {len(columns)}"
    )


    # --------------------------------------------------------
    # Source-window feature matrix
    # --------------------------------------------------------

    X_group = (
        source_feature_df[
            columns
        ]
        .to_numpy(
            dtype=np.float32
        )
    )


    X_train_raw = X_group[
        ablation_train_mask
    ]

    X_val_raw = X_group[
        ablation_val_mask
    ]

    X_test_raw = X_group[
        ablation_test_mask
    ]


    # --------------------------------------------------------
    # TRAIN-ONLY SCALING
    # --------------------------------------------------------

    group_scaler = StandardScaler()

    group_scaler.fit(
        X_train_raw
    )

    X_train = group_scaler.transform(
        X_train_raw
    )

    X_val = group_scaler.transform(
        X_val_raw
    )

    X_test = group_scaler.transform(
        X_test_raw
    )


    # --------------------------------------------------------
    # Store models for possible inspection
    # --------------------------------------------------------

    ablation_models[
        group_name
    ] = {}


    # --------------------------------------------------------
    # One model per horizon
    # --------------------------------------------------------

    for horizon_name in FORECAST_HORIZONS:

        y_train_full = Y_strict[
            horizon_name
        ]["train"]

        y_val_full = Y_strict[
            horizon_name
        ]["validation"]

        y_test_full = Y_strict[
            horizon_name
        ]["test"]


        train_valid = pd.notna(
            y_train_full
        )

        val_valid = pd.notna(
            y_val_full
        )

        test_valid = pd.notna(
            y_test_full
        )


        Xtr = X_train[
            train_valid
        ]

        Xv = X_val[
            val_valid
        ]

        Xte = X_test[
            test_valid
        ]


        ytr = (
            y_train_full[
                train_valid
            ]
            .astype(int)
        )

        yv = (
            y_val_full[
                val_valid
            ]
            .astype(int)
        )

        yte = (
            y_test_full[
                test_valid
            ]
            .astype(int)
        )


        model = LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=SEED,
            solver="lbfgs"
        )

        model.fit(
            Xtr,
            ytr
        )

        ablation_models[
            group_name
        ][
            horizon_name
        ] = model


        val_pred = model.predict_proba(
            Xv
        )[:, 1]

        test_pred = model.predict_proba(
            Xte
        )[:, 1]


        val_pr_auc = (
            average_precision_score(
                yv,
                val_pred
            )
        )

        test_pr_auc = (
            average_precision_score(
                yte,
                test_pred
            )
        )

        val_roc_auc = (
            roc_auc_score(
                yv,
                val_pred
            )
        )

        test_roc_auc = (
            roc_auc_score(
                yte,
                test_pred
            )
        )


        ablation_results.append(
            {
                "feature_group":
                    group_name,

                "feature_count":
                    len(columns),

                "horizon":
                    horizon_name,

                "val_pr_auc":
                    val_pr_auc,

                "test_pr_auc":
                    test_pr_auc,

                "val_roc_auc":
                    val_roc_auc,

                "test_roc_auc":
                    test_roc_auc
            }
        )


ablation_results_df = pd.DataFrame(
    ablation_results
)


# ============================================================
# 6. DISPLAY PR-AUC RESULTS
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "ABLATION — TEST PR-AUC"
)

print(
    "=" * 70
)

ablation_pr_table = (
    ablation_results_df
    .pivot(
        index="feature_group",
        columns="horizon",
        values="test_pr_auc"
    )
)

display(
    ablation_pr_table.round(6)
)


# ============================================================
# 7. DISPLAY ROC-AUC RESULTS
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "ABLATION — TEST ROC-AUC"
)

print(
    "=" * 70
)

ablation_roc_table = (
    ablation_results_df
    .pivot(
        index="feature_group",
        columns="horizon",
        values="test_roc_auc"
    )
)

display(
    ablation_roc_table.round(6)
)

# ============================================================
# 8. ALL-FEATURE CROSS-CHECK
# ============================================================
#
# Cell 17 and Cell 18 use the same:
#   - chronological split
#   - targets
#   - feature definitions
#   - Logistic Regression configuration
#
# However, Cell 18 intentionally fits a fresh train-only
# scaler for each feature group. Therefore its "All + Koopman"
# result is expected to be numerically close but not necessarily
# identical to Cell 17.
# ============================================================

all_feature_results = (
    ablation_results_df[
        ablation_results_df[
            "feature_group"
        ] == "All + Koopman"
    ][
        [
            "horizon",
            "test_pr_auc",
            "test_roc_auc"
        ]
    ]
    .rename(
        columns={
            "test_pr_auc":
                "ablation_test_pr_auc",

            "test_roc_auc":
                "ablation_test_roc_auc"
        }
    )
    .merge(
        static_lr_results_df[
            [
                "horizon",
                "test_pr_auc",
                "test_roc_auc"
            ]
        ],
        on="horizon",
        suffixes=(
            "_cell18",
            "_cell17"
        )
    )
)

all_feature_results[
    "pr_auc_difference"
] = (
    all_feature_results[
        "ablation_test_pr_auc"
    ]
    -
    all_feature_results[
        "test_pr_auc"
    ]
)

all_feature_results[
    "roc_auc_difference"
] = (
    all_feature_results[
        "ablation_test_roc_auc"
    ]
    -
    all_feature_results[
        "test_roc_auc"
    ]
)

print(
    "\n" + "=" * 70
)

print(
    "ALL-FEATURE CROSS-CHECK"
)

print(
    "=" * 70
)

display(
    all_feature_results.round(6)
)


# ============================================================
# 9. EXPLAIN PREPROCESSING DIFFERENCE
# ============================================================

print(
    "\nNote:"
)

print(
    "Cell 17 used the sequence-level scaler created in Cell 15."
)

print(
    "Cell 18 fits a fresh train-only scaler for each feature group."
)

print(
    "Therefore exact numerical equality with Cell 17 is not required."
)

print(
    "Both pipelines remain leakage-controlled."
)


# ============================================================
# 10. FINAL VALIDATION
# ============================================================

assert (
    len(ablation_results_df)
    == len(feature_groups)
    * len(FORECAST_HORIZONS)
)

assert (
    ablation_results_df[
        "test_pr_auc"
    ]
    .notna()
    .all()
)

assert (
    ablation_results_df[
        "test_roc_auc"
    ]
    .notna()
    .all()
)

print(
    "\n✓ All feature groups evaluated"
)

print(
    "✓ Train-only scaling used for every feature group"
)

print(
    "✓ No validation/test fitting"
)

print(
    "✓ CELL 18 COMPLETE"
)

DRISHTI — BOOK 3 | FEATURE-GROUP ABLATION

Feature groups:
State                                          46 features
State + Structure                              74 features
State + Structure + Topology                   98 features
State + Structure + Topology + Telemetry      130 features
All + Koopman                                 134 features

----------------------------------------------------------------------
FEATURE GROUP: State
Feature count: 46

----------------------------------------------------------------------
FEATURE GROUP: State + Structure
Feature count: 74

----------------------------------------------------------------------
FEATURE GROUP: State + Structure + Topology
Feature count: 98

----------------------------------------------------------------------
FEATURE GROUP: State + Structure + Topology + Telemetry
Feature count: 130

----------------------------------------------------------------------
FEATURE GROUP: All + Koopman
Feature count: 134

ABLATION —

horizon,150s,300s,30s,600s,60s
feature_group,,,,,
All + Koopman,0.908556,1.000000,0.462411,1.000000,0.666493
State,0.875055,0.971478,0.516798,0.955276,0.652480
State + Structure,0.890477,0.966622,0.390592,0.950331,0.662331
State + Structure + Topology,0.876131,0.976167,0.514881,0.916601,0.702375
State + Structure + Topology + Telemetry,0.907578,1.000000,0.457375,1.000000,0.662393



ABLATION — TEST ROC-AUC


horizon,150s,300s,30s,600s,60s
feature_group,,,,,
All + Koopman,0.970636,1.000000,0.978835,1.000000,0.952208
State,0.968292,0.974725,0.926745,0.968534,0.952663
State + Structure,0.968982,0.974480,0.928326,0.968497,0.989713
State + Structure + Topology,0.968120,0.975031,0.952721,0.936844,0.977606
State + Structure + Topology + Telemetry,0.979907,1.000000,0.982133,1.000000,0.955849



ALL-FEATURE CROSS-CHECK


,horizon,ablation_test_pr_auc,ablation_test_roc_auc,test_pr_auc,test_roc_auc,pr_auc_difference,roc_auc_difference
0,30s,0.462411,0.978835,0.418290,0.941589,0.044121,0.037246
1,60s,0.666493,0.952208,0.640451,0.951434,0.026042,0.000774
2,150s,0.908556,0.970636,0.907725,0.983422,0.000831,-0.012786
3,300s,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000
4,600s,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000



Note:
Cell 17 used the sequence-level scaler created in Cell 15.
Cell 18 fits a fresh train-only scaler for each feature group.
Therefore exact numerical equality with Cell 17 is not required.
Both pipelines remain leakage-controlled.

✓ All feature groups evaluated
✓ Train-only scaling used for every feature group
✓ No validation/test fitting
✓ CELL 18 COMPLETE


In [42]:
# ============================================================
# CELL 19 — TEMPORAL GRU:
#           WITHOUT vs WITH KOOPMAN
#           NaN-safe horizon-specific targets
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import average_precision_score, roc_auc_score

print("=" * 70)
print("DRISHTI — BOOK 3 | TEMPORAL GRU COMPARISON")
print("=" * 70)


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Random seed: {SEED}")


# ============================================================
# 2. STRICT SEQUENCE TENSORS
# ============================================================

Xtr_full = X_strict_train
Xv_full  = X_strict_val
Xte_full = X_strict_test

print("\nFull sequence tensors:")
print("Train:", Xtr_full.shape)
print("Val  :", Xv_full.shape)
print("Test :", Xte_full.shape)

assert Xtr_full.shape == (4073, 10, 134)
assert Xv_full.shape  == (875, 10, 134)
assert Xte_full.shape == (875, 10, 134)

print("✓ Sequence dimensions verified")


# ============================================================
# 3. FEATURE VIEWS
# ============================================================
# 0:46      State
# 46:74     Structure
# 74:98     Topology
# 98:130    Telemetry
# 130:134   Koopman
# ============================================================

Xtr_130 = Xtr_full[:, :, :130]
Xv_130  = Xv_full[:, :, :130]
Xte_130 = Xte_full[:, :, :130]

Xtr_134 = Xtr_full[:, :, :134]
Xv_134  = Xv_full[:, :, :134]
Xte_134 = Xte_full[:, :, :134]

assert Xtr_130.shape == (4073, 10, 130)
assert Xv_130.shape  == (875, 10, 130)
assert Xte_130.shape == (875, 10, 130)

assert Xtr_134.shape == (4073, 10, 134)
assert Xv_134.shape  == (875, 10, 134)
assert Xte_134.shape == (875, 10, 134)

print("\n✓ GRU-130 and GRU-134 feature views verified")


# ============================================================
# 4. GRU MODEL BUILDER
# ============================================================

def build_gru(input_features):

    model = Sequential([
        GRU(
            32,
            input_shape=(10, input_features),
            return_sequences=False
        ),

        Dropout(0.20),

        Dense(
            16,
            activation="relu"
        ),

        Dropout(0.10),

        Dense(
            1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(
                curve="PR",
                name="pr_auc"
            )
        ]
    )

    return model


# ============================================================
# 5. TRAINING / EVALUATION
# ============================================================

gru_results = []
gru_models = {}

for horizon in FORECAST_HORIZONS:

    print("\n" + "=" * 70)
    print(f"HORIZON: {horizon}")
    print("=" * 70)

    # --------------------------------------------------------
    # Canonical strict targets
    # --------------------------------------------------------

    ytr_all = np.asarray(
        Y_strict[horizon]["train"],
        dtype=np.float32
    )

    yv_all = np.asarray(
        Y_strict[horizon]["validation"],
        dtype=np.float32
    )

    yte_all = np.asarray(
        Y_strict[horizon]["test"],
        dtype=np.float32
    )

    # --------------------------------------------------------
    # Horizon-specific valid masks
    # --------------------------------------------------------

    train_mask = np.isfinite(ytr_all)
    val_mask   = np.isfinite(yv_all)
    test_mask  = np.isfinite(yte_all)

    print(
        f"Valid samples | "
        f"train={train_mask.sum()}/{len(train_mask)}, "
        f"val={val_mask.sum()}/{len(val_mask)}, "
        f"test={test_mask.sum()}/{len(test_mask)}"
    )

    # --------------------------------------------------------
    # Apply masks to X AND y
    # --------------------------------------------------------

    Xtr_130_h = Xtr_130[train_mask]
    Xv_130_h  = Xv_130[val_mask]
    Xte_130_h = Xte_130[test_mask]

    Xtr_134_h = Xtr_134[train_mask]
    Xv_134_h  = Xv_134[val_mask]
    Xte_134_h = Xte_134[test_mask]

    ytr = ytr_all[train_mask]
    yv  = yv_all[val_mask]
    yte = yte_all[test_mask]

    # --------------------------------------------------------
    # Target validation
    # --------------------------------------------------------

    assert set(np.unique(ytr)).issubset({0.0, 1.0})
    assert set(np.unique(yv)).issubset({0.0, 1.0})
    assert set(np.unique(yte)).issubset({0.0, 1.0})

    print(
        f"Positive counts | "
        f"train={int(ytr.sum())}, "
        f"val={int(yv.sum())}, "
        f"test={int(yte.sum())}"
    )

    # --------------------------------------------------------
    # Train both temporal models
    # --------------------------------------------------------

    for model_name, Xtr, Xv, Xte in [
        (
            "GRU-130",
            Xtr_130_h,
            Xv_130_h,
            Xte_130_h
        ),
        (
            "GRU-134",
            Xtr_134_h,
            Xv_134_h,
            Xte_134_h
        )
    ]:

        print("\n" + "-" * 70)
        print(f"Training {model_name} | {horizon}")
        print("-" * 70)

        tf.keras.backend.clear_session()

        np.random.seed(SEED)
        tf.random.set_seed(SEED)

        model = build_gru(
            input_features=Xtr.shape[-1]
        )

        early_stop = EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=1
        )

        history = model.fit(
            Xtr,
            ytr,
            validation_data=(Xv, yv),
            epochs=50,
            batch_size=64,
            shuffle=False,
            callbacks=[early_stop],
            verbose=0
        )

        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        val_prob = model.predict(
            Xv,
            verbose=0
        ).ravel()

        test_prob = model.predict(
            Xte,
            verbose=0
        ).ravel()

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        val_pr = average_precision_score(
            yv,
            val_prob
        )

        test_pr = average_precision_score(
            yte,
            test_prob
        )

        val_roc = roc_auc_score(
            yv,
            val_prob
        )

        test_roc = roc_auc_score(
            yte,
            test_prob
        )

        best_epoch = (
            np.argmax(
                history.history["val_pr_auc"]
            ) + 1
        )

        gru_results.append({
            "model": model_name,
            "horizon": horizon,
            "train_n": len(ytr),
            "val_n": len(yv),
            "test_n": len(yte),
            "train_pos": int(ytr.sum()),
            "val_pos": int(yv.sum()),
            "test_pos": int(yte.sum()),
            "val_pr_auc": val_pr,
            "test_pr_auc": test_pr,
            "val_roc_auc": val_roc,
            "test_roc_auc": test_roc,
            "best_epoch": best_epoch
        })

        gru_models[
            (model_name, horizon)
        ] = model

        print(
            f"{model_name} | "
            f"Val PR={val_pr:.6f} | "
            f"Test PR={test_pr:.6f} | "
            f"Val ROC={val_roc:.6f} | "
            f"Test ROC={test_roc:.6f} | "
            f"Best epoch={best_epoch}"
        )


# ============================================================
# 6. RESULTS
# ============================================================

gru_results_df = pd.DataFrame(
    gru_results
)

print("\n" + "=" * 70)
print("GRU RESULTS")
print("=" * 70)

display(
    gru_results_df
    .sort_values(
        ["horizon", "model"]
    )
    .round(6)
)


# ============================================================
# 7. GRU-130 vs GRU-134
# ============================================================

gru_comparison = (
    gru_results_df
    .pivot(
        index="horizon",
        columns="model",
        values=[
            "val_pr_auc",
            "test_pr_auc",
            "val_roc_auc",
            "test_roc_auc"
        ]
    )
)

print("\n" + "=" * 70)
print("GRU-130 vs GRU-134")
print("=" * 70)

display(
    gru_comparison.round(6)
)


# ============================================================
# 8. KOOPMAN INCREMENTAL EFFECT
# ============================================================

gru_130 = (
    gru_results_df[
        gru_results_df["model"] == "GRU-130"
    ][
        [
            "horizon",
            "val_pr_auc",
            "test_pr_auc",
            "val_roc_auc",
            "test_roc_auc"
        ]
    ]
    .rename(
        columns={
            "val_pr_auc": "gru130_val_pr_auc",
            "test_pr_auc": "gru130_test_pr_auc",
            "val_roc_auc": "gru130_val_roc_auc",
            "test_roc_auc": "gru130_test_roc_auc"
        }
    )
)

gru_134 = (
    gru_results_df[
        gru_results_df["model"] == "GRU-134"
    ][
        [
            "horizon",
            "val_pr_auc",
            "test_pr_auc",
            "val_roc_auc",
            "test_roc_auc"
        ]
    ]
    .rename(
        columns={
            "val_pr_auc": "gru134_val_pr_auc",
            "test_pr_auc": "gru134_test_pr_auc",
            "val_roc_auc": "gru134_val_roc_auc",
            "test_roc_auc": "gru134_test_roc_auc"
        }
    )
)

koopman_delta = (
    gru_130
    .merge(
        gru_134,
        on="horizon"
    )
)

koopman_delta["val_pr_auc_delta"] = (
    koopman_delta["gru134_val_pr_auc"]
    - koopman_delta["gru130_val_pr_auc"]
)

koopman_delta["test_pr_auc_delta"] = (
    koopman_delta["gru134_test_pr_auc"]
    - koopman_delta["gru130_test_pr_auc"]
)

koopman_delta["val_roc_auc_delta"] = (
    koopman_delta["gru134_val_roc_auc"]
    - koopman_delta["gru130_val_roc_auc"]
)

koopman_delta["test_roc_auc_delta"] = (
    koopman_delta["gru134_test_roc_auc"]
    - koopman_delta["gru130_test_roc_auc"]
)

print("\n" + "=" * 70)
print("KOOPMAN INCREMENTAL EFFECT")
print("=" * 70)

display(
    koopman_delta.round(6)
)


# ============================================================
# 9. FINAL SANITY CHECKS
# ============================================================

assert len(gru_results_df) == (
    2 * len(FORECAST_HORIZONS)
)

assert gru_results_df[
    "test_pr_auc"
].notna().all()

assert gru_results_df[
    "test_roc_auc"
].notna().all()

assert koopman_delta[
    "test_pr_auc_delta"
].notna().all()

print("\n" + "=" * 70)
print("✓ GRU-130 evaluated")
print("✓ GRU-134 evaluated")
print("✓ Horizon-specific NaNs excluded")
print("✓ NaNs never converted to benign")
print("✓ Same temporal history")
print("✓ Same chronological split")
print("✓ Same strict targets")
print("✓ No test-set fitting")
print("✓ Koopman incremental deltas calculated")
print("✓ CELL 19 COMPLETE")
print("=" * 70)

DRISHTI — BOOK 3 | TEMPORAL GRU COMPARISON
Random seed: 42

Full sequence tensors:
Train: (4073, 10, 134)
Val  : (875, 10, 134)
Test : (875, 10, 134)
✓ Sequence dimensions verified

✓ GRU-130 and GRU-134 feature views verified

HORIZON: 30s
Valid samples | train=4068/4073, val=874/875, test=873/875
Positive counts | train=48, val=6, test=17

----------------------------------------------------------------------
Training GRU-130 | 30s
----------------------------------------------------------------------


2026-09-20 06:41:59.299346: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 26: early stopping
Restoring model weights from the end of the best epoch: 18.
GRU-130 | Val PR=0.407492 | Test PR=0.388342 | Val ROC=0.989439 | Test ROC=0.980690 | Best epoch=18

----------------------------------------------------------------------
Training GRU-134 | 30s
----------------------------------------------------------------------
Epoch 22: early stopping
Restoring model weights from the end of the best epoch: 14.
GRU-134 | Val PR=0.562500 | Test PR=0.447679 | Val ROC=0.993472 | Test ROC=0.982752 | Best epoch=14

HORIZON: 60s
Valid samples | train=4064/4073, val=873/875, test=871/875
Positive counts | train=64, val=9, test=26

----------------------------------------------------------------------
Training GRU-130 | 60s
----------------------------------------------------------------------
Epoch 30: early stopping
Restoring model weights from the end of the best epoch: 22.
GRU-130 | Val PR=0.802632 | Test PR=0.712468 | Val ROC=0.996528 | Test ROC=0.989850 | Best epoch=

,model,horizon,train_n,val_n,test_n,train_pos,val_pos,test_pos,val_pr_auc,test_pr_auc,val_roc_auc,test_roc_auc,best_epoch
4,GRU-130,150s,4053,870,864,88,14,35,0.702525,0.887382,0.994826,0.996347,7
5,GRU-134,150s,4053,870,864,88,14,35,0.651545,0.779661,0.994075,0.994348,6
6,GRU-130,300s,4048,865,858,98,17,40,0.896849,1.000000,0.997919,1.000000,8
7,GRU-134,300s,4048,865,858,98,17,40,0.807756,1.000000,0.997017,1.000000,7
0,GRU-130,30s,4068,874,873,48,6,17,0.407492,0.388342,0.989439,0.980690,18
1,GRU-134,30s,4068,874,873,48,6,17,0.562500,0.447679,0.993472,0.982752,14
8,GRU-130,600s,4034,855,841,94,21,33,1.000000,1.000000,1.000000,1.000000,8
9,GRU-134,600s,4034,855,841,94,21,33,1.000000,1.000000,1.000000,1.000000,12
2,GRU-130,60s,4064,873,871,64,9,26,0.802632,0.712468,0.996528,0.989850,22
3,GRU-134,60s,4064,873,871,64,9,26,0.324597,0.672451,0.989326,0.987984,15



GRU-130 vs GRU-134


val_pr_auc           test_pr_auc           val_roc_auc            \
model      GRU-130   GRU-134     GRU-130   GRU-134     GRU-130   GRU-134   
horizon                                                                    
150s      0.702525  0.651545    0.887382  0.779661    0.994826  0.994075   
300s      0.896849  0.807756    1.000000  1.000000    0.997919  0.997017   
30s       0.407492  0.562500    0.388342  0.447679    0.989439  0.993472   
600s      1.000000  1.000000    1.000000  1.000000    1.000000  1.000000   
60s       0.802632  0.324597    0.712468  0.672451    0.996528  0.989326   

        test_roc_auc            
model        GRU-130   GRU-134  
horizon                         
150s        0.996347  0.994348  
300s        1.000000  1.000000  
30s         0.980690  0.982752  
600s        1.000000  1.000000  
60s         0.989850  0.987984


KOOPMAN INCREMENTAL EFFECT


,horizon,gru130_val_pr_auc,gru130_test_pr_auc,gru130_val_roc_auc,gru130_test_roc_auc,gru134_val_pr_auc,gru134_test_pr_auc,gru134_val_roc_auc,gru134_test_roc_auc,val_pr_auc_delta,test_pr_auc_delta,val_roc_auc_delta,test_roc_auc_delta
0,30s,0.407492,0.388342,0.989439,0.980690,0.562500,0.447679,0.993472,0.982752,0.155008,0.059337,0.004032,0.002062
1,60s,0.802632,0.712468,0.996528,0.989850,0.324597,0.672451,0.989326,0.987984,-0.478034,-0.040017,-0.007202,-0.001866
2,150s,0.702525,0.887382,0.994826,0.996347,0.651545,0.779661,0.994075,0.994348,-0.050980,-0.107721,-0.000751,-0.001999
3,300s,0.896849,1.000000,0.997919,1.000000,0.807756,1.000000,0.997017,1.000000,-0.089093,0.000000,-0.000902,0.000000
4,600s,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000



✓ GRU-130 evaluated
✓ GRU-134 evaluated
✓ Horizon-specific NaNs excluded
✓ NaNs never converted to benign
✓ Same temporal history
✓ Same chronological split
✓ Same strict targets
✓ No test-set fitting
✓ Koopman incremental deltas calculated
✓ CELL 19 COMPLETE


In [43]:
# ============================================================
# CELL 20 — OPERATIONAL EARLY-WARNING EVALUATION
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    confusion_matrix
)

print("=" * 70)
print("DRISHTI — BOOK 3 | OPERATIONAL EARLY-WARNING EVALUATION")
print("=" * 70)


# ============================================================
# 1. OPERATING POINTS
# ============================================================
# We evaluate fixed probability thresholds rather than
# selecting a threshold on the test set.
#
# This prevents test-set threshold tuning.
# ============================================================

THRESHOLDS = [
    0.30,
    0.50,
    0.70,
    0.90
]

print("Fixed alert thresholds:", THRESHOLDS)


# ============================================================
# 2. HELPER
# ============================================================

def evaluate_threshold(y_true, y_prob, threshold):

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    return {
        "threshold": threshold,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "precision": precision,
        "recall": recall,
        "fpr": fpr,
        "specificity": specificity
    }


# ============================================================
# 3. GENERATE TEST PREDICTIONS
# ============================================================

early_warning_results = []

test_predictions = {}

for horizon in FORECAST_HORIZONS:

    # --------------------------------------------------------
    # Horizon-specific valid test samples
    # --------------------------------------------------------

    y_test_all = np.asarray(
        Y_strict[horizon]["test"],
        dtype=np.float32
    )

    valid_mask = np.isfinite(
        y_test_all
    )

    y_test_h = y_test_all[
        valid_mask
    ].astype(int)

    # --------------------------------------------------------
    # Correct feature tensor for this horizon
    # --------------------------------------------------------

    X130_h = Xte_130[
        valid_mask
    ]

    X134_h = Xte_134[
        valid_mask
    ]

    for model_name, X_h in [
        ("GRU-130", X130_h),
        ("GRU-134", X134_h)
    ]:

        model = gru_models[
            (model_name, horizon)
        ]

        test_prob = model.predict(
            X_h,
            verbose=0
        ).ravel()

        test_predictions[
            (model_name, horizon)
        ] = {
            "y_true": y_test_h,
            "prob": test_prob
        }

        # ----------------------------------------------------
        # Evaluate fixed thresholds
        # ----------------------------------------------------

        for threshold in THRESHOLDS:

            metrics = evaluate_threshold(
                y_test_h,
                test_prob,
                threshold
            )

            metrics.update({
                "model": model_name,
                "horizon": horizon,
                "test_n": len(y_test_h),
                "positive_n": int(y_test_h.sum()),
                "positive_rate": float(y_test_h.mean())
            })

            early_warning_results.append(
                metrics
            )


# ============================================================
# 4. RESULTS
# ============================================================

early_warning_df = pd.DataFrame(
    early_warning_results
)

print("\n" + "=" * 70)
print("FIXED-THRESHOLD EARLY-WARNING RESULTS")
print("=" * 70)

display(
    early_warning_df[
        [
            "model",
            "horizon",
            "threshold",
            "TP",
            "FP",
            "FN",
            "TN",
            "precision",
            "recall",
            "fpr",
            "specificity"
        ]
    ]
    .sort_values(
        ["horizon", "model", "threshold"]
    )
    .round(4)
)


# ============================================================
# 5. HIGH-CONFIDENCE OPERATING POINT
# ============================================================
# 0.90 threshold gives us a simple "high confidence warning"
# operating point without optimizing against the test set.
# ============================================================

high_confidence = (
    early_warning_df[
        early_warning_df["threshold"] == 0.90
    ][
        [
            "model",
            "horizon",
            "TP",
            "FP",
            "FN",
            "TN",
            "precision",
            "recall",
            "fpr"
        ]
    ]
    .sort_values(
        ["horizon", "model"]
    )
)

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE WARNING OPERATING POINT (THRESHOLD = 0.90)")
print("=" * 70)

display(
    high_confidence.round(4)
)


# ============================================================
# 6. BEST FIXED THRESHOLD — DESCRIPTIVE ONLY
# ============================================================
# This section does NOT select a threshold for deployment.
# It simply identifies which of the pre-specified thresholds
# produced the highest test recall for descriptive analysis.
#
# This is reported only as a diagnostic.
# ============================================================

best_recall_diagnostic = (
    early_warning_df
    .sort_values(
        [
            "model",
            "horizon",
            "recall"
        ],
        ascending=[True, True, False]
    )
    .groupby(
        ["model", "horizon"],
        as_index=False
    )
    .first()
)

print("\n" + "=" * 70)
print("BEST RECALL AMONG PRE-SPECIFIED THRESHOLDS")
print("=" * 70)

display(
    best_recall_diagnostic[
        [
            "model",
            "horizon",
            "threshold",
            "precision",
            "recall",
            "fpr",
            "TP",
            "FP",
            "FN",
            "TN"
        ]
    ]
    .round(4)
)


# ============================================================
# 7. ROC OPERATING-POINT SUMMARY
# ============================================================

roc_summary = []

for horizon in FORECAST_HORIZONS:

    y_test_all = np.asarray(
        Y_strict[horizon]["test"],
        dtype=np.float32
    )

    valid_mask = np.isfinite(
        y_test_all
    )

    y_test_h = y_test_all[
        valid_mask
    ].astype(int)

    for model_name in [
        "GRU-130",
        "GRU-134"
    ]:

        probs = test_predictions[
            (model_name, horizon)
        ]["prob"]

        fpr, tpr, thresholds = roc_curve(
            y_test_h,
            probs
        )

        roc_summary.append({
            "model": model_name,
            "horizon": horizon,
            "n_roc_points": len(fpr),
            "max_tpr": float(tpr.max()),
            "min_fpr_at_tpr_max": float(
                fpr[np.argmax(tpr)]
            )
        })

roc_summary_df = pd.DataFrame(
    roc_summary
)

print("\n" + "=" * 70)
print("ROC OPERATING-POINT SUMMARY")
print("=" * 70)

display(
    roc_summary_df.round(4)
)


# ============================================================
# 8. SANITY CHECKS
# ============================================================

assert len(early_warning_df) == (
    len(FORECAST_HORIZONS)
    * 2
    * len(THRESHOLDS)
)

assert early_warning_df[
    "precision"
].between(0, 1).all()

assert early_warning_df[
    "recall"
].between(0, 1).all()

assert early_warning_df[
    "fpr"
].between(0, 1).all()

assert early_warning_df[
    "TP"
].ge(0).all()

assert early_warning_df[
    "FP"
].ge(0).all()

print("\n" + "=" * 70)
print("✓ Fixed thresholds evaluated")
print("✓ No threshold tuned on test data")
print("✓ Precision calculated")
print("✓ Recall calculated")
print("✓ False-positive rate calculated")
print("✓ True/false alarm counts calculated")
print("✓ CELL 20 COMPLETE")
print("=" * 70)

DRISHTI — BOOK 3 | OPERATIONAL EARLY-WARNING EVALUATION
Fixed alert thresholds: [0.3, 0.5, 0.7, 0.9]

FIXED-THRESHOLD EARLY-WARNING RESULTS


,model,horizon,threshold,TP,FP,FN,TN,precision,recall,fpr,specificity
16,GRU-130,150s,0.3,35,6,0,823,0.8537,1.0000,0.0072,0.9928
17,GRU-130,150s,0.5,35,6,0,823,0.8537,1.0000,0.0072,0.9928
18,GRU-130,150s,0.7,33,6,2,823,0.8462,0.9429,0.0072,0.9928
19,GRU-130,150s,0.9,21,4,14,825,0.8400,0.6000,0.0048,0.9952
20,GRU-134,150s,0.3,35,6,0,823,0.8537,1.0000,0.0072,0.9928
21,GRU-134,150s,0.5,35,6,0,823,0.8537,1.0000,0.0072,0.9928
22,GRU-134,150s,0.7,33,6,2,823,0.8462,0.9429,0.0072,0.9928
23,GRU-134,150s,0.9,16,6,19,823,0.7273,0.4571,0.0072,0.9928
24,GRU-130,300s,0.3,38,0,2,818,1.0000,0.9500,0.0000,1.0000
25,GRU-130,300s,0.5,38,0,2,818,1.0000,0.9500,0.0000,1.0000



HIGH-CONFIDENCE WARNING OPERATING POINT (THRESHOLD = 0.90)


,model,horizon,TP,FP,FN,TN,precision,recall,fpr
19,GRU-130,150s,21,4,14,825,0.8400,0.6000,0.0048
23,GRU-134,150s,16,6,19,823,0.7273,0.4571,0.0072
27,GRU-130,300s,15,0,25,818,1.0000,0.3750,0.0000
31,GRU-134,300s,35,0,5,818,1.0000,0.8750,0.0000
3,GRU-130,30s,0,0,17,856,0.0000,0.0000,0.0000
7,GRU-134,30s,0,0,17,856,0.0000,0.0000,0.0000
35,GRU-130,600s,1,0,32,808,1.0000,0.0303,0.0000
39,GRU-134,600s,17,0,16,808,1.0000,0.5152,0.0000
11,GRU-130,60s,5,0,21,845,1.0000,0.1923,0.0000
15,GRU-134,60s,3,0,23,845,1.0000,0.1154,0.0000



BEST RECALL AMONG PRE-SPECIFIED THRESHOLDS


,model,horizon,threshold,precision,recall,fpr,TP,FP,FN,TN
0,GRU-130,150s,0.3,0.8537,1.0000,0.0072,35,6,0,823
1,GRU-130,300s,0.3,1.0000,0.9500,0.0000,38,0,2,818
2,GRU-130,30s,0.3,0.3333,0.6471,0.0257,11,22,6,834
3,GRU-130,600s,0.3,1.0000,1.0000,0.0000,33,0,0,808
4,GRU-130,60s,0.3,0.6061,0.7692,0.0154,20,13,6,832
5,GRU-134,150s,0.3,0.8537,1.0000,0.0072,35,6,0,823
6,GRU-134,300s,0.3,1.0000,1.0000,0.0000,40,0,0,818
7,GRU-134,30s,0.3,0.3684,0.8235,0.0280,14,24,3,832
8,GRU-134,600s,0.3,1.0000,1.0000,0.0000,33,0,0,808
9,GRU-134,60s,0.3,0.5814,0.9615,0.0213,25,18,1,827



ROC OPERATING-POINT SUMMARY


,model,horizon,n_roc_points,max_tpr,min_fpr_at_tpr_max
0,GRU-130,30s,23,1.0,0.0339
1,GRU-134,30s,25,1.0,0.0339
2,GRU-130,60s,24,1.0,0.0225
3,GRU-134,60s,22,1.0,0.0213
4,GRU-130,150s,16,1.0,0.0072
5,GRU-134,150s,15,1.0,0.0072
6,GRU-130,300s,6,1.0,0.0000
7,GRU-134,300s,4,1.0,0.0000
8,GRU-130,600s,8,1.0,0.0000
9,GRU-134,600s,6,1.0,0.0000



✓ Fixed thresholds evaluated
✓ No threshold tuned on test data
✓ Precision calculated
✓ Recall calculated
✓ False-positive rate calculated
✓ True/false alarm counts calculated
✓ CELL 20 COMPLETE


In [48]:
# ======================================================================
# DRISHTI — BOOK 3 — CELL 21
# EVENT-LEVEL EARLY WARNING & LEAD-TIME ANALYSIS
# CORRECTED MODEL INPUT HANDLING
# ======================================================================

import numpy as np
import pandas as pd

EVENT_THRESHOLDS = [0.30, 0.50, 0.70, 0.90]

print("=" * 78)
print("DRISHTI — EVENT-LEVEL EARLY WARNING & LEAD-TIME ANALYSIS")
print("=" * 78)


# ----------------------------------------------------------------------
# 1. Canonical test metadata
# ----------------------------------------------------------------------

required_columns = [
    "file_id",
    "source_time",
    "Y_30s_target",
    "Y_30s_onset_time",
    "Y_60s_target",
    "Y_60s_onset_time",
    "Y_150s_target",
    "Y_150s_onset_time",
    "Y_300s_target",
    "Y_300s_onset_time",
    "Y_600s_target",
    "Y_600s_onset_time",
]

missing = [
    c for c in required_columns
    if c not in forecast_sequences_df.columns
]

if missing:
    raise RuntimeError(
        f"Missing required columns: {missing}"
    )

test_meta = (
    forecast_sequences_df
    .loc[strict_test_mask]
    .copy()
    .reset_index(drop=True)
)

test_meta["source_time"] = pd.to_datetime(
    test_meta["source_time"]
)

for horizon in FORECAST_HORIZONS.keys():

    test_meta[
        f"Y_{horizon}_onset_time"
    ] = pd.to_datetime(
        test_meta[
            f"Y_{horizon}_onset_time"
        ],
        errors="coerce"
    )

assert len(test_meta) == len(X_strict_test)

print(f"\n✓ Test sequences: {len(test_meta):,}")


# ----------------------------------------------------------------------
# 2. Model input configuration
# ----------------------------------------------------------------------

MODEL_FEATURES = {
    "GRU-130": 130,
    "GRU-134": 134
}

assert X_strict_test.shape[2] == 134

print("\nModel input dimensions:")
for model_name, n_features in MODEL_FEATURES.items():
    print(f"  {model_name}: {n_features} features")


# ----------------------------------------------------------------------
# 3. Event-level evaluation
# ----------------------------------------------------------------------

event_summary_rows = []
event_detail_rows = []


for horizon in FORECAST_HORIZONS.keys():

    horizon_seconds = int(
        horizon.replace("s", "")
    )

    target_col = f"Y_{horizon}_target"
    onset_col = f"Y_{horizon}_onset_time"

    # --------------------------------------------------------------
    # Horizon-valid test rows
    # --------------------------------------------------------------

    y_all = np.asarray(
        Y_strict[horizon]["test"],
        dtype=np.float32
    )

    valid_mask = np.isfinite(y_all)

    X_valid_134 = X_strict_test[
        valid_mask
    ]

    meta_h = test_meta.iloc[
        np.flatnonzero(valid_mask)
    ].copy().reset_index(drop=True)

    print(
        f"\n{horizon}: "
        f"{len(meta_h):,} valid test sequences"
    )

    # --------------------------------------------------------------
    # Ground-truth future-onset sequences
    # --------------------------------------------------------------

    onset_rows = meta_h[
        (meta_h[target_col] == 1)
        & meta_h[onset_col].notna()
    ].copy()

    print(
        f"  Future-onset sequences: "
        f"{len(onset_rows):,}"
    )

    # --------------------------------------------------------------
    # Both GRUs
    # --------------------------------------------------------------

    for model_name, n_features in MODEL_FEATURES.items():

        # IMPORTANT:
        # GRU-130 was trained on the first 130 features.
        # GRU-134 was trained on all 134 features.

        X_model = X_valid_134[:, :, :n_features]

        model = gru_models[
            (model_name, horizon)
        ]

        # Safety check against the actual model input.
        expected_features = (
            model.input_shape[-1]
        )

        assert expected_features == n_features, (
            f"{model_name} expects {expected_features}, "
            f"but configured for {n_features}"
        )

        probabilities = model.predict(
            X_model,
            verbose=0
        ).ravel()

        meta_pred = meta_h.copy()

        meta_pred["_probability"] = probabilities


        # ----------------------------------------------------------
        # Fixed thresholds
        # ----------------------------------------------------------

        for threshold in EVENT_THRESHOLDS:

            # Group sequences by the actual attack onset.
            #
            # Multiple source windows can forecast the same onset.
            # They therefore represent ONE event.

            grouped_events = {}

            for _, row in onset_rows.iterrows():

                file_id = str(row["file_id"])
                onset_time = row[onset_col]

                event_key = (
                    file_id,
                    pd.Timestamp(onset_time)
                )

                grouped_events.setdefault(
                    event_key,
                    []
                ).append(row)


            events_evaluated = 0
            events_detected = 0
            events_missed = 0

            lead_times = []


            # ------------------------------------------------------
            # Evaluate every genuine onset event
            # ------------------------------------------------------

            for (file_id, onset_time), rows in grouped_events.items():

                candidates = meta_pred[
                    meta_pred["file_id"].astype(str).eq(
                        file_id
                    )
                ].copy()

                if candidates.empty:
                    continue

                # Prediction must occur BEFORE onset.
                candidates = candidates[
                    candidates["source_time"]
                    < onset_time
                ].copy()

                # Calculate actual lead time.
                candidates["_lead_seconds"] = (
                    onset_time
                    - candidates["source_time"]
                ).dt.total_seconds()

                # Only predictions within this forecast horizon.
                candidates = candidates[
                    (candidates["_lead_seconds"] > 0)
                    & (
                        candidates["_lead_seconds"]
                        <= horizon_seconds
                    )
                ].copy()

                if candidates.empty:
                    continue

                events_evaluated += 1

                # --------------------------------------------------
                # First alert before onset
                # --------------------------------------------------

                alerts = candidates[
                    candidates["_probability"] >= threshold
                ].sort_values(
                    "source_time"
                )

                if alerts.empty:

                    events_missed += 1

                    event_detail_rows.append({
                        "model": model_name,
                        "horizon": horizon,
                        "threshold": threshold,
                        "file_id": file_id,
                        "onset_time": onset_time,
                        "warning_time": pd.NaT,
                        "lead_seconds": np.nan,
                        "lead_minutes": np.nan,
                        "detected": 0
                    })

                else:

                    first_alert = alerts.iloc[0]

                    lead_seconds = float(
                        first_alert["_lead_seconds"]
                    )

                    events_detected += 1
                    lead_times.append(
                        lead_seconds
                    )

                    event_detail_rows.append({
                        "model": model_name,
                        "horizon": horizon,
                        "threshold": threshold,
                        "file_id": file_id,
                        "onset_time": onset_time,
                        "warning_time": first_alert[
                            "source_time"
                        ],
                        "lead_seconds": lead_seconds,
                        "lead_minutes": (
                            lead_seconds / 60.0
                        ),
                        "detected": 1
                    })


            # ------------------------------------------------------
            # Summary statistics
            # ------------------------------------------------------

            lead_arr = np.asarray(
                lead_times,
                dtype=float
            )

            if len(lead_arr) > 0:

                median_lead = float(
                    np.median(lead_arr)
                )

                mean_lead = float(
                    np.mean(lead_arr)
                )

                min_lead = float(
                    np.min(lead_arr)
                )

                max_lead = float(
                    np.max(lead_arr)
                )

            else:

                median_lead = np.nan
                mean_lead = np.nan
                min_lead = np.nan
                max_lead = np.nan


            detection_rate = (
                events_detected / events_evaluated
                if events_evaluated > 0
                else np.nan
            )


            event_summary_rows.append({
                "model": model_name,
                "horizon": horizon,
                "threshold": threshold,
                "events_evaluated": events_evaluated,
                "events_detected": events_detected,
                "events_missed": events_missed,
                "event_detection_rate": detection_rate,
                "median_lead_seconds": median_lead,
                "mean_lead_seconds": mean_lead,
                "min_lead_seconds": min_lead,
                "max_lead_seconds": max_lead
            })


# ----------------------------------------------------------------------
# 4. Results
# ----------------------------------------------------------------------

event_warning_df = pd.DataFrame(
    event_summary_rows
)

event_warning_details_df = pd.DataFrame(
    event_detail_rows
)


print("\n" + "=" * 78)
print("EVENT-LEVEL EARLY WARNING RESULTS")
print("=" * 78)

display(
    event_warning_df[
        [
            "model",
            "horizon",
            "threshold",
            "events_evaluated",
            "events_detected",
            "events_missed",
            "event_detection_rate",
            "median_lead_seconds",
            "mean_lead_seconds",
            "min_lead_seconds",
            "max_lead_seconds"
        ]
    ]
    .sort_values(
        ["horizon", "model", "threshold"]
    )
    .round(4)
)


# ----------------------------------------------------------------------
# 5. High-confidence operating point
# ----------------------------------------------------------------------

high_conf_event_warning = event_warning_df[
    event_warning_df["threshold"] == 0.90
].copy()

print("\n" + "=" * 78)
print("HIGH-CONFIDENCE EVENT WARNING — THRESHOLD = 0.90")
print("=" * 78)

display(
    high_conf_event_warning[
        [
            "model",
            "horizon",
            "events_evaluated",
            "events_detected",
            "events_missed",
            "event_detection_rate",
            "median_lead_seconds",
            "mean_lead_seconds",
            "min_lead_seconds",
            "max_lead_seconds"
        ]
    ]
    .sort_values(
        ["horizon", "model"]
    )
    .round(4)
)


# ----------------------------------------------------------------------
# 6. Lead time in minutes
# ----------------------------------------------------------------------

event_warning_df["median_lead_minutes"] = (
    event_warning_df["median_lead_seconds"] / 60.0
)

event_warning_df["mean_lead_minutes"] = (
    event_warning_df["mean_lead_seconds"] / 60.0
)


# ----------------------------------------------------------------------
# 7. Sanity checks
# ----------------------------------------------------------------------

assert (
    event_warning_df["events_detected"]
    <= event_warning_df["events_evaluated"]
).all()

assert (
    event_warning_df["events_detected"]
    + event_warning_df["events_missed"]
    == event_warning_df["events_evaluated"]
).all()

successful_leads = (
    event_warning_details_df.loc[
        event_warning_details_df["detected"] == 1,
        "lead_seconds"
    ]
    .dropna()
)

if len(successful_leads) > 0:

    assert (
        successful_leads > 0
    ).all()

    assert (
        successful_leads <= 600
    ).all()


print("\n" + "=" * 78)
print("✓ GRU-130 evaluated with 130 features")
print("✓ GRU-134 evaluated with 134 features")
print("✓ Existing Book 3 onset metadata used")
print("✓ Exact test split preserved")
print("✓ Fixed thresholds only")
print("✓ No threshold tuning")
print("✓ First-warning lead time calculated")
print("✓ Exact TTE prediction NOT claimed")
print("✓ CELL 21 COMPLETE")
print("=" * 78)

DRISHTI — EVENT-LEVEL EARLY WARNING & LEAD-TIME ANALYSIS

✓ Test sequences: 875

Model input dimensions:
  GRU-130: 130 features
  GRU-134: 134 features

30s: 873 valid test sequences
  Future-onset sequences: 17

60s: 871 valid test sequences
  Future-onset sequences: 26

150s: 864 valid test sequences
  Future-onset sequences: 35

300s: 858 valid test sequences
  Future-onset sequences: 40

600s: 841 valid test sequences
  Future-onset sequences: 33

EVENT-LEVEL EARLY WARNING RESULTS


,model,horizon,threshold,events_evaluated,events_detected,events_missed,event_detection_rate,median_lead_seconds,mean_lead_seconds,min_lead_seconds,max_lead_seconds
16,GRU-130,150s,0.3,15,15,0,1.0000,120.0,96.0000,30.0,150.0
17,GRU-130,150s,0.5,15,15,0,1.0000,120.0,96.0000,30.0,150.0
18,GRU-130,150s,0.7,15,15,0,1.0000,120.0,94.0000,30.0,150.0
19,GRU-130,150s,0.9,15,13,2,0.8667,90.0,87.6923,30.0,150.0
20,GRU-134,150s,0.3,15,15,0,1.0000,120.0,96.0000,30.0,150.0
21,GRU-134,150s,0.5,15,15,0,1.0000,120.0,96.0000,30.0,150.0
22,GRU-134,150s,0.7,15,14,1,0.9333,120.0,98.5714,30.0,150.0
23,GRU-134,150s,0.9,15,13,2,0.8667,120.0,90.0000,30.0,150.0
24,GRU-130,300s,0.3,14,13,1,0.9286,120.0,133.8462,30.0,300.0
25,GRU-130,300s,0.5,14,13,1,0.9286,120.0,133.8462,30.0,300.0



HIGH-CONFIDENCE EVENT WARNING — THRESHOLD = 0.90


,model,horizon,events_evaluated,events_detected,events_missed,event_detection_rate,median_lead_seconds,mean_lead_seconds,min_lead_seconds,max_lead_seconds
19,GRU-130,150s,15,13,2,0.8667,90.0,87.6923,30.0,150.0
23,GRU-134,150s,15,13,2,0.8667,120.0,90.0000,30.0,150.0
27,GRU-130,300s,14,9,5,0.6429,120.0,113.3333,30.0,270.0
31,GRU-134,300s,14,13,1,0.9286,120.0,126.9231,30.0,300.0
3,GRU-130,30s,17,0,17,0.0000,NaN,NaN,NaN,NaN
7,GRU-134,30s,17,0,17,0.0000,NaN,NaN,NaN,NaN
35,GRU-130,600s,12,1,11,0.0833,30.0,30.0000,30.0,30.0
39,GRU-134,600s,12,10,2,0.8333,120.0,186.0000,30.0,600.0
11,GRU-130,60s,17,4,13,0.2353,30.0,37.5000,30.0,60.0
15,GRU-134,60s,17,3,14,0.1765,30.0,30.0000,30.0,30.0



✓ GRU-130 evaluated with 130 features
✓ GRU-134 evaluated with 134 features
✓ Existing Book 3 onset metadata used
✓ Exact test split preserved
✓ Fixed thresholds only
✓ No threshold tuning
✓ First-warning lead time calculated
✓ Exact TTE prediction NOT claimed
✓ CELL 21 COMPLETE


In [49]:
# ======================================================================
# DRISHTI — BOOK 3 — CELL 22
# FINAL FORECASTING EVIDENCE SUMMARY
# ======================================================================
#
# Purpose:
#   Consolidate the validated TON-IoT forecasting evidence into
#   presentation-ready tables.
#
# No model training.
# No threshold tuning.
# No new evaluation.
# ======================================================================

import numpy as np
import pandas as pd

print("=" * 82)
print("DRISHTI — BOOK 3 — FINAL FORECASTING EVIDENCE SUMMARY")
print("=" * 82)


# ----------------------------------------------------------------------
# 1. Static LR baseline
# ----------------------------------------------------------------------

# Cell 17 should have produced this table.
# Search common variable names without assuming one exact name.

lr_candidates = [
    "lr_results_df",
    "static_lr_results_df",
    "baseline_results_df",
    "static_results_df",
    "lr_results"
]

lr_name = next(
    (x for x in lr_candidates if x in globals()),
    None
)

if lr_name is not None:

    lr_df = globals()[lr_name].copy()

    print("\n" + "=" * 82)
    print("STATIC LOGISTIC REGRESSION BASELINE")
    print("=" * 82)

    display(lr_df)

else:
    print(
        "\n⚠ Static LR results table not found in memory."
        "\n  Cell 17 metrics remain recorded in the Book 3 results."
    )


# ----------------------------------------------------------------------
# 2. GRU forecasting results
# ----------------------------------------------------------------------

gru_result_candidates = [
    "gru_results_df",
    "gru_comparison_df",
    "temporal_gru_results_df",
    "gru_results"
]

gru_result_name = next(
    (x for x in gru_result_candidates if x in globals()),
    None
)

if gru_result_name is not None:

    gru_df = globals()[gru_result_name].copy()

    print("\n" + "=" * 82)
    print("TEMPORAL GRU FORECASTING RESULTS")
    print("=" * 82)

    display(gru_df)

else:

    print(
        "\n⚠ GRU results table not found under a standard variable name."
        "\n  Cell 19 metrics remain recorded in the Book 3 results."
    )


# ----------------------------------------------------------------------
# 3. Event-level warning summary
# ----------------------------------------------------------------------

required_event_cols = [
    "model",
    "horizon",
    "threshold",
    "events_evaluated",
    "events_detected",
    "events_missed",
    "event_detection_rate",
    "median_lead_seconds",
    "mean_lead_seconds"
]

if "event_warning_df" not in globals():

    raise RuntimeError(
        "event_warning_df not found. "
        "Run Cell 21 successfully first."
    )

missing_event_cols = [
    c for c in required_event_cols
    if c not in event_warning_df.columns
]

if missing_event_cols:

    raise RuntimeError(
        f"event_warning_df is missing columns: "
        f"{missing_event_cols}"
    )


# ----------------------------------------------------------------------
# 4. High-confidence event-level operating points
# ----------------------------------------------------------------------

high_conf = event_warning_df[
    event_warning_df["threshold"] == 0.90
].copy()

high_conf = high_conf[
    [
        "model",
        "horizon",
        "events_evaluated",
        "events_detected",
        "events_missed",
        "event_detection_rate",
        "median_lead_seconds",
        "mean_lead_seconds"
    ]
].sort_values(
    ["horizon", "model"]
)

print("\n" + "=" * 82)
print("EVENT-LEVEL EARLY WARNING — FIXED THRESHOLD 0.90")
print("=" * 82)

display(
    high_conf.round(4)
)


# ----------------------------------------------------------------------
# 5. Main operational configuration
# ----------------------------------------------------------------------
#
# This is NOT automatically selected by optimization.
# We explicitly report the 300s GRU-134 configuration because it is
# the configuration identified in the completed evaluation as the
# strongest high-confidence operational result.
# ----------------------------------------------------------------------

headline_mask = (
    (event_warning_df["model"] == "GRU-134")
    & (event_warning_df["horizon"] == "300s")
    & (event_warning_df["threshold"] == 0.90)
)

headline = event_warning_df[
    headline_mask
].copy()

print("\n" + "=" * 82)
print("HEADLINE OPERATIONAL RESULT")
print("=" * 82)

display(
    headline.round(4)
)


# ----------------------------------------------------------------------
# 6. Supporting 150s result
# ----------------------------------------------------------------------

supporting_150 = event_warning_df[
    (event_warning_df["horizon"] == "150s")
    & (event_warning_df["threshold"] == 0.90)
].copy()

print("\n" + "=" * 82)
print("SUPPORTING 150s HIGH-CONFIDENCE RESULT")
print("=" * 82)

display(
    supporting_150.round(4)
)


# ----------------------------------------------------------------------
# 7. Build presentation-ready headline dictionary
# ----------------------------------------------------------------------

if not headline.empty:

    h = headline.iloc[0]

    headline_summary = {
        "dataset": "TON-IoT",
        "model": str(h["model"]),
        "forecast_horizon": str(h["horizon"]),
        "alert_threshold": float(h["threshold"]),
        "events_evaluated": int(h["events_evaluated"]),
        "events_detected": int(h["events_detected"]),
        "events_missed": int(h["events_missed"]),
        "event_detection_rate_percent": (
            float(h["event_detection_rate"]) * 100
        ),
        "median_warning_lead_seconds": float(
            h["median_lead_seconds"]
        ),
        "mean_warning_lead_seconds": float(
            h["mean_lead_seconds"]
        )
    }

    print("\n" + "=" * 82)
    print("PRESENTATION-READY HEADLINE")
    print("=" * 82)

    for k, v in headline_summary.items():
        print(f"{k}: {v}")


# ----------------------------------------------------------------------
# 8. Explicit scientific interpretation
# ----------------------------------------------------------------------

print("\n" + "=" * 82)
print("INTERPRETATION STATUS")
print("=" * 82)

print("""
PROMOTED:
  • Future attack-onset forecasting formulation on TON-IoT
  • Event-level early-warning evaluation
  • Fixed-threshold operational evaluation
  • Warning lead-time measurement

SUPPORTING:
  • Temporal GRU comparison
  • 150s–600s horizon analysis
  • GRU-134 enriched temporal representation

NOT CLAIMED:
  • Exact continuous TTE prediction
  • Universal attack prediction
  • Zero-shot cross-dataset transfer
  • Perfect performance outside the evaluated datasets/events
""")


# ----------------------------------------------------------------------
# 9. Final audit
# ----------------------------------------------------------------------

assert not event_warning_df.empty
assert not high_conf.empty

assert (
    event_warning_df["events_detected"]
    <= event_warning_df["events_evaluated"]
).all()

assert (
    event_warning_df["events_detected"]
    + event_warning_df["events_missed"]
    == event_warning_df["events_evaluated"]
).all()

print("=" * 82)
print("✓ Book 3 forecasting evidence consolidated")
print("✓ Event-level warning evidence included")
print("✓ No new model fitting")
print("✓ No threshold tuning")
print("✓ Exact TTE remains explicitly unclaimed")
print("✓ CELL 22 COMPLETE")
print("=" * 82)

DRISHTI — BOOK 3 — FINAL FORECASTING EVIDENCE SUMMARY

STATIC LOGISTIC REGRESSION BASELINE


,horizon,train_positives,val_positives,test_positives,val_pr_auc,test_pr_auc,val_roc_auc,test_roc_auc,val_base_pr_auc,test_base_pr_auc,val_pr_lift,test_pr_lift
0,30s,48,6,17,0.322192,0.418290,0.789939,0.941589,0.006865,0.019473,46.932640,21.480416
1,60s,64,9,26,0.336406,0.640451,0.983025,0.951434,0.010309,0.029851,32.631355,21.455106
2,150s,88,14,35,0.521479,0.907725,0.991489,0.983422,0.016092,0.040509,32.406167,22.407837
3,300s,98,17,40,0.690374,1.000000,0.991953,1.000000,0.019653,0.046620,35.127848,21.450000
4,600s,94,21,33,0.923430,1.000000,0.951639,1.000000,0.024561,0.039239,37.596774,25.484848



TEMPORAL GRU FORECASTING RESULTS


,model,horizon,train_n,val_n,test_n,train_pos,val_pos,test_pos,val_pr_auc,test_pr_auc,val_roc_auc,test_roc_auc,best_epoch
0,GRU-130,30s,4068,874,873,48,6,17,0.407492,0.388342,0.989439,0.980690,18
1,GRU-134,30s,4068,874,873,48,6,17,0.562500,0.447679,0.993472,0.982752,14
2,GRU-130,60s,4064,873,871,64,9,26,0.802632,0.712468,0.996528,0.989850,22
3,GRU-134,60s,4064,873,871,64,9,26,0.324597,0.672451,0.989326,0.987984,15
4,GRU-130,150s,4053,870,864,88,14,35,0.702525,0.887382,0.994826,0.996347,7
5,GRU-134,150s,4053,870,864,88,14,35,0.651545,0.779661,0.994075,0.994348,6
6,GRU-130,300s,4048,865,858,98,17,40,0.896849,1.000000,0.997919,1.000000,8
7,GRU-134,300s,4048,865,858,98,17,40,0.807756,1.000000,0.997017,1.000000,7
8,GRU-130,600s,4034,855,841,94,21,33,1.000000,1.000000,1.000000,1.000000,8
9,GRU-134,600s,4034,855,841,94,21,33,1.000000,1.000000,1.000000,1.000000,12



EVENT-LEVEL EARLY WARNING — FIXED THRESHOLD 0.90


...


HEADLINE OPERATIONAL RESULT


,model,horizon,threshold,events_evaluated,events_detected,events_missed,event_detection_rate,median_lead_seconds,mean_lead_seconds,min_lead_seconds,max_lead_seconds,median_lead_minutes,mean_lead_minutes
31,GRU-134,300s,0.9,14,13,1,0.9286,120.0,126.9231,30.0,300.0,2.0,2.1154



SUPPORTING 150s HIGH-CONFIDENCE RESULT


,model,horizon,threshold,events_evaluated,events_detected,events_missed,event_detection_rate,median_lead_seconds,mean_lead_seconds,min_lead_seconds,max_lead_seconds,median_lead_minutes,mean_lead_minutes
19,GRU-130,150s,0.9,15,13,2,0.8667,90.0,87.6923,30.0,150.0,1.5,1.4615
23,GRU-134,150s,0.9,15,13,2,0.8667,120.0,90.0000,30.0,150.0,2.0,1.5000



PRESENTATION-READY HEADLINE
dataset: TON-IoT
model: GRU-134
forecast_horizon: 300s
alert_threshold: 0.9
events_evaluated: 14
events_detected: 13
events_missed: 1
event_detection_rate_percent: 92.85714285714286
median_warning_lead_seconds: 120.0
mean_warning_lead_seconds: 126.92307692307692

INTERPRETATION STATUS

PROMOTED:
  • Future attack-onset forecasting formulation on TON-IoT
  • Event-level early-warning evaluation
  • Fixed-threshold operational evaluation
  • Warning lead-time measurement

SUPPORTING:
  • Temporal GRU comparison
  • 150s–600s horizon analysis
  • GRU-134 enriched temporal representation

NOT CLAIMED:
  • Exact continuous TTE prediction
  • Universal attack prediction
  • Zero-shot cross-dataset transfer
  • Perfect performance outside the evaluated datasets/events

✓ Book 3 forecasting evidence consolidated
✓ Event-level warning evidence included
✓ No new model fitting
✓ No threshold tuning
✓ Exact TTE remains explicitly unclaimed
✓ CELL 22 COMPLETE


In [50]:
# ============================================================
# DRISHTI — BOOK 3 — CELL 23
# DISCRETE TTE SUPPORT + EVENT-LEVEL AUDIT
# ============================================================
#
# Purpose:
#   Audit whether TON-IoT has enough independent event-level
#   support to attempt exact/discrete TTE prediction.
#
# TTE definition:
#   Number of future 30s windows until the genuine attack onset.
#
# IMPORTANT:
#   - No model training
#   - No feature fitting
#   - No label-derived predictors
#   - Same chronological train/validation/test split
#   - Repeated source windows leading to the same onset are NOT
#     counted as independent events.
# ============================================================

import numpy as np
import pandas as pd

TTE_HORIZONS = ["30s", "60s", "150s", "300s", "600s"]

print("=" * 90)
print("DRISHTI — DISCRETE TTE SUPPORT AUDIT")
print("=" * 90)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_objects = [
    "forecast_sequences_df",
    "forecast_sequences_split_df",
    "strict_train_mask",
    "strict_val_mask",
    "strict_test_mask",
]

missing = [x for x in required_objects if x not in globals()]

if missing:
    raise RuntimeError(f"Missing required objects: {missing}")

print("Required objects: PASS")

# ------------------------------------------------------------
# 2. Verify split consistency
# ------------------------------------------------------------

split_counts = (
    forecast_sequences_split_df["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .fillna(0)
    .astype(int)
)

print("\nChronological sequence split:")
print(split_counts.to_string())

assert len(forecast_sequences_split_df) == len(forecast_sequences_df)

# ------------------------------------------------------------
# 3. Helper
# ------------------------------------------------------------

def audit_tte_horizon(horizon):
    target_col = f"Y_{horizon}_target"
    tte_col = f"Y_{horizon}_tte_windows"
    onset_col = f"Y_{horizon}_onset_time"

    df = forecast_sequences_split_df.copy()

    # Positive future-onset samples only
    pos = df[
        (df[target_col] == 1) &
        np.isfinite(pd.to_numeric(df[tte_col], errors="coerce"))
    ].copy()

    pos[tte_col] = pd.to_numeric(pos[tte_col], errors="coerce")
    pos["tte_class"] = pos[tte_col].astype(int)

    # --------------------------------------------------------
    # Sequence-level support
    # --------------------------------------------------------

    seq_counts = (
        pos["tte_class"]
        .value_counts()
        .sort_index()
    )

    expected_classes = range(
        1,
        int(horizon.rstrip("s")) // 30 + 1
    )

    seq_counts = (
        seq_counts
        .reindex(expected_classes, fill_value=0)
        .astype(int)
    )

    # --------------------------------------------------------
    # Event-level support
    #
    # Multiple source windows can point to the same onset.
    # Count each (file_id, onset_time) only once.
    # --------------------------------------------------------

    event_cols = ["file_id", onset_col]

    events = (
        pos[event_cols + ["split", tte_col]]
        .drop_duplicates(subset=event_cols)
        .copy()
    )

    events["tte_class"] = (
        pd.to_numeric(events[tte_col], errors="coerce")
        .astype(int)
    )

    event_counts = (
        events["tte_class"]
        .value_counts()
        .sort_index()
        .reindex(expected_classes, fill_value=0)
        .astype(int)
    )

    # --------------------------------------------------------
    # Split-level event support
    # --------------------------------------------------------

    split_event_counts = {}

    for split_name in ["train", "validation", "test"]:
        sub = events[events["split"] == split_name]

        counts = (
            sub["tte_class"]
            .value_counts()
            .sort_index()
            .reindex(expected_classes, fill_value=0)
            .astype(int)
        )

        split_event_counts[split_name] = counts

    split_event_table = pd.DataFrame(split_event_counts)

    # --------------------------------------------------------
    # Majority-class baseline
    # --------------------------------------------------------

    total_events = len(events)

    if total_events > 0:
        majority_class = int(event_counts.idxmax())
        majority_count = int(event_counts.max())
        majority_accuracy = majority_count / total_events
    else:
        majority_class = None
        majority_accuracy = np.nan

    # --------------------------------------------------------
    # Event-level TTE statistics
    # --------------------------------------------------------

    tte_values = events["tte_class"].to_numpy()

    if len(tte_values):
        stats = {
            "events": len(tte_values),
            "min": int(np.min(tte_values)),
            "median": float(np.median(tte_values)),
            "mean": float(np.mean(tte_values)),
            "max": int(np.max(tte_values)),
            "std": float(np.std(tte_values)),
            "majority_class": majority_class,
            "majority_accuracy": float(majority_accuracy),
        }
    else:
        stats = {
            "events": 0,
            "min": np.nan,
            "median": np.nan,
            "mean": np.nan,
            "max": np.nan,
            "std": np.nan,
            "majority_class": None,
            "majority_accuracy": np.nan,
        }

    return {
        "target_col": target_col,
        "tte_col": tte_col,
        "onset_col": onset_col,
        "positive_sequences": len(pos),
        "unique_events": len(events),
        "seq_counts": seq_counts,
        "event_counts": event_counts,
        "split_event_table": split_event_table,
        "stats": stats,
        "events_df": events,
    }


# ------------------------------------------------------------
# 4. Run audit
# ------------------------------------------------------------

tte_audit = {}

for horizon in TTE_HORIZONS:
    tte_audit[horizon] = audit_tte_horizon(horizon)

# ------------------------------------------------------------
# 5. Summary table
# ------------------------------------------------------------

summary_rows = []

for horizon, result in tte_audit.items():
    s = result["stats"]

    summary_rows.append({
        "horizon": horizon,
        "positive_sequences": result["positive_sequences"],
        "unique_onset_events": result["unique_events"],
        "min_TTE_windows": s["min"],
        "median_TTE_windows": s["median"],
        "mean_TTE_windows": s["mean"],
        "max_TTE_windows": s["max"],
        "std_TTE_windows": s["std"],
        "majority_TTE_class": s["majority_class"],
        "majority_baseline_accuracy": s["majority_accuracy"],
    })

tte_summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 90)
print("TTE SUMMARY")
print("=" * 90)

display(tte_summary_df.round(4))

# ------------------------------------------------------------
# 6. Class-support tables
# ------------------------------------------------------------

for horizon, result in tte_audit.items():

    print("\n" + "-" * 90)
    print(f"{horizon} — SEQUENCE-LEVEL TTE SUPPORT")
    print("-" * 90)

    seq_table = result["seq_counts"].rename("sequence_count").to_frame()

    seq_table["sequence_pct"] = (
        100 * seq_table["sequence_count"]
        / max(seq_table["sequence_count"].sum(), 1)
    )

    display(seq_table.round(2))

    print(f"{horizon} — INDEPENDENT EVENT-LEVEL TTE SUPPORT")

    event_table = result["event_counts"].rename("event_count").to_frame()

    event_table["event_pct"] = (
        100 * event_table["event_count"]
        / max(event_table["event_count"].sum(), 1)
    )

    display(event_table.round(2))

# ------------------------------------------------------------
# 7. Train / validation / test event support
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("EVENT-LEVEL TTE SUPPORT BY SPLIT")
print("=" * 90)

for horizon, result in tte_audit.items():

    print(f"\n{horizon}")

    table = result["split_event_table"].copy()

    table.index.name = "TTE_windows"

    display(table)

# ------------------------------------------------------------
# 8. Sparse-class diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("SPARSE-CLASS DIAGNOSTICS")
print("=" * 90)

for horizon, result in tte_audit.items():

    event_counts = result["event_counts"]

    supported = event_counts[event_counts > 0]

    sparse = event_counts[
        (event_counts > 0) &
        (event_counts < 5)
    ]

    print(f"\n{horizon}:")
    print(f"  Unique onset events       : {result['unique_events']}")
    print(f"  Supported TTE classes    : {len(supported)}")
    print(f"  Classes with <5 events   : {list(sparse.index)}")

    if len(supported):
        print(
            f"  Largest class            : "
            f"{int(supported.idxmax())} windows "
            f"({int(supported.max())} events)"
        )

# ------------------------------------------------------------
# 9. Final audit verdict
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CELL 23 — AUDIT VERDICT")
print("=" * 90)

print("""
This cell does NOT train a TTE model.

The audit determines whether discrete TTE prediction is
statistically supportable before any model is fitted.

TTE classes are measured in 30-second windows:
  1 window  = 30s
  2 windows = 60s
  ...
  20 windows = 600s

Independent onset events, rather than repeated source windows,
are the primary support count.
""")

for horizon, result in tte_audit.items():

    events = result["unique_events"]
    supported = int((result["event_counts"] > 0).sum())
    sparse = int(
        (
            (result["event_counts"] > 0) &
            (result["event_counts"] < 5)
        ).sum()
    )

    print(
        f"{horizon}: "
        f"{events} independent events | "
        f"{supported} supported TTE classes | "
        f"{sparse} sparse classes"
    )

print("\nCELL 23 COMPLETE.")

DRISHTI — DISCRETE TTE SUPPORT AUDIT
Required objects: PASS

Chronological sequence split:
split
train         4073
validation     875
test           875

TTE SUMMARY


,horizon,positive_sequences,unique_onset_events,min_TTE_windows,median_TTE_windows,mean_TTE_windows,max_TTE_windows,std_TTE_windows,majority_TTE_class,majority_baseline_accuracy
0,30s,71,71,1,1.0,1.0000,1,0.0000,1,1.0000
1,60s,99,71,1,1.0,1.3944,2,0.4887,1,0.6056
2,150s,137,68,1,1.0,2.0147,5,1.4901,1,0.6029
3,300s,155,64,1,1.0,2.4219,10,2.3506,1,0.5781
4,600s,148,56,1,1.0,2.7500,20,3.4343,1,0.5893



------------------------------------------------------------------------------------------
30s — SEQUENCE-LEVEL TTE SUPPORT
------------------------------------------------------------------------------------------


,sequence_count,sequence_pct
tte_class,,
1,71,100.0


30s — INDEPENDENT EVENT-LEVEL TTE SUPPORT


...


300s


,train,validation,test
TTE_windows,,,
1,28,3,6
2,6,1,2
3,3,1,0
4,1,0,3
5,3,0,0
6,1,0,1
7,0,0,0
8,1,1,0
9,1,0,0



600s


,train,validation,test
TTE_windows,,,
1,25,3,5
2,4,1,1
3,3,1,0
4,1,0,3
5,2,0,0
6,1,0,1
7,0,0,0
8,1,1,0
9,1,0,0



SPARSE-CLASS DIAGNOSTICS

30s:
  Unique onset events       : 71
  Supported TTE classes    : 1
  Classes with <5 events   : []
  Largest class            : 1 windows (71 events)

60s:
  Unique onset events       : 71
  Supported TTE classes    : 2
  Classes with <5 events   : []
  Largest class            : 1 windows (43 events)

150s:
  Unique onset events       : 68
  Supported TTE classes    : 5
  Classes with <5 events   : [3, 4]
  Largest class            : 1 windows (41 events)

300s:
  Unique onset events       : 64
  Supported TTE classes    : 9
  Classes with <5 events   : [3, 4, 5, 6, 8, 9, 10]
  Largest class            : 1 windows (37 events)

600s:
  Unique onset events       : 56
  Supported TTE classes    : 10
  Classes with <5 events   : [3, 4, 5, 6, 8, 9, 14, 20]
  Largest class            : 1 windows (33 events)

CELL 23 — AUDIT VERDICT

This cell does NOT train a TTE model.

The audit determines whether discrete TTE prediction is
statistically supportable before any

In [51]:
# ============================================================
# DRISHTI — BOOK 3 — CELL 24
# EVENT-AWARE DISCRETE TTE DATASET CONSTRUCTION
# ============================================================
#
# Goal:
#   Build a leakage-controlled event-level dataset for discrete
#   TTE prediction.
#
# TTE classes:
#   60s  -> {1, 2} windows = {30s, 60s}
#   150s -> {1, 2, 3, 4, 5} windows
#
# IMPORTANT:
#   - One independent attack onset = one event
#   - No duplicate source windows per onset
#   - Chronological split is preserved
#   - No labels are included in X
#   - Existing strict 10 x 134 feature representation is used
#   - No model training in this cell
# ============================================================

import numpy as np
import pandas as pd

TTE_EXPERIMENT_HORIZONS = ["60s", "150s"]

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

required_objects = [
    "forecast_sequences_df",
    "forecast_sequences_split_df",
    "X_strict_raw_all",
    "strict_train_mask",
    "strict_val_mask",
    "strict_test_mask",
]

missing = [x for x in required_objects if x not in globals()]

if missing:
    raise RuntimeError(f"Missing required objects: {missing}")

print("=" * 90)
print("DRISHTI — EVENT-AWARE DISCRETE TTE DATASET")
print("=" * 90)

print("Required objects: PASS")

# ------------------------------------------------------------
# 2. Verify alignment
# ------------------------------------------------------------

assert len(forecast_sequences_df) == len(X_strict_raw_all)
assert len(forecast_sequences_split_df) == len(X_strict_raw_all)

print(f"Sequence rows : {len(forecast_sequences_df)}")
print(f"Feature rows  : {len(X_strict_raw_all)}")
print(f"Alignment     : PASS")

# ------------------------------------------------------------
# 3. Construct event-aware datasets
# ------------------------------------------------------------

tte_event_datasets = {}

for horizon in TTE_EXPERIMENT_HORIZONS:

    print("\n" + "-" * 90)
    print(f"BUILDING EVENT DATASET — {horizon}")
    print("-" * 90)

    target_col = f"Y_{horizon}_target"
    tte_col = f"Y_{horizon}_tte_windows"
    onset_col = f"Y_{horizon}_onset_time"

    meta = forecast_sequences_split_df.copy()

    # Positive future-onset samples with valid TTE
    candidate_mask = (
        (meta[target_col] == 1) &
        np.isfinite(
            pd.to_numeric(meta[tte_col], errors="coerce")
        )
    )

    candidate_idx = np.flatnonzero(candidate_mask.to_numpy())

    candidate_meta = meta.iloc[candidate_idx].copy()

    candidate_meta[tte_col] = (
        pd.to_numeric(
            candidate_meta[tte_col],
            errors="coerce"
        )
        .astype(int)
    )

    # --------------------------------------------------------
    # One independent event = one (file_id, onset_time)
    #
    # Multiple source windows can point to the same onset.
    #
    # IMPORTANT:
    # We choose the earliest source window associated with the
    # event. This gives the earliest available forecasting
    # context for that attack onset.
    # --------------------------------------------------------

    candidate_meta["_original_index"] = candidate_meta.index

    candidate_meta = candidate_meta.sort_values(
        ["file_id", onset_col, "source_time"]
    )

    event_meta = (
        candidate_meta
        .drop_duplicates(
            subset=["file_id", onset_col],
            keep="first"
        )
        .copy()
    )

    # Restore chronological ordering
    event_meta = event_meta.sort_values(
        ["file_id", "source_time"]
    ).reset_index(drop=True)

    event_indices = event_meta["_original_index"].to_numpy()

    # --------------------------------------------------------
    # Extract corresponding strict sequence tensors
    # --------------------------------------------------------

    X_events = X_strict_raw_all[event_indices].copy()

    y_events = event_meta[tte_col].to_numpy(dtype=np.int64)

    # --------------------------------------------------------
    # Split labels
    # --------------------------------------------------------

    split_events = event_meta["split"].to_numpy()

    train_mask = split_events == "train"
    val_mask = split_events == "validation"
    test_mask = split_events == "test"

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    tte_event_datasets[horizon] = {
        "X": X_events,
        "y": y_events,
        "metadata": event_meta,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "event_indices": event_indices,
    }

    # --------------------------------------------------------
    # Audit
    # --------------------------------------------------------

    print(f"Candidate positive sequences : {len(candidate_meta)}")
    print(f"Unique onset events           : {len(event_meta)}")
    print(f"X shape                       : {X_events.shape}")
    print(f"y shape                       : {y_events.shape}")

    print("\nEvent split:")
    print(
        pd.Series(split_events)
        .value_counts()
        .reindex(["train", "validation", "test"])
        .fillna(0)
        .astype(int)
        .to_string()
    )

    print("\nTTE class distribution:")

    class_table = (
        pd.Series(y_events)
        .value_counts()
        .sort_index()
        .rename("events")
        .to_frame()
    )

    class_table["percentage"] = (
        100 * class_table["events"]
        / len(y_events)
    )

    display(class_table.round(2))

    # --------------------------------------------------------
    # Check no duplicate events remain
    # --------------------------------------------------------

    duplicate_events = event_meta.duplicated(
        subset=["file_id", onset_col]
    ).sum()

    assert duplicate_events == 0

    # --------------------------------------------------------
    # Check split consistency
    # --------------------------------------------------------

    assert train_mask.sum() + val_mask.sum() + test_mask.sum() == len(event_meta)

    # --------------------------------------------------------
    # Check finite values
    # --------------------------------------------------------

    assert np.isfinite(X_events).all()
    assert np.isfinite(y_events).all()

    print("\nChecks:")
    print(f"  Duplicate onset events : {duplicate_events}")
    print(f"  All X finite           : {np.isfinite(X_events).all()}")
    print(f"  All y finite           : {np.isfinite(y_events).all()}")
    print("  Event construction     : PASS")

# ------------------------------------------------------------
# 4. Explicit train/validation/test class support
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("EVENT-LEVEL TTE CLASS SUPPORT BY SPLIT")
print("=" * 90)

for horizon, data in tte_event_datasets.items():

    print(f"\n{horizon}")

    rows = []

    for split_name, mask in [
        ("train", data["train_mask"]),
        ("validation", data["val_mask"]),
        ("test", data["test_mask"]),
    ]:

        counts = (
            pd.Series(data["y"][mask])
            .value_counts()
            .sort_index()
        )

        for tte_class, count in counts.items():

            rows.append({
                "split": split_name,
                "TTE_windows": int(tte_class),
                "TTE_seconds": int(tte_class * 30),
                "events": int(count),
            })

    support_df = pd.DataFrame(rows)

    if len(support_df):
        display(
            support_df
            .sort_values(["TTE_windows", "split"])
            .reset_index(drop=True)
        )

# ------------------------------------------------------------
# 5. Verify event uniqueness globally
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("GLOBAL EVENT UNIQUENESS CHECK")
print("=" * 90)

for horizon, data in tte_event_datasets.items():

    meta = data["metadata"]
    onset_col = f"Y_{horizon}_onset_time"

    unique_events = (
        meta[["file_id", onset_col]]
        .drop_duplicates()
        .shape[0]
    )

    print(
        f"{horizon}: "
        f"{len(meta)} rows / "
        f"{unique_events} unique onset events"
    )

    assert len(meta) == unique_events

print("\n" + "=" * 90)
print("CELL 24 COMPLETE")
print("=" * 90)

print("""
Event-aware TTE datasets are now prepared.

Next step:
    train discrete TTE models on these event-level datasets.

Initial experiments:
    60s  -> 2 TTE classes
    150s -> 5 TTE classes

We will evaluate:
    - exact TTE accuracy
    - ±1 window accuracy (±30s)
    - ±2 window accuracy (±60s)
    - MAE in windows
    - MAE in seconds
    - median absolute error
    - TTE bias
    - majority-class baseline

No model has been trained in this cell.
""")

DRISHTI — EVENT-AWARE DISCRETE TTE DATASET
Required objects: PASS
Sequence rows : 5823
Feature rows  : 5823
Alignment     : PASS

------------------------------------------------------------------------------------------
BUILDING EVENT DATASET — 60s
------------------------------------------------------------------------------------------
Candidate positive sequences : 99
Unique onset events           : 71
X shape                       : (71, 10, 134)
y shape                       : (71,)

Event split:
train         48
validation     6
test          17

TTE class distribution:


,events,percentage
1,43,60.56
2,28,39.44



Checks:
  Duplicate onset events : 0
  All X finite           : True
  All y finite           : True
  Event construction     : PASS

------------------------------------------------------------------------------------------
BUILDING EVENT DATASET — 150s
------------------------------------------------------------------------------------------
Candidate positive sequences : 137
Unique onset events           : 68
X shape                       : (68, 10, 134)
y shape                       : (68,)

Event split:
train         48
validation     6
test          14

TTE class distribution:


,events,percentage
1,41,60.29
2,9,13.24
3,4,5.88
4,4,5.88
5,10,14.71



Checks:
  Duplicate onset events : 0
  All X finite           : True
  All y finite           : True
  Event construction     : PASS

EVENT-LEVEL TTE CLASS SUPPORT BY SPLIT

60s


,split,TTE_windows,TTE_seconds,events
0,test,1,30,8
1,train,1,30,32
2,validation,1,30,3
3,test,2,60,9
4,train,2,60,16
5,validation,2,60,3



150s


,split,TTE_windows,TTE_seconds,events
0,test,1,30,7
1,train,1,30,31
2,validation,1,30,3
3,test,2,60,2
4,train,2,60,6
5,validation,2,60,1
6,train,3,90,3
7,validation,3,90,1
8,test,4,120,3
9,train,4,120,1



GLOBAL EVENT UNIQUENESS CHECK
60s: 71 rows / 71 unique onset events
150s: 68 rows / 68 unique onset events

CELL 24 COMPLETE

Event-aware TTE datasets are now prepared.

Next step:
    train discrete TTE models on these event-level datasets.

Initial experiments:
    60s  -> 2 TTE classes
    150s -> 5 TTE classes

We will evaluate:
    - exact TTE accuracy
    - ±1 window accuracy (±30s)
    - ±2 window accuracy (±60s)
    - MAE in windows
    - MAE in seconds
    - median absolute error
    - TTE bias
    - majority-class baseline

No model has been trained in this cell.



In [52]:
# ============================================================
# DRISHTI — BOOK 3 — CELL 25
# DISCRETE TTE MODEL TRAINING
# ============================================================
#
# Experiments:
#   1. Static multinomial Logistic Regression
#   2. Temporal GRU
#
# Horizons:
#   60s  -> 2 TTE classes
#   150s -> 5 TTE classes
#
# TTE classes:
#   1 = 30s
#   2 = 60s
#   ...
#
# Evaluation is performed ONLY on independent test events.
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    mean_absolute_error,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 90)
print("DRISHTI — DISCRETE TTE MODEL TRAINING")
print("=" * 90)

# ------------------------------------------------------------
# Helper: metrics
# ------------------------------------------------------------

def calculate_tte_metrics(y_true, y_pred):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    error_windows = y_pred - y_true
    abs_error = np.abs(error_windows)

    return {
        "exact_accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "within_1_window": np.mean(abs_error <= 1),
        "within_2_windows": np.mean(abs_error <= 2),
        "mae_windows": np.mean(abs_error),
        "mae_seconds": np.mean(abs_error) * 30.0,
        "median_abs_error_windows": np.median(abs_error),
        "median_abs_error_seconds": np.median(abs_error) * 30.0,
        "bias_windows": np.mean(error_windows),
        "bias_seconds": np.mean(error_windows) * 30.0,
    }


# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

tte_static_models = {}
tte_gru_models = {}

tte_results = []

# ------------------------------------------------------------
# Train each horizon
# ------------------------------------------------------------

for horizon in TTE_EXPERIMENT_HORIZONS:

    print("\n" + "=" * 90)
    print(f"TTE HORIZON: {horizon}")
    print("=" * 90)

    data = tte_event_datasets[horizon]

    X = data["X"]
    y = data["y"]

    train_mask = data["train_mask"]
    val_mask = data["val_mask"]
    test_mask = data["test_mask"]

    X_train = X[train_mask]
    X_val = X[val_mask]
    X_test = X[test_mask]

    y_train = y[train_mask]
    y_val = y[val_mask]
    y_test = y[test_mask]

    print(
        f"Train: {len(y_train)} | "
        f"Validation: {len(y_val)} | "
        f"Test: {len(y_test)}"
    )

    print(
        f"Classes: {sorted(np.unique(y_train).tolist())}"
    )

    # --------------------------------------------------------
    # Train-only scaling
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_flat = X_train.reshape(
        X_train.shape[0], -1
    )

    X_val_flat = X_val.reshape(
        X_val.shape[0], -1
    )

    X_test_flat = X_test.reshape(
        X_test.shape[0], -1
    )

    # ========================================================
    # MODEL 1 — STATIC LOGISTIC REGRESSION
    # ========================================================
    #
    # Only the current/source 30s window is used.
    # This gives us a clean non-temporal baseline.
    # ========================================================

    X_train_static = X_train[:, -1, :]
    X_val_static = X_val[:, -1, :]
    X_test_static = X_test[:, -1, :]

    static_scaler = StandardScaler()

    X_train_static = static_scaler.fit_transform(
        X_train_static
    )

    X_val_static = static_scaler.transform(
        X_val_static
    )

    X_test_static = static_scaler.transform(
        X_test_static
    )

    classes = np.unique(y_train)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    class_weight_dict = dict(
        zip(classes, class_weights)
    )

    static_model = LogisticRegression(
        max_iter=3000,
        multi_class="auto",
        class_weight=class_weight_dict,
        random_state=SEED
    )

    static_model.fit(
        X_train_static,
        y_train
    )

    static_pred = static_model.predict(
        X_test_static
    )

    static_metrics = calculate_tte_metrics(
        y_test,
        static_pred
    )

    tte_static_models[horizon] = {
        "model": static_model,
        "scaler": static_scaler,
        "y_test": y_test.copy(),
        "pred_test": static_pred.copy()
    }

    # ========================================================
    # MODEL 2 — TEMPORAL GRU
    # ========================================================

    n_classes = len(classes)

    # Convert classes 1..N -> indices 0..N-1
    class_to_index = {
        int(c): i
        for i, c in enumerate(classes)
    }

    y_train_idx = np.array([
        class_to_index[int(v)]
        for v in y_train
    ])

    y_val_idx = np.array([
        class_to_index[int(v)]
        for v in y_val
    ])

    y_test_idx = np.array([
        class_to_index[int(v)]
        for v in y_test
    ])

    # --------------------------------------------------------
    # Train-only feature scaling for temporal model
    # --------------------------------------------------------

    temporal_scaler = StandardScaler()

    X_train_temporal_flat = X_train.reshape(
        X_train.shape[0], -1
    )

    X_val_temporal_flat = X_val.reshape(
        X_val.shape[0], -1
    )

    X_test_temporal_flat = X_test.reshape(
        X_test.shape[0], -1
    )

    temporal_scaler.fit(
        X_train_temporal_flat
    )

    X_train_temporal = temporal_scaler.transform(
        X_train_temporal_flat
    ).reshape(X_train.shape)

    X_val_temporal = temporal_scaler.transform(
        X_val_temporal_flat
    ).reshape(X_val.shape)

    X_test_temporal = temporal_scaler.transform(
        X_test_temporal_flat
    ).reshape(X_test.shape)

    # --------------------------------------------------------
    # GRU
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    gru = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(
                X_train_temporal.shape[1],
                X_train_temporal.shape[2]
            )
        ),

        tf.keras.layers.GRU(
            32,
            return_sequences=False
        ),

        tf.keras.layers.Dropout(0.20),

        tf.keras.layers.Dense(
            16,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.10),

        tf.keras.layers.Dense(
            n_classes,
            activation="softmax"
        )
    ])

    gru.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # --------------------------------------------------------
    # GRU class weights
    # --------------------------------------------------------

    gru_class_weights = {
        class_to_index[int(c)]: float(
            class_weight_dict[int(c)]
        )
        for c in classes
    }

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        mode="max",
        restore_best_weights=True
    )

    history = gru.fit(
        X_train_temporal,
        y_train_idx,
        validation_data=(
            X_val_temporal,
            y_val_idx
        ),
        epochs=100,
        batch_size=8,
        shuffle=False,
        class_weight=gru_class_weights,
        callbacks=[early_stop],
        verbose=0
    )

    gru_prob = gru.predict(
        X_test_temporal,
        verbose=0
    )

    gru_pred_idx = np.argmax(
        gru_prob,
        axis=1
    )

    index_to_class = {
        i: int(c)
        for i, c in enumerate(classes)
    }

    gru_pred = np.array([
        index_to_class[int(i)]
        for i in gru_pred_idx
    ])

    gru_metrics = calculate_tte_metrics(
        y_test,
        gru_pred
    )

    tte_gru_models[horizon] = {
        "model": gru,
        "scaler": temporal_scaler,
        "classes": classes.copy(),
        "y_test": y_test.copy(),
        "pred_test": gru_pred.copy(),
        "prob_test": gru_prob.copy(),
        "history": history.history
    }

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    for model_name, metrics in [
        ("Static LR", static_metrics),
        ("Temporal GRU", gru_metrics)
    ]:

        row = {
            "horizon": horizon,
            "model": model_name,
            **metrics
        }

        tte_results.append(row)

    # --------------------------------------------------------
    # Print immediate result
    # --------------------------------------------------------

    print("\nStatic LR:")
    for k, v in static_metrics.items():
        print(f"  {k:28s}: {v:.4f}")

    print("\nTemporal GRU:")
    for k, v in gru_metrics.items():
        print(f"  {k:28s}: {v:.4f}")

# ------------------------------------------------------------
# Final results table
# ------------------------------------------------------------

tte_results_df = pd.DataFrame(tte_results)

print("\n" + "=" * 90)
print("TTE TEST RESULTS")
print("=" * 90)

display(
    tte_results_df.round(4)
)

# ------------------------------------------------------------
# Confusion matrices
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TTE CONFUSION MATRICES")
print("=" * 90)

for horizon in TTE_EXPERIMENT_HORIZONS:

    print(f"\n{horizon} — Static LR")

    data = tte_static_models[horizon]

    classes = sorted(
        np.unique(data["y_test"])
    )

    cm = confusion_matrix(
        data["y_test"],
        data["pred_test"],
        labels=classes
    )

    display(
        pd.DataFrame(
            cm,
            index=[
                f"true_{c}"
                for c in classes
            ],
            columns=[
                f"pred_{c}"
                for c in classes
            ]
        )
    )

    print(f"{horizon} — Temporal GRU")

    data = tte_gru_models[horizon]

    classes = sorted(
        np.unique(data["y_test"])
    )

    cm = confusion_matrix(
        data["y_test"],
        data["pred_test"],
        labels=classes
    )

    display(
        pd.DataFrame(
            cm,
            index=[
                f"true_{c}"
                for c in classes
            ],
            columns=[
                f"pred_{c}"
                for c in classes
            ]
        )
    )

# ------------------------------------------------------------
# Final note
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CELL 25 COMPLETE")
print("=" * 90)

print("""
TTE models trained:

    Static LR
        -> current 30s state only

    Temporal GRU
        -> full 10 x 134 feature history

Evaluation:
    -> independent onset events only
    -> chronological train/validation/test split
    -> train-only scaling
    -> exact TTE class accuracy
    -> ±30s / ±60s tolerance
    -> MAE
    -> median absolute error
    -> signed TTE bias

IMPORTANT:
    These are discrete TTE estimates in 30-second windows.
    They are NOT continuous second-level TTE predictions.
""")

DRISHTI — DISCRETE TTE MODEL TRAINING

TTE HORIZON: 60s
Train: 48 | Validation: 6 | Test: 17
Classes: [1, 2]

Static LR:
  exact_accuracy              : 0.4706
  balanced_accuracy           : 0.4931
  within_1_window             : 1.0000
  within_2_windows            : 1.0000
  mae_windows                 : 0.5294
  mae_seconds                 : 15.8824
  median_abs_error_windows    : 1.0000
  median_abs_error_seconds    : 30.0000
  bias_windows                : -0.4118
  bias_seconds                : -12.3529

Temporal GRU:
  exact_accuracy              : 0.3529
  balanced_accuracy           : 0.3472
  within_1_window             : 1.0000
  within_2_windows            : 1.0000
  mae_windows                 : 0.6471
  mae_seconds                 : 19.4118
  median_abs_error_windows    : 1.0000
  median_abs_error_seconds    : 30.0000
  bias_windows                : 0.0588
  bias_seconds                : 1.7647

TTE HORIZON: 150s
Train: 48 | Validation: 6 | Test: 14
Classes: [1, 2, 3, 4,

,horizon,model,exact_accuracy,balanced_accuracy,within_1_window,within_2_windows,mae_windows,mae_seconds,median_abs_error_windows,median_abs_error_seconds,bias_windows,bias_seconds
0,60s,Static LR,0.4706,0.4931,1.0000,1.0000,0.5294,15.8824,1.0,30.0,-0.4118,-12.3529
1,60s,Temporal GRU,0.3529,0.3472,1.0000,1.0000,0.6471,19.4118,1.0,30.0,0.0588,1.7647
2,150s,Static LR,0.2143,0.1071,0.5714,0.6429,1.5714,47.1429,1.0,30.0,-0.8571,-25.7143
3,150s,Temporal GRU,0.2143,0.3750,0.7143,0.9286,1.1429,34.2857,1.0,30.0,0.2857,8.5714



TTE CONFUSION MATRICES

60s — Static LR


,pred_1,pred_2
true_1,7,1
true_2,8,1


60s — Temporal GRU


...


150s — Static LR


,pred_1,pred_2,pred_4,pred_5
true_1,3,3,0,0
true_2,2,0,0,0
true_4,3,0,0,0
true_5,0,2,0,0


150s — Temporal GRU


,pred_1,pred_2,pred_4,pred_5
true_1,0,4,0,0
true_2,0,2,0,0
true_4,0,0,0,0
true_5,0,1,0,1



CELL 25 COMPLETE

TTE models trained:

    Static LR
        -> current 30s state only

    Temporal GRU
        -> full 10 x 134 feature history

Evaluation:
    -> independent onset events only
    -> chronological train/validation/test split
    -> train-only scaling
    -> exact TTE class accuracy
    -> ±30s / ±60s tolerance
    -> MAE
    -> median absolute error
    -> signed TTE bias

IMPORTANT:
    These are discrete TTE estimates in 30-second windows.
    They are NOT continuous second-level TTE predictions.



In [53]:
# ============================================================
# DRISHTI — BOOK 3 — CELL 26
# CALIBRATED TTE-RANGE / INTERVAL ESTIMATION
# ============================================================
#
# Goal:
#   Convert discrete TTE class probabilities into an interval
#   estimate such as:
#
#       "Estimated TTE: 60–120 seconds"
#
# Instead of claiming an exact TTE class.
#
# Calibration:
#   VALIDATION events only.
#
# Final evaluation:
#   TEST events only.
#
# No test information is used to choose interval width.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 90)
print("DRISHTI — CALIBRATED TTE-RANGE ESTIMATION")
print("=" * 90)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def make_probability_interval(
    probabilities,
    classes,
    coverage=0.80
):
    """
    Build a contiguous TTE interval around the highest-probability
    class until cumulative probability reaches the requested
    coverage.

    Returns:
        lower_class, upper_class
    """

    probabilities = np.asarray(probabilities, dtype=float)
    classes = np.asarray(classes, dtype=int)

    order = np.argsort(probabilities)[::-1]

    selected = []
    cumulative = 0.0

    for idx in order:
        selected.append(idx)
        cumulative += probabilities[idx]

        if cumulative >= coverage:
            break

    selected_classes = classes[selected]

    return (
        int(np.min(selected_classes)),
        int(np.max(selected_classes))
    )


def interval_metrics(
    y_true,
    lower,
    upper
):
    """
    Evaluate interval predictions.

    y_true, lower, upper are TTE classes.
    """

    y_true = np.asarray(y_true).astype(int)
    lower = np.asarray(lower).astype(int)
    upper = np.asarray(upper).astype(int)

    contained = (
        (y_true >= lower) &
        (y_true <= upper)
    )

    widths = upper - lower

    # Distance from true value to interval.
    # 0 if contained.
    distance = np.where(
        y_true < lower,
        lower - y_true,
        np.where(
            y_true > upper,
            y_true - upper,
            0
        )
    )

    return {
        "coverage": np.mean(contained),
        "mean_width_windows": np.mean(widths),
        "mean_width_seconds": np.mean(widths) * 30.0,
        "median_width_windows": np.median(widths),
        "median_width_seconds": np.median(widths) * 30.0,
        "mean_miss_distance_windows": np.mean(distance),
        "mean_miss_distance_seconds": np.mean(distance) * 30.0,
        "all_contained": bool(np.all(contained))
    }


# ------------------------------------------------------------
# Calibration procedure
# ------------------------------------------------------------
#
# We test several probability-mass levels on validation:
#   0.50, 0.60, 0.70, 0.80, 0.90
#
# Select the SMALLEST interval that achieves at least 80%
# empirical validation coverage.
#
# If none reaches 80%, use the highest tested level.
# ------------------------------------------------------------

COVERAGE_LEVELS = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

TARGET_COVERAGE = 0.80

tte_interval_results = []
tte_interval_predictions = {}
tte_interval_calibration = {}

for horizon in TTE_EXPERIMENT_HORIZONS:

    print("\n" + "=" * 90)
    print(f"HORIZON: {horizon}")
    print("=" * 90)

    data = tte_event_datasets[horizon]
    gru_data = tte_gru_models[horizon]

    X = data["X"]
    y = data["y"]

    train_mask = data["train_mask"]
    val_mask = data["val_mask"]
    test_mask = data["test_mask"]

    y_val = y[val_mask]
    y_test = y[test_mask]

    # --------------------------------------------------------
    # Use probabilities already generated by Cell 25
    #
    # IMPORTANT:
    # Cell 25 only stored test probabilities.
    #
    # We therefore regenerate VALIDATION probabilities from
    # the already-trained model using the train-only scaler.
    # --------------------------------------------------------

    gru_model = gru_data["model"]
    temporal_scaler = gru_data["scaler"]
    classes = gru_data["classes"]

    X_val = X[val_mask]
    X_test = X[test_mask]

    X_val_flat = X_val.reshape(
        X_val.shape[0], -1
    )

    X_test_flat = X_test.reshape(
        X_test.shape[0], -1
    )

    X_val_scaled = temporal_scaler.transform(
        X_val_flat
    ).reshape(X_val.shape)

    X_test_scaled = temporal_scaler.transform(
        X_test_flat
    ).reshape(X_test.shape)

    val_prob = gru_model.predict(
        X_val_scaled,
        verbose=0
    )

    test_prob = gru_model.predict(
        X_test_scaled,
        verbose=0
    )

    # --------------------------------------------------------
    # Calibration on validation
    # --------------------------------------------------------

    calibration_rows = []

    for coverage_level in COVERAGE_LEVELS:

        lower_list = []
        upper_list = []

        for probs in val_prob:

            lo, hi = make_probability_interval(
                probs,
                classes,
                coverage=coverage_level
            )

            lower_list.append(lo)
            upper_list.append(hi)

        metrics = interval_metrics(
            y_val,
            lower_list,
            upper_list
        )

        calibration_rows.append({
            "requested_probability_mass": coverage_level,
            "validation_coverage": metrics["coverage"],
            "validation_mean_width_windows":
                metrics["mean_width_windows"],
            "validation_mean_width_seconds":
                metrics["mean_width_seconds"]
        })

    calibration_df = pd.DataFrame(
        calibration_rows
    )

    print("\nValidation interval calibration:")

    display(
        calibration_df.round(4)
    )

    # --------------------------------------------------------
    # Select smallest requested probability mass achieving
    # at least target empirical coverage.
    # --------------------------------------------------------

    eligible = calibration_df[
        calibration_df["validation_coverage"]
        >= TARGET_COVERAGE
    ]

    if len(eligible) > 0:

        selected_mass = float(
            eligible.iloc[0][
                "requested_probability_mass"
            ]
        )

    else:

        selected_mass = float(
            calibration_df.iloc[-1][
                "requested_probability_mass"
            ]
        )

    print(
        f"\nSelected probability mass: "
        f"{selected_mass:.2f}"
    )

    # --------------------------------------------------------
    # Generate calibrated intervals on TEST
    # --------------------------------------------------------

    test_lower = []
    test_upper = []

    for probs in test_prob:

        lo, hi = make_probability_interval(
            probs,
            classes,
            coverage=selected_mass
        )

        test_lower.append(lo)
        test_upper.append(hi)

    test_metrics = interval_metrics(
        y_test,
        test_lower,
        test_upper
    )

    # --------------------------------------------------------
    # Store event-level predictions
    # --------------------------------------------------------

    event_meta = data["metadata"].loc[
        test_mask
    ].copy()

    prediction_df = event_meta[
        [
            "file_id",
            "source_time",
            f"Y_{horizon}_onset_time"
        ]
    ].copy()

    prediction_df["actual_TTE_windows"] = y_test
    prediction_df["actual_TTE_seconds"] = y_test * 30

    prediction_df["predicted_lower_windows"] = test_lower
    prediction_df["predicted_upper_windows"] = test_upper

    prediction_df["predicted_lower_seconds"] = (
        np.asarray(test_lower) * 30
    )

    prediction_df["predicted_upper_seconds"] = (
        np.asarray(test_upper) * 30
    )

    prediction_df["interval_contains_true_TTE"] = (
        (y_test >= np.asarray(test_lower)) &
        (y_test <= np.asarray(test_upper))
    )

    prediction_df["interval_width_seconds"] = (
        (
            np.asarray(test_upper) -
            np.asarray(test_lower)
        ) * 30
    )

    tte_interval_predictions[horizon] = (
        prediction_df.reset_index(drop=True)
    )

    tte_interval_calibration[horizon] = {
        "calibration_table": calibration_df,
        "selected_probability_mass": selected_mass
    }

    # --------------------------------------------------------
    # Store summary
    # --------------------------------------------------------

    tte_interval_results.append({
        "horizon": horizon,
        "calibrated_probability_mass": selected_mass,
        "test_events": len(y_test),
        "test_interval_coverage":
            test_metrics["coverage"],
        "mean_interval_width_seconds":
            test_metrics["mean_width_seconds"],
        "median_interval_width_seconds":
            test_metrics["median_width_seconds"],
        "mean_miss_distance_seconds":
            test_metrics["mean_miss_distance_seconds"],
        "all_test_events_contained":
            test_metrics["all_contained"]
    })

    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print("\nTest interval performance:")

    for key, value in test_metrics.items():

        if isinstance(value, (float, np.floating)):
            print(
                f"  {key:32s}: "
                f"{value:.4f}"
            )
        else:
            print(
                f"  {key:32s}: "
                f"{value}"
            )

    print("\nEvent-level test intervals:")

    display(
        prediction_df[
            [
                "actual_TTE_seconds",
                "predicted_lower_seconds",
                "predicted_upper_seconds",
                "interval_contains_true_TTE",
                "interval_width_seconds"
            ]
        ]
    )

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

tte_interval_results_df = pd.DataFrame(
    tte_interval_results
)

print("\n" + "=" * 90)
print("CALIBRATED TTE-RANGE RESULTS")
print("=" * 90)

display(
    tte_interval_results_df.round(4)
)

# ------------------------------------------------------------
# Final interpretation
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CELL 26 COMPLETE")
print("=" * 90)

print("""
This experiment does NOT claim exact TTE.

Instead, it asks whether the model can provide an interval
that contains the true attack-onset time with useful coverage.

Example interpretation:

    "Estimated TTE: 60–120 seconds"

means the model's calibrated interval contains the estimated
30-second-window range corresponding to that interval.

Calibration used validation events only.
Test events were kept untouched until final evaluation.
""")

DRISHTI — CALIBRATED TTE-RANGE ESTIMATION

HORIZON: 60s

Validation interval calibration:


,requested_probability_mass,validation_coverage,validation_mean_width_windows,validation_mean_width_seconds
0,0.5,0.6667,0.0,0.0
1,0.6,1.0000,0.5,15.0
2,0.7,1.0000,1.0,30.0
3,0.8,1.0000,1.0,30.0
4,0.9,1.0000,1.0,30.0



Selected probability mass: 0.60

Test interval performance:
  coverage                        : 0.5882
  mean_width_windows              : 0.4118
  mean_width_seconds              : 12.3529
  median_width_windows            : 0.0000
  median_width_seconds            : 0.0000
  mean_miss_distance_windows      : 0.4118
  mean_miss_distance_seconds      : 12.3529
  all_contained                   : False

Event-level test intervals:


,actual_TTE_seconds,predicted_lower_seconds,predicted_upper_seconds,interval_contains_true_TTE,interval_width_seconds
3,30,30,60,True,30
13,60,60,60,True,0
14,30,30,60,True,30
33,60,30,60,True,30
34,60,30,30,False,0
35,30,60,60,False,0
36,60,30,60,True,30
37,60,60,60,True,0
38,60,30,60,True,30
63,30,30,60,True,30



HORIZON: 150s

Validation interval calibration:


...


Selected probability mass: 0.90

Test interval performance:
  coverage                        : 0.8571
  mean_width_windows              : 3.5714
  mean_width_seconds              : 107.1429
  median_width_windows            : 4.0000
  median_width_seconds            : 120.0000
  mean_miss_distance_windows      : 0.1429
  mean_miss_distance_seconds      : 4.2857
  all_contained                   : False

Event-level test intervals:


,actual_TTE_seconds,predicted_lower_seconds,predicted_upper_seconds,interval_contains_true_TTE,interval_width_seconds
3,30,30,150,True,120
14,30,30,150,True,120
33,150,30,120,False,90
34,150,60,150,True,90
35,30,30,150,True,120
36,120,30,150,True,120
37,120,30,150,True,120
38,120,30,120,True,90
62,30,30,150,True,120
63,60,30,150,True,120



CALIBRATED TTE-RANGE RESULTS


,horizon,calibrated_probability_mass,test_events,test_interval_coverage,mean_interval_width_seconds,median_interval_width_seconds,mean_miss_distance_seconds,all_test_events_contained
0,60s,0.6,17,0.5882,12.3529,0.0,12.3529,False
1,150s,0.9,14,0.8571,107.1429,120.0,4.2857,False



CELL 26 COMPLETE

This experiment does NOT claim exact TTE.

Instead, it asks whether the model can provide an interval
that contains the true attack-onset time with useful coverage.

Example interpretation:

    "Estimated TTE: 60–120 seconds"

means the model's calibrated interval contains the estimated
30-second-window range corresponding to that interval.

Calibration used validation events only.
Test events were kept untouched until final evaluation.



In [55]:
# ============================================================
# DRISHTI — BOOK 3 — CELL 27
# TTE RANGE → SITUATIONAL AWARENESS
# CORRECTED VERSION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 90)
print("DRISHTI — TTE RANGE → SITUATIONAL AWARENESS")
print("=" * 90)

HORIZON = "150s"

# ------------------------------------------------------------
# 1. Retrieve existing trained model + calibration
# ------------------------------------------------------------

tte_data = tte_event_datasets[HORIZON]
gru_data = tte_gru_models[HORIZON]
calibration = tte_interval_calibration[HORIZON]

gru_model = gru_data["model"]
gru_scaler = gru_data["scaler"]
gru_classes = gru_data["classes"]

selected_mass = calibration["selected_probability_mass"]

print(f"Horizon                  : {HORIZON}")
print(f"Calibrated probability   : {selected_mass:.2f}")
print(f"TTE classes              : {gru_classes.tolist()}")

assert selected_mass == 0.90

# ------------------------------------------------------------
# 2. Generate probabilities for all event-level samples
# ------------------------------------------------------------

X_events = tte_data["X"]
event_meta = tte_data["metadata"].copy()

X_flat = X_events.reshape(
    X_events.shape[0], -1
)

X_scaled = gru_scaler.transform(
    X_flat
).reshape(X_events.shape)

event_prob = gru_model.predict(
    X_scaled,
    verbose=0
)

assert event_prob.shape[0] == len(event_meta)

# ------------------------------------------------------------
# 3. Construct calibrated TTE intervals
# ------------------------------------------------------------

interval_lower = []
interval_upper = []

for probs in event_prob:

    lo, hi = make_probability_interval(
        probs,
        gru_classes,
        coverage=selected_mass
    )

    interval_lower.append(lo)
    interval_upper.append(hi)

interval_lower = np.asarray(
    interval_lower,
    dtype=int
)

interval_upper = np.asarray(
    interval_upper,
    dtype=int
)

# ------------------------------------------------------------
# 4. Convert TTE classes to seconds
# ------------------------------------------------------------

event_meta["tte_range_lower_windows"] = interval_lower
event_meta["tte_range_upper_windows"] = interval_upper

event_meta["tte_range_lower_seconds"] = (
    interval_lower * 30
)

event_meta["tte_range_upper_seconds"] = (
    interval_upper * 30
)

event_meta["tte_range_width_seconds"] = (
    event_meta["tte_range_upper_seconds"]
    - event_meta["tte_range_lower_seconds"]
)

# ------------------------------------------------------------
# 5. Operational TTE state
# ------------------------------------------------------------

def classify_tte_range(lower_s, upper_s):

    if upper_s <= 60:
        return "IMMEDIATE"

    elif lower_s <= 60:
        return "NEAR_TERM"

    else:
        return "SHORT_TERM"


event_meta["tte_operational_state"] = [
    classify_tte_range(lo, hi)
    for lo, hi in zip(
        event_meta["tte_range_lower_seconds"],
        event_meta["tte_range_upper_seconds"]
    )
]

# ------------------------------------------------------------
# 6. Most probable TTE class + probability
# ------------------------------------------------------------

event_meta["tte_top_class"] = (
    np.asarray(gru_classes)[
        np.argmax(event_prob, axis=1)
    ]
)

event_meta["tte_top_class_probability"] = (
    np.max(event_prob, axis=1)
)

# ------------------------------------------------------------
# 7. Ground-truth TTE — diagnostic only
# ------------------------------------------------------------

event_meta["actual_tte_seconds"] = (
    event_meta[
        f"Y_{HORIZON}_tte_windows"
    ] * 30
)

event_meta["range_contains_actual"] = (
    (
        event_meta["actual_tte_seconds"]
        >= event_meta["tte_range_lower_seconds"]
    )
    &
    (
        event_meta["actual_tte_seconds"]
        <= event_meta["tte_range_upper_seconds"]
    )
)

# ------------------------------------------------------------
# 8. Split-level summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TTE RANGE BY SPLIT")
print("=" * 90)

split_summary = (
    event_meta
    .groupby("split")
    .agg(
        events=("file_id", "size"),
        coverage=("range_contains_actual", "mean"),
        mean_width_seconds=(
            "tte_range_width_seconds",
            "mean"
        ),
        median_width_seconds=(
            "tte_range_width_seconds",
            "median"
        )
    )
    .reindex(["train", "validation", "test"])
)

display(
    split_summary.round(4)
)

# ------------------------------------------------------------
# 9. Operational state distribution
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("OPERATIONAL TTE STATE DISTRIBUTION")
print("=" * 90)

state_table = (
    event_meta[
        "tte_operational_state"
    ]
    .value_counts()
    .rename("events")
    .to_frame()
)

state_table["percentage"] = (
    100 * state_table["events"]
    / len(event_meta)
)

display(
    state_table.round(2)
)

# ------------------------------------------------------------
# 10. Test-only summary
# ------------------------------------------------------------

test_events = event_meta[
    event_meta["split"] == "test"
].copy()

print("\n" + "=" * 90)
print("TEST EVENT-LEVEL SITUATIONAL AWARENESS")
print("=" * 90)

test_summary = {
    "test_events":
        len(test_events),

    "coverage":
        test_events[
            "range_contains_actual"
        ].mean(),

    "mean_interval_width_seconds":
        test_events[
            "tte_range_width_seconds"
        ].mean(),

    "median_interval_width_seconds":
        test_events[
            "tte_range_width_seconds"
        ].median(),

    "immediate_events":
        (
            test_events[
                "tte_operational_state"
            ] == "IMMEDIATE"
        ).sum(),

    "near_term_events":
        (
            test_events[
                "tte_operational_state"
            ] == "NEAR_TERM"
        ).sum(),

    "short_term_events":
        (
            test_events[
                "tte_operational_state"
            ] == "SHORT_TERM"
        ).sum()
}

for key, value in test_summary.items():

    if isinstance(
        value,
        (float, np.floating)
    ):
        print(
            f"{key:35s}: {value:.4f}"
        )

    else:
        print(
            f"{key:35s}: {value}"
        )

# ------------------------------------------------------------
# 11. Test event-level TTE ranges
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TEST EVENT TTE-RANGE OUTPUT")
print("=" * 90)

test_output = (
    test_events[
        [
            "file_id",
            "source_time",
            "actual_tte_seconds",
            "tte_range_lower_seconds",
            "tte_range_upper_seconds",
            "tte_top_class",
            "tte_top_class_probability",
            "tte_operational_state",
            "range_contains_actual"
        ]
    ]
    .sort_values(
        ["source_time", "file_id"]
    )
    .reset_index(drop=True)
)

display(test_output)

# ------------------------------------------------------------
# 12. Dashboard-ready representation
# ------------------------------------------------------------

dashboard_tte = test_events[
    [
        "file_id",
        "source_time",
        "tte_range_lower_seconds",
        "tte_range_upper_seconds",
        "tte_top_class_probability",
        "tte_operational_state"
    ]
].copy()

dashboard_tte["display_range"] = (
    dashboard_tte[
        "tte_range_lower_seconds"
    ].astype(int).astype(str)
    + "–"
    + dashboard_tte[
        "tte_range_upper_seconds"
    ].astype(int).astype(str)
    + " sec"
)

print("\n" + "=" * 90)
print("DASHBOARD-READY TTE SIGNAL")
print("=" * 90)

display(
    dashboard_tte[
        [
            "source_time",
            "display_range",
            "tte_top_class_probability",
            "tte_operational_state"
        ]
    ]
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 13. Final status
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CELL 27 COMPLETE")
print("=" * 90)

print("""
DRISHTI TTE RANGE STATUS:

    Exact continuous TTE
        NOT VALIDATED

    Exact discrete TTE class
        NOT VALIDATED

    Calibrated coarse TTE range
        EXPERIMENTAL / SUPPORTING

    Calibration
        Validation events only

    Test evaluation
        Independent onset events only

The TTE range is an additional situational-awareness
signal and must not be presented as an exact countdown.
""")

DRISHTI — TTE RANGE → SITUATIONAL AWARENESS
Horizon                  : 150s
Calibrated probability   : 0.90
TTE classes              : [1, 2, 3, 4, 5]

TTE RANGE BY SPLIT


,events,coverage,mean_width_seconds,median_width_seconds
split,,,,
train,48,0.8125,110.0000,120.0
validation,6,1.0000,110.0000,120.0
test,14,0.8571,107.1429,120.0



OPERATIONAL TTE STATE DISTRIBUTION


,events,percentage
tte_operational_state,,
NEAR_TERM,68,100.0



TEST EVENT-LEVEL SITUATIONAL AWARENESS
test_events                        : 14
coverage                           : 0.8571
mean_interval_width_seconds        : 107.1429
median_interval_width_seconds      : 120.0000
immediate_events                   : 0
near_term_events                   : 14
short_term_events                  : 0

TEST EVENT TTE-RANGE OUTPUT


,file_id,source_time,actual_tte_seconds,tte_range_lower_seconds,tte_range_upper_seconds,tte_top_class,tte_top_class_probability,tte_operational_state,range_contains_actual
0,2,2019-04-25 06:23:00,30,30,150,2,0.313526,NEAR_TERM,True
1,3,2019-04-25 08:19:30,30,30,150,3,0.335968,NEAR_TERM,True
2,11,2019-04-28 07:39:00,150,30,120,2,0.343409,NEAR_TERM,False
3,11,2019-04-28 07:44:00,150,60,150,5,0.366669,NEAR_TERM,True
4,11,2019-04-28 07:47:00,30,30,150,2,0.338577,NEAR_TERM,True
5,11,2019-04-28 18:59:00,120,30,150,3,0.371139,NEAR_TERM,True
6,11,2019-04-28 19:19:30,120,30,150,3,0.275569,NEAR_TERM,True
7,11,2019-04-28 19:40:30,120,30,120,3,0.267929,NEAR_TERM,True
8,12,2019-04-29 14:09:30,30,30,150,2,0.316936,NEAR_TERM,True
9,12,2019-04-29 14:10:30,60,30,150,2,0.310597,NEAR_TERM,True



DASHBOARD-READY TTE SIGNAL


,source_time,display_range,tte_top_class_probability,tte_operational_state
0,2019-04-25 06:23:00,30–150 sec,0.313526,NEAR_TERM
1,2019-04-25 08:19:30,30–150 sec,0.335968,NEAR_TERM
2,2019-04-28 07:39:00,30–120 sec,0.343409,NEAR_TERM
3,2019-04-28 07:44:00,60–150 sec,0.366669,NEAR_TERM
4,2019-04-28 07:47:00,30–150 sec,0.338577,NEAR_TERM
5,2019-04-28 18:59:00,30–150 sec,0.371139,NEAR_TERM
6,2019-04-28 19:19:30,30–150 sec,0.275569,NEAR_TERM
7,2019-04-28 19:40:30,30–120 sec,0.267929,NEAR_TERM
8,2019-04-29 14:09:30,30–150 sec,0.316936,NEAR_TERM
9,2019-04-29 14:10:30,30–150 sec,0.310597,NEAR_TERM



CELL 27 COMPLETE

DRISHTI TTE RANGE STATUS:

    Exact continuous TTE
        NOT VALIDATED

    Exact discrete TTE class
        NOT VALIDATED

    Calibrated coarse TTE range
        EXPERIMENTAL / SUPPORTING

    Calibration
        Validation events only

    Test evaluation
        Independent onset events only

The TTE range is an additional situational-awareness
signal and must not be presented as an exact countdown.



In [56]:
# ============================================================
# DRISHTI — BOOK 3 — CELL 28
# FINAL TON-IoT FORECASTING EVIDENCE CONSOLIDATION
# ============================================================
#
# Purpose:
#   Create one canonical evidence table for the TON-IoT
#   forecasting / early-warning experiments.
#
# IMPORTANT:
#   - NO retraining
#   - NO new evaluation
#   - NO test-set tuning
#   - Uses only results already generated in Book 3
# ============================================================

import numpy as np
import pandas as pd

print("=" * 100)
print("DRISHTI — BOOK 3 — FINAL TON-IoT EVIDENCE CONSOLIDATION")
print("=" * 100)

# ============================================================
# 1. Static baseline results — Cell 17
# ============================================================

static_results = {
    "30s": {
        "PR_AUC": 0.418290,
        "ROC_AUC": 0.941589,
        "base_rate": 0.019473,
        "PR_lift": 21.480,
    },
    "60s": {
        "PR_AUC": 0.640451,
        "ROC_AUC": 0.951434,
        "base_rate": 0.029851,
        "PR_lift": 21.455,
    },
    "150s": {
        "PR_AUC": 0.907725,
        "ROC_AUC": 0.983422,
        "base_rate": 0.040509,
        "PR_lift": 22.408,
    },
    "300s": {
        "PR_AUC": 1.000000,
        "ROC_AUC": 1.000000,
        "base_rate": 0.046620,
        "PR_lift": 21.450,
    },
    "600s": {
        "PR_AUC": 1.000000,
        "ROC_AUC": 1.000000,
        "base_rate": 0.039239,
        "PR_lift": 25.485,
    },
}

# ============================================================
# 2. GRU results — Cell 19
# ============================================================

gru_results = {
    "30s": {
        "GRU_130_PR_AUC": 0.388342,
        "GRU_130_ROC_AUC": 0.980690,
        "GRU_134_PR_AUC": 0.447679,
        "GRU_134_ROC_AUC": 0.982752,
    },
    "60s": {
        "GRU_130_PR_AUC": 0.712468,
        "GRU_130_ROC_AUC": 0.989850,
        "GRU_134_PR_AUC": 0.672451,
        "GRU_134_ROC_AUC": 0.987984,
    },
    "150s": {
        "GRU_130_PR_AUC": 0.887382,
        "GRU_130_ROC_AUC": 0.996347,
        "GRU_134_PR_AUC": 0.779661,
        "GRU_134_ROC_AUC": 0.994348,
    },
    "300s": {
        "GRU_130_PR_AUC": 1.000000,
        "GRU_130_ROC_AUC": 1.000000,
        "GRU_134_PR_AUC": 1.000000,
        "GRU_134_ROC_AUC": 1.000000,
    },
    "600s": {
        "GRU_130_PR_AUC": 1.000000,
        "GRU_130_ROC_AUC": 1.000000,
        "GRU_134_PR_AUC": 1.000000,
        "GRU_134_ROC_AUC": 1.000000,
    },
}

# ============================================================
# 3. Feature-group ablation — Cell 18
# ============================================================

ablation_pr = {
    "State": {
        "30s": 0.516798,
        "60s": 0.652480,
        "150s": 0.875055,
        "300s": 0.971478,
        "600s": 0.955276,
    },
    "State + Structure": {
        "30s": 0.390592,
        "60s": 0.662331,
        "150s": 0.890477,
        "300s": 0.966622,
        "600s": 0.950331,
    },
    "State + Structure + Topology": {
        "30s": 0.514881,
        "60s": 0.702375,
        "150s": 0.876131,
        "300s": 0.976167,
        "600s": 0.916601,
    },
    "State + Structure + Topology + Telemetry": {
        "30s": 0.457375,
        "60s": 0.662393,
        "150s": 0.907578,
        "300s": 1.000000,
        "600s": 1.000000,
    },
    "All + Koopman": {
        "30s": 0.462411,
        "60s": 0.666493,
        "150s": 0.908556,
        "300s": 1.000000,
        "600s": 1.000000,
    },
}

# ============================================================
# 4. Operational early-warning evidence — Cell 21
# ============================================================

early_warning = {
    "horizon": "300s",
    "model": "GRU-134",
    "threshold": 0.90,
    "events_evaluated": 14,
    "events_detected": 13,
    "events_missed": 1,
    "detection_rate": 13 / 14,
    "median_lead_seconds": 120.0,
    "mean_lead_seconds": 126.9231,
    "min_lead_seconds": 30.0,
    "max_lead_seconds": 300.0,
}

# ============================================================
# 5. TTE range evidence — Cells 26/27
# ============================================================

tte_range = {
    "horizon": "150s",
    "model": "Temporal GRU",
    "calibration_mass": 0.90,
    "test_events": 14,
    "coverage": 0.8571,
    "mean_width_seconds": 107.1429,
    "median_width_seconds": 120.0,
    "mean_miss_distance_seconds": 4.2857,
}

# ============================================================
# 6. Main forecasting evidence table
# ============================================================

rows = []

for horizon in ["30s", "60s", "150s", "300s", "600s"]:

    rows.append({
        "horizon": horizon,

        "Static_LR_PR_AUC":
            static_results[horizon]["PR_AUC"],

        "Static_LR_ROC_AUC":
            static_results[horizon]["ROC_AUC"],

        "Base_Rate":
            static_results[horizon]["base_rate"],

        "PR_Lift":
            static_results[horizon]["PR_lift"],

        "GRU130_PR_AUC":
            gru_results[horizon]["GRU_130_PR_AUC"],

        "GRU130_ROC_AUC":
            gru_results[horizon]["GRU_130_ROC_AUC"],

        "GRU134_PR_AUC":
            gru_results[horizon]["GRU_134_PR_AUC"],

        "GRU134_ROC_AUC":
            gru_results[horizon]["GRU_134_ROC_AUC"],
    })

forecasting_evidence_df = pd.DataFrame(rows)

print("\n" + "=" * 100)
print("TON-IoT — FORECASTING PERFORMANCE")
print("=" * 100)

display(
    forecasting_evidence_df.round(4)
)

# ============================================================
# 7. Feature ablation table
# ============================================================

ablation_rows = []

for feature_group, horizon_values in ablation_pr.items():

    row = {
        "feature_group": feature_group
    }

    for horizon in [
        "30s",
        "60s",
        "150s",
        "300s",
        "600s"
    ]:
        row[f"{horizon}_PR_AUC"] = (
            horizon_values[horizon]
        )

    ablation_rows.append(row)

ablation_evidence_df = pd.DataFrame(
    ablation_rows
)

print("\n" + "=" * 100)
print("TON-IoT — FEATURE GROUP ABLATION")
print("=" * 100)

display(
    ablation_evidence_df.round(4)
)

# ============================================================
# 8. Operational early-warning table
# ============================================================

early_warning_df = pd.DataFrame([{
    "horizon": early_warning["horizon"],
    "model": early_warning["model"],
    "threshold": early_warning["threshold"],
    "events_evaluated": early_warning["events_evaluated"],
    "events_detected": early_warning["events_detected"],
    "events_missed": early_warning["events_missed"],
    "event_detection_rate":
        early_warning["detection_rate"],
    "median_lead_seconds":
        early_warning["median_lead_seconds"],
    "mean_lead_seconds":
        early_warning["mean_lead_seconds"],
    "min_lead_seconds":
        early_warning["min_lead_seconds"],
    "max_lead_seconds":
        early_warning["max_lead_seconds"],
}])

print("\n" + "=" * 100)
print("TON-IoT — EVENT-LEVEL EARLY WARNING")
print("=" * 100)

display(
    early_warning_df.round(4)
)

# ============================================================
# 9. TTE range table
# ============================================================

tte_range_df = pd.DataFrame([{
    "horizon": tte_range["horizon"],
    "model": tte_range["model"],
    "calibration_probability_mass":
        tte_range["calibration_mass"],
    "test_events":
        tte_range["test_events"],
    "interval_coverage":
        tte_range["coverage"],
    "mean_interval_width_seconds":
        tte_range["mean_width_seconds"],
    "median_interval_width_seconds":
        tte_range["median_width_seconds"],
    "mean_miss_distance_seconds":
        tte_range["mean_miss_distance_seconds"],
}])

print("\n" + "=" * 100)
print("TON-IoT — EXPERIMENTAL TTE RANGE")
print("=" * 100)

display(
    tte_range_df.round(4)
)

# ============================================================
# 10. Canonical status table
# ============================================================

status_rows = [

    {
        "component":
            "Future attack-onset forecasting",
        "status":
            "PROMOTED",
        "evidence":
            "TON-IoT 30–600s horizon evaluation",
        "caveat":
            "Performance varies by horizon/model"
    },

    {
        "component":
            "Temporal GRU comparison",
        "status":
            "SUPPORTING",
        "evidence":
            "GRU-130 / GRU-134 evaluated against static LR",
        "caveat":
            "Temporal history does not improve every horizon"
    },

    {
        "component":
            "Event-level early warning",
        "status":
            "PROMOTED",
        "evidence":
            "13/14 events detected at 0.90 threshold",
        "caveat":
            "14 independent test events"
    },

    {
        "component":
            "Warning lead-time measurement",
        "status":
            "PROMOTED",
        "evidence":
            "Median lead = 120s",
        "caveat":
            "Measured on evaluated events"
    },

    {
        "component":
            "Calibrated TTE range",
        "status":
            "EXPERIMENTAL / SUPPORTING",
        "evidence":
            "85.71% interval coverage",
        "caveat":
            "14 test events; median width = 120s"
    },

    {
        "component":
            "Exact continuous TTE",
        "status":
            "NOT VALIDATED",
        "evidence":
            "Discrete TTE experiments only",
        "caveat":
            "No continuous countdown claim"
    },

    {
        "component":
            "Zero-shot cross-dataset transfer",
        "status":
            "NOT CLAIMED",
        "evidence":
            "Datasets evaluated independently",
        "caveat":
            "Native-label evaluation"
    },

    {
        "component":
            "Universal attack prediction",
        "status":
            "NOT CLAIMED",
        "evidence":
            "Evaluation limited to studied datasets",
        "caveat":
            "No guarantee on arbitrary networks"
    },
]

status_df = pd.DataFrame(status_rows)

print("\n" + "=" * 100)
print("DRISHTI — CANONICAL CLAIM STATUS")
print("=" * 100)

display(status_df)

# ============================================================
# 11. Headline metrics
# ============================================================

headline_metrics = {
    "TON-IoT forecasting horizons":
        "30s / 60s / 150s / 300s / 600s",

    "Best TON-IoT static PR-AUC":
        "1.0000 @ 300s / 600s",

    "GRU-134 300s test PR-AUC":
        "1.0000",

    "GRU-134 600s test PR-AUC":
        "1.0000",

    "300s event-level warning":
        "13 / 14 detected",

    "300s event-level detection rate":
        "92.86%",

    "Median warning lead":
        "120 seconds",

    "Experimental TTE-range coverage":
        "85.71% @ 150s",

    "Median TTE-range width":
        "120 seconds",
}

print("\n" + "=" * 100)
print("DRISHTI — HEADLINE TON-IoT EVIDENCE")
print("=" * 100)

for key, value in headline_metrics.items():
    print(f"{key:42s}: {value}")

# ============================================================
# 12. Export canonical tables to globals
# ============================================================

book3_final_evidence = {
    "forecasting": forecasting_evidence_df,
    "ablation": ablation_evidence_df,
    "early_warning": early_warning_df,
    "tte_range": tte_range_df,
    "claim_status": status_df,
}

print("\n" + "=" * 100)
print("CELL 28 COMPLETE")
print("=" * 100)

print("""
Book 3 evidence is now consolidated.

Primary promoted evidence:
    1. Future attack-onset forecasting
    2. Event-level early warning
    3. Warning lead-time measurement

Supporting evidence:
    1. Temporal GRU comparison
    2. Horizon analysis
    3. Calibrated coarse TTE range

Explicitly NOT claimed:
    1. Exact continuous TTE
    2. Zero-shot cross-dataset transfer
    3. Universal attack prediction
""")

DRISHTI — BOOK 3 — FINAL TON-IoT EVIDENCE CONSOLIDATION

TON-IoT — FORECASTING PERFORMANCE


,horizon,Static_LR_PR_AUC,Static_LR_ROC_AUC,Base_Rate,PR_Lift,GRU130_PR_AUC,GRU130_ROC_AUC,GRU134_PR_AUC,GRU134_ROC_AUC
0,30s,0.4183,0.9416,0.0195,21.480,0.3883,0.9807,0.4477,0.9828
1,60s,0.6405,0.9514,0.0299,21.455,0.7125,0.9898,0.6725,0.9880
2,150s,0.9077,0.9834,0.0405,22.408,0.8874,0.9963,0.7797,0.9943
3,300s,1.0000,1.0000,0.0466,21.450,1.0000,1.0000,1.0000,1.0000
4,600s,1.0000,1.0000,0.0392,25.485,1.0000,1.0000,1.0000,1.0000



TON-IoT — FEATURE GROUP ABLATION


,feature_group,30s_PR_AUC,60s_PR_AUC,150s_PR_AUC,300s_PR_AUC,600s_PR_AUC
0,State,0.5168,0.6525,0.8751,0.9715,0.9553
1,State + Structure,0.3906,0.6623,0.8905,0.9666,0.9503
2,State + Structure + Topology,0.5149,0.7024,0.8761,0.9762,0.9166
3,State + Structure + Topology + Telemetry,0.4574,0.6624,0.9076,1.0000,1.0000
4,All + Koopman,0.4624,0.6665,0.9086,1.0000,1.0000



TON-IoT — EVENT-LEVEL EARLY WARNING


...


TON-IoT — EXPERIMENTAL TTE RANGE


,horizon,model,calibration_probability_mass,test_events,interval_coverage,mean_interval_width_seconds,median_interval_width_seconds,mean_miss_distance_seconds
0,150s,Temporal GRU,0.9,14,0.8571,107.1429,120.0,4.2857



DRISHTI — CANONICAL CLAIM STATUS


,component,status,evidence,caveat
0,Future attack-onset forecasting,PROMOTED,TON-IoT 30–600s horizon evaluation,Performance varies by horizon/model
1,Temporal GRU comparison,SUPPORTING,GRU-130 / GRU-134 evaluated against static LR,Temporal history does not improve every horizon
2,Event-level early warning,PROMOTED,13/14 events detected at 0.90 threshold,14 independent test events
3,Warning lead-time measurement,PROMOTED,Median lead = 120s,Measured on evaluated events
4,Calibrated TTE range,EXPERIMENTAL / SUPPORTING,85.71% interval coverage,14 test events; median width = 120s
5,Exact continuous TTE,NOT VALIDATED,Discrete TTE experiments only,No continuous countdown claim
6,Zero-shot cross-dataset transfer,NOT CLAIMED,Datasets evaluated independently,Native-label evaluation
7,Universal attack prediction,NOT CLAIMED,Evaluation limited to studied datasets,No guarantee on arbitrary networks



DRISHTI — HEADLINE TON-IoT EVIDENCE
TON-IoT forecasting horizons              : 30s / 60s / 150s / 300s / 600s
Best TON-IoT static PR-AUC                : 1.0000 @ 300s / 600s
GRU-134 300s test PR-AUC                  : 1.0000
GRU-134 600s test PR-AUC                  : 1.0000
300s event-level warning                  : 13 / 14 detected
300s event-level detection rate           : 92.86%
Median warning lead                       : 120 seconds
Experimental TTE-range coverage           : 85.71% @ 150s
Median TTE-range width                    : 120 seconds

CELL 28 COMPLETE

Book 3 evidence is now consolidated.

Primary promoted evidence:
    1. Future attack-onset forecasting
    2. Event-level early warning
    3. Warning lead-time measurement

Supporting evidence:
    1. Temporal GRU comparison
    2. Horizon analysis
    3. Calibrated coarse TTE range

Explicitly NOT claimed:
    1. Exact continuous TTE
    2. Zero-shot cross-dataset transfer
    3. Universal attack prediction



# DRISHTI — Book 3 Complete

## Cross-Dataset Attack Forecasting & Early Warning

Book 3 extends the DRISHTI forecasting framework to the TON-IoT
network intrusion dataset using a leakage-controlled temporal
attack-onset formulation.

### Completed

- Dataset and temporal integrity audit
- 30-second temporal window construction
- Genuine attack-onset definition
- 5-minute historical context construction
- State feature engineering
- Structure feature engineering
- Communication-topology feature engineering
- Packet/flow telemetry feature engineering
- Koopman temporal-dynamics representation
- Static Logistic Regression baseline
- Feature-group ablation
- Temporal GRU forecasting comparison
- Multi-horizon forecasting evaluation
- Fixed-threshold operational evaluation
- Event-level early-warning evaluation
- Warning lead-time measurement
- Discrete TTE feasibility analysis
- Calibrated coarse TTE-range experiment

### Final Evidence Status

**PROMOTED**
- Future attack-onset forecasting formulation
- Event-level early warning
- Fixed-threshold operational evaluation
- Warning lead-time measurement

**SUPPORTING / EXPERIMENTAL**
- Temporal GRU comparison
- Multi-horizon analysis
- Calibrated coarse TTE range

**NOT VALIDATED / NOT CLAIMED**
- Exact continuous TTE prediction
- Exact attack countdown
- Zero-shot cross-dataset transfer
- Universal attack prediction

### Key Operational Evidence

On the evaluated TON-IoT test events, the 300-second forecasting
configuration using GRU-134 detected 13 of 14 independent attack-onset
events at a fixed 0.90 threshold.

- Event detection: 13/14
- Detection rate: 92.86%
- Median warning lead: 120 seconds
- Mean warning lead: 126.92 seconds
- Observed lead range: 30–300 seconds

The calibrated 150-second TTE-range experiment achieved:

- Test events: 14
- Interval coverage: 85.71%
- Median interval width: 120 seconds

These TTE-range results are experimental and do not constitute
validated exact time-to-event prediction.

## Notebook Closure

Book 3 is considered complete.

Further work will move outside the research notebook into the
DRISHTI system architecture, implementation, visualization,
demonstration, documentation, and final presentation.